# Notebook 2 – Cleaning and Feature Engineering

This notebook prepares the BT network data for the two modelling notebooks and adds the Point of Interest (POI) information created in `1_POI_Dataset_Overpass` (Notebook 1).

There are two parts because the project uses BT data at two different time resolutions:

- **daily BT data** is prepared for `3_Model_1` (Notebook 3);
- **hourly BT data** is prepared for `4_Model_2` (Notebook 4).

## Daily Data

The first part processes all available daily BT files.

The files are checked to make sure they contain the expected data and columns. Empty files are identified and excluded, while the usable files are cleaned and saved as reusable Parquet files.

A `location_id` is also created from the BT grid coordinates so that observations belonging to the same geographic location can be identified across different files and dates.

## Adding Point of Interest Information

The POI dataset created in `1_POI_Dataset_Overpass` (Notebook 1) is then combined with the BT locations.

For every unique BT grid location, the notebook calculates:

- the distance to the nearest POI in each available POI category;
- the distance to the nearest transport hub;
- the distance to the nearest POI overall;
- the type of the nearest POI;
- the name of the nearest POI when available.

These values provide geographic context for each BT observation. For example, later analysis can determine whether unusual network activity occurred close to locations such as stadiums, railway stations, hotels or tourist attractions.

The completed daily data, including the POI information, is saved as:

`bt_final_model_dataset`

This becomes the input for `3_Model_1` (Notebook 3).

## Hourly Data

The second part repeats the preparation process for the hourly BT files.

The hourly files are discovered, checked, cleaned and saved as reusable Parquet files.

The original hourly timing information is retained, and `hourly_datetime` and `hour` are also created so that `4_Model_2` (Notebook 4) can investigate how network activity changes throughout the day.

The same POI information is added to the hourly BT locations so that both models use consistent geographic context.

The completed hourly data is saved as:

`bt_final_hourly_model_dataset`

This becomes the input for `4_Model_2` (Notebook 4).

## Reusing Previous Work

The notebook is designed so that it does not need to repeat all of the processing every time it is run.

Existing valid cleaned files, POI calculations and final datasets are reused where possible. If new BT files or new BT grid locations are added, only the additional work required is carried out.

The POI dataset is also fingerprinted. If the POI data from `1_POI_Dataset_Overpass` (Notebook 1) changes, the notebook detects this and updates the required POI information.

The final workflow is therefore:

**Daily BT data → cleaning + POI information → `bt_final_model_dataset` → `3_Model_1` (Notebook 3)**

**Hourly BT data → cleaning + POI information → `bt_final_hourly_model_dataset` → `4_Model_2` (Notebook 4)**

`2_Cleaning_and_Feature_Engineering` (Notebook 2) therefore provides the complete data-preparation stage between the original BT and POI data and the two modelling notebooks.

# Notebook Roadmap

| Section | Cells | Purpose |
|---|---:|---|
| 1. Project Setup | 1–3 | Prepare the Python environment and shared project folders |
| 2. Daily BT File Discovery | 4–9 | Find, inspect and classify all available daily BT files |
| 3. Daily Data Cleaning | 10–12 | Define the daily BT columns, data types and cleaning process |
| 4. Complete Daily Cleaning Test | 13–16 | Test the cleaning process on one complete daily file |
| 5. Clean All Daily Partitions | 17–20 | Create or reuse one cleaned partition for every usable daily file |
| 6. Daily BT Locations | 21–22 | Build the current set of unique daily BT grid locations |
| 7. POI Preparation | 23–25 | Load the processed POI dataset and dynamically define its contextual features |
| 8. Daily POI Feature Creation | 26–29 | Create, reuse or rebuild the shared grid-level POI information |
| 9. Complete Daily POI Join Test | 30 | Test the dynamic POI join on one complete daily partition |
| 10. Final Daily Model Dataset | 31–32 | Create or reuse the POI-enriched daily model-ready partitions |
| 11. Daily Final Verification | 33–35 | Verify and complete the daily Model 1 dataset |
| 12. Hourly Data Configuration and Discovery | 36–39 | Configure the hourly workflow and discover the current hourly dataset structure |
| 13. Complete Hourly Source Audit | 40–42 | Validate every hourly source file and define the hourly cleaning structure |
| 14. Hourly Cleaning | 43–46 | Test, create and validate the cleaned hourly partitions |
| 15. Hourly BT Locations and POI Coverage | 47–50 | Identify hourly grid locations and extend the shared POI lookup where required |
| 16. Final Hourly Model Dataset | 51–54 | Create, enrich and independently verify the hourly model-ready partitions |
| 17. Notebook Completion | 55 | Confirm that both daily and hourly modelling datasets are complete |

The notebook contains 55 numbered code cells.

The daily and hourly preprocessing workflows remain separate because the two BT datasets represent different temporal resolutions, but both use the same dynamically maintained grid-level POI context.

The workflow is designed to adapt to future compatible BT files and changes to the processed POI dataset while reusing previously validated work wherever possible.

# Python Package Setup

This cell checks that the Python packages required by the project are available.

Packages that are already installed are left unchanged. If a required package is missing, it is installed automatically into the Python environment used by this notebook.

This allows the notebook to run on another computer without requiring the packages to be installed manually beforehand.

In [1]:
# ================================================================
# CHECK AND INSTALL REQUIRED PYTHON PACKAGES
# ================================================================

import importlib.util
import subprocess
import sys


required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "duckdb": "duckdb",
    "geopandas": "geopandas",
    "fiona": "fiona",
    "shapely": "shapely",
    "osmnx": "osmnx",
    "pyproj": "pyproj",
    "folium": "folium",
    "IPython": "ipython"
}


print("=" * 80)
print("CHECK AND INSTALL REQUIRED PYTHON PACKAGES")
print("=" * 80)


installed_packages = []
missing_packages = []


for import_name, package_name in required_packages.items():

    if importlib.util.find_spec(import_name) is None:

        missing_packages.append(
            package_name
        )

    else:

        installed_packages.append(
            package_name
        )


print(
    f"\nPackages already available: "
    f"{len(installed_packages):,}"
)

for package_name in installed_packages:

    print(
        f"  - {package_name}"
    )


if missing_packages:

    print(
        f"\nPackages requiring installation: "
        f"{len(missing_packages):,}"
    )

    for package_name in missing_packages:

        print(
            f"  - {package_name}"
        )


    print(
        "\nInstalling missing packages..."
    )


    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            *missing_packages
        ]
    )


    print(
        "\nMissing packages were installed successfully."
    )

else:

    print(
        "\nAll required packages are already installed."
    )


print(
    "\nPython package setup complete."
)

CHECK AND INSTALL REQUIRED PYTHON PACKAGES

Packages already available: 15
  - numpy
  - pandas
  - pyarrow
  - matplotlib
  - seaborn
  - scikit-learn
  - joblib
  - duckdb
  - geopandas
  - fiona
  - shapely
  - osmnx
  - pyproj
  - folium
  - ipython

All required packages are already installed.

Python package setup complete.


# Section 1 - Daily Data

# 1. Project Setup

This section imports the required Python packages, finds the project folder and prepares the input and output locations used by the notebook.

In [2]:
# ================================================================
# CELL 1 - IMPORT CORE PACKAGES
# ================================================================

from pathlib import Path

import csv
import gc
import hashlib
import json
import platform
import re
import sys
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from IPython.display import display


pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    200
)

pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.6f}"
)


print("=" * 80)
print("CELL 1 - IMPORT CORE PACKAGES")
print("=" * 80)

print(
    "\nPython libraries imported successfully."
)


software_versions = pd.DataFrame(
    {
        "Software": [
            "Python",
            "Operating System",
            "pandas",
            "NumPy",
            "PyArrow"
        ],
        "Version": [
            sys.version.split()[0],
            platform.platform(),
            pd.__version__,
            np.__version__,
            pa.__version__
        ]
    }
)


display(
    software_versions
)

CELL 1 - IMPORT CORE PACKAGES

Python libraries imported successfully.


,Software,Version
0,Python,3.13.5
1,Operating System,Windows-11-10.0.26200-SP0
2,pandas,2.2.3
3,NumPy,2.1.3
4,PyArrow,19.0.0


## What Cell 1 Does

This cell imports the main Python packages used in the notebook.

These packages are used to find files, process large tables, save Parquet outputs, measure processing time and manage memory.

The software versions are displayed to support reproducibility when the project is opened on another computer.

In [3]:
# ================================================================
# CELL 2 - CONFIGURE PROJECT FOLDERS
# ================================================================

EXPECTED_DAILY_FOLDER = Path(
    "data",
    "bt_daily_data"
)

EXPECTED_POI_FILE = Path(
    "data",
    "processed_poi_locations",
    "BT_POI_Dataset.gpkg"
)


def find_project_folder():
    """
    Search the current folder and its parent folders for the
    BT dissertation project structure.
    """

    current_folder = Path.cwd().resolve()

    candidate_folders = [
        current_folder,
        *current_folder.parents
    ]

    for candidate_folder in candidate_folders:

        expected_daily_path = (
            candidate_folder
            / EXPECTED_DAILY_FOLDER
        )

        if expected_daily_path.exists():

            return candidate_folder

    raise FileNotFoundError(
        "Could not locate the BT_Dissertation_AL project folder. "
        "The notebook must be stored inside the project folder or "
        "one of its subfolders, and data/bt_daily_data must exist."
    )


PROJECT_FOLDER = find_project_folder()

DATA_FOLDER = (
    PROJECT_FOLDER
    / "data"
)

DAILY_INPUT_FOLDER = (
    DATA_FOLDER
    / "bt_daily_data"
)

HOURLY_INPUT_FOLDER = (
    DATA_FOLDER
    / "bt_hourly_data"
)

POI_INPUT_FILE = (
    PROJECT_FOLDER
    / EXPECTED_POI_FILE
)


CLEANED_MODEL_DATA_FOLDER = (
    DATA_FOLDER
    / "cleaned_model_data"
)

DAILY_CLEANED_PARTITIONS_FOLDER = (
    CLEANED_MODEL_DATA_FOLDER
    / "daily_cleaned_partitions"
)

POI_LOCATION_FEATURES_FOLDER = (
    CLEANED_MODEL_DATA_FOLDER
    / "poi_location_features"
)

FINAL_MODEL_DATASET_FOLDER = (
    CLEANED_MODEL_DATA_FOLDER
    / "bt_final_model_dataset"
)


DAILY_CLEANED_PARTITIONS_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

POI_LOCATION_FEATURES_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_MODEL_DATASET_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


POI_LOCATION_FEATURES_FILE = (
    POI_LOCATION_FEATURES_FOLDER
    / "BT_Grid_POI_Features.parquet"
)


print("=" * 80)
print("CELL 2 - CONFIGURE PROJECT FOLDERS")
print("=" * 80)


project_folder_table = pd.DataFrame(
    {
        "Purpose": [
            "Project folder",
            "Daily BT input",
            "Hourly BT data",
            "Processed POI input",
            "Cleaned model data",
            "Daily cleaned partitions",
            "POI location features",
            "Final model dataset"
        ],
        "Location": [
            str(PROJECT_FOLDER),
            str(DAILY_INPUT_FOLDER),
            str(HOURLY_INPUT_FOLDER),
            str(POI_INPUT_FILE),
            str(CLEANED_MODEL_DATA_FOLDER),
            str(DAILY_CLEANED_PARTITIONS_FOLDER),
            str(POI_LOCATION_FEATURES_FOLDER),
            str(FINAL_MODEL_DATASET_FOLDER)
        ]
    }
)

display(
    project_folder_table
)


print(
    "\nThe project folders were located and "
    "configured successfully."
)

CELL 2 - CONFIGURE PROJECT FOLDERS


,Purpose,Location
0,Project folder,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
1,Daily BT input,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
2,Hourly BT data,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
3,Processed POI input,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
4,Cleaned model data,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
5,Daily cleaned partitions,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
6,POI location features,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
7,Final model dataset,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...



The project folders were located and configured successfully.


## What Cell 2 Does

This cell finds the main project folder and defines the locations of the input and output data used by the daily and shared POI stages of the notebook.

The paths are created relative to the project folder rather than being tied to one computer-specific location.

At this stage, the prepared outputs are organised into three folders:

* `daily_cleaned_partitions` for the cleaned daily BT files;
* `poi_location_features` for the reusable grid-level POI location dataset;
* `bt_final_model_dataset` for the final daily model-ready files used by Notebook 3.

The separate hourly cleaned and final output locations are configured later when the hourly section begins.


In [4]:
# ================================================================
# CELL 3 - VERIFY REQUIRED PROJECT INPUTS
# ================================================================

required_input_checks = pd.DataFrame(
    {
        "Input": [
            "Project folder",
            "Daily BT input folder",
            "Processed POI dataset"
        ],
        "Location": [
            str(PROJECT_FOLDER),
            str(DAILY_INPUT_FOLDER),
            str(POI_INPUT_FILE)
        ],
        "Exists": [
            PROJECT_FOLDER.exists(),
            DAILY_INPUT_FOLDER.exists(),
            POI_INPUT_FILE.exists()
        ]
    }
)


output_folder_checks = pd.DataFrame(
    {
        "Output": [
            "Cleaned model data folder",
            "Daily cleaned partitions",
            "POI location features",
            "Final model dataset"
        ],
        "Location": [
            str(CLEANED_MODEL_DATA_FOLDER),
            str(DAILY_CLEANED_PARTITIONS_FOLDER),
            str(POI_LOCATION_FEATURES_FOLDER),
            str(FINAL_MODEL_DATASET_FOLDER)
        ],
        "Exists": [
            CLEANED_MODEL_DATA_FOLDER.exists(),
            DAILY_CLEANED_PARTITIONS_FOLDER.exists(),
            POI_LOCATION_FEATURES_FOLDER.exists(),
            FINAL_MODEL_DATASET_FOLDER.exists()
        ]
    }
)


raw_input_paths = [
    DAILY_INPUT_FOLDER.resolve(),
    HOURLY_INPUT_FOLDER.resolve(),
    POI_INPUT_FILE.resolve()
]

output_paths = [
    DAILY_CLEANED_PARTITIONS_FOLDER.resolve(),
    POI_LOCATION_FEATURES_FOLDER.resolve(),
    FINAL_MODEL_DATASET_FOLDER.resolve()
]


input_output_overlap = any(
    output_path == input_path
    or output_path in input_path.parents
    or input_path in output_path.parents
    for output_path in output_paths
    for input_path in raw_input_paths
)


distinct_output_folders = (
    len(set(output_paths))
    == len(output_paths)
)


print("=" * 80)
print("CELL 3 - VERIFY REQUIRED PROJECT INPUTS")
print("=" * 80)

print("\nRequired inputs:")

display(
    required_input_checks
)

print("\nOutput folders:")

display(
    output_folder_checks
)

print(
    f"\nInput and output path overlap: "
    f"{input_output_overlap}"
)

print(
    f"Output folders are distinct: "
    f"{distinct_output_folders}"
)


assert required_input_checks[
    "Exists"
].all()

assert output_folder_checks[
    "Exists"
].all()

assert not input_output_overlap

assert distinct_output_folders


print(
    "\nAll required inputs and output folders "
    "were verified successfully."
)

CELL 3 - VERIFY REQUIRED PROJECT INPUTS

Required inputs:


,Input,Location,Exists
0,Project folder,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...,True
1,Daily BT input folder,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...,True
2,Processed POI dataset,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...,True



Output folders:


,Output,Location,Exists
0,Cleaned model data folder,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...,True
1,Daily cleaned partitions,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...,True
2,POI location features,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...,True
3,Final model dataset,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...,True



Input and output path overlap: False
Output folders are distinct: True

All required inputs and output folders were verified successfully.


## What Cell 3 Does

This cell checks that the inputs required for the daily and shared POI stages are available and that the corresponding output folders have been created correctly.

It confirms that the project folder, daily BT input folder and processed POI dataset produced by Notebook 1 exist.

It also checks that the daily output locations are separate from the original input locations so that the source data are not overwritten.

The hourly input and output locations are configured and verified separately when the hourly section begins.

The notebook stops at this stage if an essential daily or POI input is missing.


# 2. BT File Discovery

This section finds all supplied BT files and checks which daily files can be used.

The number of files is not fixed. Any additional daily CSV files placed in the input folder will be included automatically when the notebook is rerun.

In [5]:
# ================================================================
# CELL 4 - DISCOVER BT DATA FILES
# ================================================================

daily_csv_files = sorted(
    [
        file_path
        for file_path
        in DAILY_INPUT_FOLDER.glob(
            "*.csv"
        )
        if file_path.is_file()
    ]
)


hourly_csv_files = sorted(
    [
        file_path
        for file_path
        in HOURLY_INPUT_FOLDER.glob(
            "*.csv"
        )
        if file_path.is_file()
    ]
    if HOURLY_INPUT_FOLDER.exists()
    else []
)


total_daily_size_bytes = sum(
    file_path.stat().st_size
    for file_path
    in daily_csv_files
)


duplicate_daily_filenames = (
    len(daily_csv_files)
    -
    len(
        {
            file_path.name
            for file_path
            in daily_csv_files
        }
    )
)


file_discovery_summary = pd.DataFrame(
    {
        "Dataset": [
            "Daily BT files",
            "Hourly BT files"
        ],
        "Files Found": [
            len(daily_csv_files),
            len(hourly_csv_files)
        ],
        "Used in Daily Section": [
            True,
            False
        ]
    }
)


print("=" * 80)
print("CELL 4 - DISCOVER BT DATA FILES")
print("=" * 80)

display(
    file_discovery_summary
)


if daily_csv_files:

    print(
        f"\nFirst daily file:\n"
        f"{daily_csv_files[0].name}"
    )

    print(
        f"\nFinal daily file:\n"
        f"{daily_csv_files[-1].name}"
    )


print(
    f"\nCombined daily-file size: "
    f"{total_daily_size_bytes / 1_000_000_000:,.2f} GB"
)

print(
    f"Duplicate daily filenames: "
    f"{duplicate_daily_filenames:,}"
)


assert len(daily_csv_files) > 0

assert duplicate_daily_filenames == 0


print(
    "\nThe supplied BT data files were "
    "discovered successfully."
)

CELL 4 - DISCOVER BT DATA FILES


,Dataset,Files Found,Used in Daily Section
0,Daily BT files,95,True
1,Hourly BT files,4,False



First daily file:
GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-04_2026-04-05_Neon_v93_csv.csv

Final daily file:
GEO_Export_Neon_530k_5590k_668k_5650k_2026-07-24_2026-07-25_Neon_v93_csv.csv

Combined daily-file size: 15.89 GB
Duplicate daily filenames: 0

The supplied BT data files were discovered successfully.


## What Cell 4 Does

This cell finds all CSV files in the daily and hourly BT data folders.

The daily files are passed into the daily preprocessing workflow in this section.

The hourly files are also discovered and counted here, but they are deliberately excluded from the daily workflow because they are processed separately in the hourly section later in this notebook.

The cell also reports the combined size of the daily source data and checks that no daily filenames are duplicated.


In [6]:
# ================================================================
# CELL 5 - DEFINE THE EXPECTED BT SCHEMA
# ================================================================

SOURCE_TABLE_PREFIX = (
    "b_geolte_daily_bin."
)


ORIGINAL_BT_COLUMNS = [
    "xbin",
    "ybin",
    "averagedownlinkthroughput",
    "averageuplinkthroughput",
    "csfallbackattempts",
    "irathandoverattempts",
    "linearaveragersrp",
    "linearaveragersrq",
    "minutesofuse",
    "numberofconnections",
    "pedestrianminutesofuse",
    "s1handoverattempts",
    "s1handoverfailures",
    "s1handoversuccesses",
    "stationaryminutesofuse",
    "totalconnectionblocks",
    "totalconnectiondrops",
    "totalconnectionnormalreleases",
    "totaldownlinkdataduration",
    "totaldownlinkvolume",
    "totalerabblocks",
    "totalerabdrops",
    "totalerabnormalreleases",
    "totaluplinkdataduration",
    "totaluplinkvolume",
    "vehicularminutesofuse",
    "voiceminutesofuse",
    "x2handoverattempts",
    "x2handoverfailures",
    "x2handoversuccesses",
    "indoorminutesofuse",
    "outdoorminutesofuse",
    "en_dt"
]


RAW_BT_COLUMNS = [
    (
        f"{SOURCE_TABLE_PREFIX}"
        f"{column_name}"
    )
    for column_name
    in ORIGINAL_BT_COLUMNS
]


CLEANED_BT_COLUMNS = [
    "location_id",
    *ORIGINAL_BT_COLUMNS
]


BT_IDENTIFIER_COLUMNS = [
    "location_id"
]


BT_COORDINATE_COLUMNS = [
    "xbin",
    "ybin"
]


BT_TIME_COLUMNS = [
    "en_dt"
]


BT_CONTEXT_COLUMNS = (
    BT_IDENTIFIER_COLUMNS
    +
    BT_COORDINATE_COLUMNS
    +
    BT_TIME_COLUMNS
)


BT_MEASUREMENT_COLUMNS = [
    column_name
    for column_name
    in ORIGINAL_BT_COLUMNS
    if column_name
    not in (
        BT_COORDINATE_COLUMNS
        +
        BT_TIME_COLUMNS
    )
]


column_structure_summary = pd.DataFrame(
    {
        "Column Group": [
            "Supplied BT columns",
            "Grid-location identifier",
            "Cleaned partition columns",
            "BT network measurements",
            "BT coordinate columns",
            "BT time columns"
        ],
        "Columns": [
            len(
                ORIGINAL_BT_COLUMNS
            ),
            len(
                BT_IDENTIFIER_COLUMNS
            ),
            len(
                CLEANED_BT_COLUMNS
            ),
            len(
                BT_MEASUREMENT_COLUMNS
            ),
            len(
                BT_COORDINATE_COLUMNS
            ),
            len(
                BT_TIME_COLUMNS
            )
        ]
    }
)


print("=" * 80)
print("CELL 5 - DEFINE THE EXPECTED BT SCHEMA")
print("=" * 80)


display(
    column_structure_summary
)


print(
    "\nCleaned BT column order:"
)


display(
    pd.DataFrame(
        {
            "Column Number": range(
                1,
                len(
                    CLEANED_BT_COLUMNS
                )
                +
                1
            ),
            "Column": (
                CLEANED_BT_COLUMNS
            )
        }
    )
)


assert len(
    ORIGINAL_BT_COLUMNS
) == 33


assert len(
    RAW_BT_COLUMNS
) == 33


assert len(
    CLEANED_BT_COLUMNS
) == 34


assert len(
    BT_MEASUREMENT_COLUMNS
) == 30


assert CLEANED_BT_COLUMNS[
    0
] == "location_id"


assert len(
    set(
        CLEANED_BT_COLUMNS
    )
) == len(
    CLEANED_BT_COLUMNS
)


print(
    "\nThe expected BT schema was "
    "defined successfully."
)


print(
    "\nThe POI and final model schemas will be "
    "defined dynamically after the current "
    "processed POI dataset is loaded."
)

CELL 5 - DEFINE THE EXPECTED BT SCHEMA


,Column Group,Columns
0,Supplied BT columns,33
1,Grid-location identifier,1
2,Cleaned partition columns,34
3,BT network measurements,30
4,BT coordinate columns,2
5,BT time columns,1



Cleaned BT column order:


,Column Number,Column
0,1,location_id
1,2,xbin
2,3,ybin
3,4,averagedownlinkthroughput
4,5,averageuplinkthroughput
5,6,csfallbackattempts
6,7,irathandoverattempts
7,8,linearaveragersrp
8,9,linearaveragersrq
9,10,minutesofuse



The expected BT schema was defined successfully.

The POI and final model schemas will be defined dynamically after the current processed POI dataset is loaded.


## What Cell 5 Does

This cell defines the expected structure of the supplied BT data.

Each source file must contain the same 33 BT columns:

- two grid-coordinate columns;
- 30 network measurement columns;
- one date column.

The cleaning process later adds `location_id`, producing 34 cleaned columns.

The POI and final model schemas are not fixed here. Their sizes are determined dynamically after the current processed POI dataset is loaded.

In [7]:
# ================================================================
# CELL 6 - DISPLAY THE BT DATA DICTIONARY
# ================================================================

bt_column_definitions = {
    "location_id":
        "Stable identifier created from the BT grid coordinates.",
    "xbin":
        "UTM easting coordinate of the BT grid location.",
    "ybin":
        "UTM northing coordinate of the BT grid location.",
    "averagedownlinkthroughput":
        "Average downlink data throughput recorded for the observation.",
    "averageuplinkthroughput":
        "Average uplink data throughput recorded for the observation.",
    "csfallbackattempts":
        "Number of circuit-switched fallback attempts.",
    "irathandoverattempts":
        "Number of inter-radio-access-technology handover attempts.",
    "linearaveragersrp":
        "Average Reference Signal Received Power value.",
    "linearaveragersrq":
        "Average Reference Signal Received Quality value.",
    "minutesofuse":
        "Total recorded minutes of network use.",
    "numberofconnections":
        "Total number of recorded network connections.",
    "pedestrianminutesofuse":
        "Minutes of use associated with pedestrian activity.",
    "s1handoverattempts":
        "Number of attempted S1 handovers.",
    "s1handoverfailures":
        "Number of failed S1 handovers.",
    "s1handoversuccesses":
        "Number of successful S1 handovers.",
    "stationaryminutesofuse":
        "Minutes of use associated with stationary activity.",
    "totalconnectionblocks":
        "Total number of blocked connection events.",
    "totalconnectiondrops":
        "Total number of unexpectedly dropped connections.",
    "totalconnectionnormalreleases":
        "Total number of connections ending through a normal release.",
    "totaldownlinkdataduration":
        "Total recorded duration of downlink data activity.",
    "totaldownlinkvolume":
        "Total recorded downlink data volume.",
    "totalerabblocks":
        "Total number of blocked E-RAB events.",
    "totalerabdrops":
        "Total number of dropped E-RAB events.",
    "totalerabnormalreleases":
        "Total number of normally released E-RABs.",
    "totaluplinkdataduration":
        "Total recorded duration of uplink data activity.",
    "totaluplinkvolume":
        "Total recorded uplink data volume.",
    "vehicularminutesofuse":
        "Minutes of use associated with vehicular activity.",
    "voiceminutesofuse":
        "Total recorded minutes of voice use.",
    "x2handoverattempts":
        "Number of attempted X2 handovers.",
    "x2handoverfailures":
        "Number of failed X2 handovers.",
    "x2handoversuccesses":
        "Number of successful X2 handovers.",
    "indoorminutesofuse":
        "Minutes of use classified as indoor activity.",
    "outdoorminutesofuse":
        "Minutes of use classified as outdoor activity.",
    "en_dt":
        "Date associated with the daily BT grid observation."
}


bt_column_categories = {
    "location_id": "Identifier",
    "xbin": "Location",
    "ybin": "Location",
    "averagedownlinkthroughput": "Throughput",
    "averageuplinkthroughput": "Throughput",
    "csfallbackattempts": "Connection Activity",
    "irathandoverattempts": "Handovers",
    "linearaveragersrp": "Signal",
    "linearaveragersrq": "Signal",
    "minutesofuse": "Usage",
    "numberofconnections": "Connection Activity",
    "pedestrianminutesofuse": "Mobility",
    "s1handoverattempts": "Handovers",
    "s1handoverfailures": "Handovers",
    "s1handoversuccesses": "Handovers",
    "stationaryminutesofuse": "Mobility",
    "totalconnectionblocks": "Connection Reliability",
    "totalconnectiondrops": "Connection Reliability",
    "totalconnectionnormalreleases": "Connection Reliability",
    "totaldownlinkdataduration": "Data Duration",
    "totaldownlinkvolume": "Data Volume",
    "totalerabblocks": "E-RAB Reliability",
    "totalerabdrops": "E-RAB Reliability",
    "totalerabnormalreleases": "E-RAB Reliability",
    "totaluplinkdataduration": "Data Duration",
    "totaluplinkvolume": "Data Volume",
    "vehicularminutesofuse": "Mobility",
    "voiceminutesofuse": "Usage",
    "x2handoverattempts": "Handovers",
    "x2handoverfailures": "Handovers",
    "x2handoversuccesses": "Handovers",
    "indoorminutesofuse": "Usage Environment",
    "outdoorminutesofuse": "Usage Environment",
    "en_dt": "Time"
}


bt_expected_types = {
    column_name: (
        "String"
        if column_name == "location_id"
        else
        "Datetime"
        if column_name == "en_dt"
        else
        "Numeric"
    )
    for column_name
    in CLEANED_BT_COLUMNS
}


bt_later_roles = {
    column_name: (
        "POI join and location analysis"
        if column_name == "location_id"
        else
        "Location and interpretation"
        if column_name in BT_COORDINATE_COLUMNS
        else
        "Temporal interpretation"
        if column_name == "en_dt"
        else
        "Candidate network measurement"
    )
    for column_name
    in CLEANED_BT_COLUMNS
}


bt_data_dictionary = pd.DataFrame(
    {
        "Column Number": range(
            1,
            len(CLEANED_BT_COLUMNS) + 1
        ),
        "Column": CLEANED_BT_COLUMNS,
        "Category": [
            bt_column_categories[column_name]
            for column_name
            in CLEANED_BT_COLUMNS
        ],
        "Expected Type": [
            bt_expected_types[column_name]
            for column_name
            in CLEANED_BT_COLUMNS
        ],
        "Definition": [
            bt_column_definitions[column_name]
            for column_name
            in CLEANED_BT_COLUMNS
        ],
        "Later Role": [
            bt_later_roles[column_name]
            for column_name
            in CLEANED_BT_COLUMNS
        ]
    }
)


bt_category_summary = (
    bt_data_dictionary
    .groupby(
        "Category"
    )
    .size()
    .reset_index(
        name="Columns"
    )
    .sort_values(
        [
            "Columns",
            "Category"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


print("=" * 80)
print("CELL 6 - DISPLAY THE BT DATA DICTIONARY")
print("=" * 80)

display(
    bt_data_dictionary
)

print("\nBT column-category summary:")

display(
    bt_category_summary
)


assert len(bt_data_dictionary) == 34

assert bt_data_dictionary[
    "Column"
].is_unique

assert bt_data_dictionary[
    [
        "Category",
        "Expected Type",
        "Definition",
        "Later Role"
    ]
].notna().all().all()


print(
    "\nAll cleaned BT columns were documented "
    "successfully."
)

CELL 6 - DISPLAY THE BT DATA DICTIONARY


,Column Number,Column,Category,Expected Type,Definition,Later Role
0,1,location_id,Identifier,String,Stable identifier created from the BT grid coo...,POI join and location analysis
1,2,xbin,Location,Numeric,UTM easting coordinate of the BT grid location.,Location and interpretation
2,3,ybin,Location,Numeric,UTM northing coordinate of the BT grid location.,Location and interpretation
3,4,averagedownlinkthroughput,Throughput,Numeric,Average downlink data throughput recorded for ...,Candidate network measurement
4,5,averageuplinkthroughput,Throughput,Numeric,Average uplink data throughput recorded for th...,Candidate network measurement
5,6,csfallbackattempts,Connection Activity,Numeric,Number of circuit-switched fallback attempts.,Candidate network measurement
6,7,irathandoverattempts,Handovers,Numeric,Number of inter-radio-access-technology handov...,Candidate network measurement
7,8,linearaveragersrp,Signal,Numeric,Average Reference Signal Received Power value.,Candidate network measurement
8,9,linearaveragersrq,Signal,Numeric,Average Reference Signal Received Quality value.,Candidate network measurement
9,10,minutesofuse,Usage,Numeric,Total recorded minutes of network use.,Candidate network measurement



BT column-category summary:


,Category,Columns
0,Handovers,7
1,Connection Reliability,3
2,E-RAB Reliability,3
3,Mobility,3
4,Connection Activity,2
5,Data Duration,2
6,Data Volume,2
7,Location,2
8,Signal,2
9,Throughput,2



All cleaned BT columns were documented successfully.


## What Cell 6 Does

This cell displays a data dictionary for the cleaned BT columns.

The table gives each column a category, expected data type, short definition and later purpose.

The original BT measurements are kept in their supplied form. The identifier, coordinates and date are retained so that results can be linked to locations and dates.

In [8]:
# ================================================================
# CELL 7 - INSPECT ALL DAILY-FILE HEADERS
# ================================================================

header_inspection_records = []


for file_number, file_path in enumerate(
    daily_csv_files,
    start=1
):

    try:

        header_frame = pd.read_csv(
            file_path,
            nrows=0
        )

        observed_raw_columns = list(
            header_frame.columns
        )

        missing_raw_columns = [
            column_name
            for column_name
            in RAW_BT_COLUMNS
            if column_name not in observed_raw_columns
        ]

        unexpected_raw_columns = [
            column_name
            for column_name
            in observed_raw_columns
            if column_name not in RAW_BT_COLUMNS
        ]

        duplicate_header_columns = (
            len(observed_raw_columns)
            -
            len(set(observed_raw_columns))
        )

        correct_column_order = (
            observed_raw_columns
            == RAW_BT_COLUMNS
        )

        correct_column_count = (
            len(observed_raw_columns)
            == len(RAW_BT_COLUMNS)
        )

        valid_header = (
            correct_column_count
            and correct_column_order
            and len(missing_raw_columns) == 0
            and len(unexpected_raw_columns) == 0
            and duplicate_header_columns == 0
        )

        header_error = None

    except Exception as error:

        observed_raw_columns = []
        missing_raw_columns = RAW_BT_COLUMNS.copy()
        unexpected_raw_columns = []
        duplicate_header_columns = 0
        correct_column_order = False
        correct_column_count = False
        valid_header = False
        header_error = str(error)


    header_inspection_records.append(
        {
            "File Number": file_number,
            "Filename": file_path.name,
            "Observed Columns": len(
                observed_raw_columns
            ),
            "Expected Columns": len(
                RAW_BT_COLUMNS
            ),
            "Correct Column Count": (
                correct_column_count
            ),
            "Correct Column Order": (
                correct_column_order
            ),
            "Missing Columns": len(
                missing_raw_columns
            ),
            "Unexpected Columns": len(
                unexpected_raw_columns
            ),
            "Duplicate Columns": (
                duplicate_header_columns
            ),
            "Valid Header": valid_header,
            "Header Error": header_error
        }
    )


daily_header_audit = pd.DataFrame(
    header_inspection_records
)


valid_header_files = int(
    daily_header_audit[
        "Valid Header"
    ].sum()
)

invalid_header_files = int(
    (
        ~daily_header_audit[
            "Valid Header"
        ]
    ).sum()
)


print("=" * 80)
print("CELL 7 - INSPECT ALL DAILY-FILE HEADERS")
print("=" * 80)

header_validation_summary = pd.DataFrame(
    {
        "Check": [
            "Daily files inspected",
            "Valid headers",
            "Invalid headers",
            "Expected raw columns",
            "Files with missing columns",
            "Files with unexpected columns",
            "Files with duplicate columns",
            "Files with incorrect column order"
        ],
        "Result": [
            len(
                daily_header_audit
            ),
            valid_header_files,
            invalid_header_files,
            len(
                RAW_BT_COLUMNS
            ),
            int(
                (
                    daily_header_audit[
                        "Missing Columns"
                    ] > 0
                ).sum()
            ),
            int(
                (
                    daily_header_audit[
                        "Unexpected Columns"
                    ] > 0
                ).sum()
            ),
            int(
                (
                    daily_header_audit[
                        "Duplicate Columns"
                    ] > 0
                ).sum()
            ),
            int(
                (
                    ~daily_header_audit[
                        "Correct Column Order"
                    ]
                ).sum()
            )
        ]
    }
)

display(
    header_validation_summary
)


if invalid_header_files > 0:

    print("\nFiles with invalid headers:")

    display(
        daily_header_audit.loc[
            ~daily_header_audit[
                "Valid Header"
            ]
        ]
    )


assert len(
    daily_header_audit
) == len(
    daily_csv_files
)

assert valid_header_files > 0


print(
    "\nThe daily-file header inspection "
    "was completed successfully."
)

CELL 7 - INSPECT ALL DAILY-FILE HEADERS


,Check,Result
0,Daily files inspected,95
1,Valid headers,95
2,Invalid headers,0
3,Expected raw columns,33
4,Files with missing columns,0
5,Files with unexpected columns,0
6,Files with duplicate columns,0
7,Files with incorrect column order,0



The daily-file header inspection was completed successfully.


## What Cell 7 Does

This cell checks the header of every daily CSV file without loading the full data.

Each header is compared with the expected 33-column BT structure. The checks identify missing columns, unexpected columns, duplicated names or changes in column order.

This ensures that only files with the correct structure continue to the cleaning stage.

In [9]:
# ================================================================
# CELL 8 - COUNT ROWS AND CLASSIFY DAILY FILES
# ================================================================

row_count_start_time = time.time()

daily_file_audit_records = []


def count_csv_data_rows(
    file_path
):
    """
    Count CSV data rows without loading the complete file
    into a pandas DataFrame.
    """

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as csv_file:

        row_count = sum(
            1
            for _ in csv_file
        )

    return max(
        row_count - 1,
        0
    )


for file_number, file_path in enumerate(
    daily_csv_files,
    start=1
):

    header_row = daily_header_audit.loc[
        daily_header_audit[
            "Filename"
        ] == file_path.name
    ].iloc[0]

    try:

        observation_count = count_csv_data_rows(
            file_path
        )

        row_count_error = None

    except Exception as error:

        observation_count = 0
        row_count_error = str(error)


    valid_header = bool(
        header_row[
            "Valid Header"
        ]
    )

    populated_file = (
        observation_count > 0
    )

    usable_file = (
        valid_header
        and populated_file
        and row_count_error is None
    )


    if row_count_error is not None:

        exclusion_reason = (
            f"Row-count error: "
            f"{row_count_error}"
        )

    elif not valid_header:

        exclusion_reason = (
            "Invalid BT header"
        )

    elif not populated_file:

        exclusion_reason = (
            "Header-only file with no observations"
        )

    else:

        exclusion_reason = None


    daily_file_audit_records.append(
        {
            "File Number": file_number,
            "Filename": file_path.name,
            "File Path": file_path,
            "File Size (MB)": round(
                file_path.stat().st_size
                / 1_000_000,
                2
            ),
            "Rows": int(
                observation_count
            ),
            "Valid Header": valid_header,
            "Populated": populated_file,
            "Usable": usable_file,
            "Exclusion Reason": (
                exclusion_reason
            ),
            "Row Count Error": (
                row_count_error
            )
        }
    )


daily_file_audit = pd.DataFrame(
    daily_file_audit_records
)


supplied_daily_files = len(
    daily_file_audit
)

populated_daily_files = int(
    daily_file_audit[
        "Populated"
    ].sum()
)

usable_daily_files = int(
    daily_file_audit[
        "Usable"
    ].sum()
)

excluded_daily_files = (
    supplied_daily_files
    -
    usable_daily_files
)

total_usable_rows = int(
    daily_file_audit.loc[
        daily_file_audit[
            "Usable"
        ],
        "Rows"
    ].sum()
)


print("=" * 80)
print("CELL 8 - COUNT ROWS AND CLASSIFY DAILY FILES")
print("=" * 80)

daily_file_classification_summary = pd.DataFrame(
    {
        "Result": [
            "Supplied daily files",
            "Populated daily files",
            "Usable daily files",
            "Excluded daily files",
            "Rows across usable files",
            "Processing time (minutes)"
        ],
        "Value": [
            supplied_daily_files,
            populated_daily_files,
            usable_daily_files,
            excluded_daily_files,
            total_usable_rows,
            round(
                (
                    time.time()
                    - row_count_start_time
                ) / 60,
                2
            )
        ]
    }
)

display(
    daily_file_classification_summary
)

print("\nDaily-file audit:")

display(
    daily_file_audit[
        [
            "File Number",
            "Filename",
            "File Size (MB)",
            "Rows",
            "Valid Header",
            "Populated",
            "Usable",
            "Exclusion Reason"
        ]
    ]
)


assert supplied_daily_files == len(
    daily_csv_files
)

assert usable_daily_files > 0

assert total_usable_rows > 0


print(
    "\nAll supplied daily files were counted "
    "and classified successfully."
)

CELL 8 - COUNT ROWS AND CLASSIFY DAILY FILES


,Result,Value
0,Supplied daily files,95.000000
1,Populated daily files,90.000000
2,Usable daily files,90.000000
3,Excluded daily files,5.000000
4,Rows across usable files,"119,267,039.000000"
5,Processing time (minutes),0.880000



Daily-file audit:


,File Number,Filename,File Size (MB),Rows,Valid Header,Populated,Usable,Exclusion Reason
0,1,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,175.760000,1322741,True,True,True,None
1,2,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,176.370000,1325263,True,True,True,None
2,3,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,181.810000,1371738,True,True,True,None
3,4,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,162.210000,1223418,True,True,True,None
4,5,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,171.240000,1294137,True,True,True,None
5,6,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,0.000000,0,True,False,False,Header-only file with no observations
6,7,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,162.210000,1215695,True,True,True,None
7,8,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,170.290000,1276920,True,True,True,None
8,9,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,169.620000,1267340,True,True,True,None
9,10,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,164.330000,1232265,True,True,True,None



All supplied daily files were counted and classified successfully.


## What Cell 8 Does

This cell counts the observations in every discovered daily file.

A file is classified as usable when:

- its header matches the expected BT structure;
- it contains at least one observation;
- its row count can be read successfully.

Empty or invalid files are recorded with an exclusion reason rather than being silently ignored.

In [10]:
# ================================================================
# CELL 9 - CONFIRM USABLE DAILY PARTITIONS
# ================================================================

usable_daily_file_audit = (
    daily_file_audit.loc[
        daily_file_audit[
            "Usable"
        ]
    ]
    .copy()
    .sort_values(
        "Filename"
    )
    .reset_index(
        drop=True
    )
)


excluded_daily_file_audit = (
    daily_file_audit.loc[
        ~daily_file_audit[
            "Usable"
        ]
    ]
    .copy()
    .sort_values(
        "Filename"
    )
    .reset_index(
        drop=True
    )
)


usable_daily_files = [
    Path(
        file_path
    )
    for file_path
    in usable_daily_file_audit[
        "File Path"
    ]
]


usable_daily_file_audit[
    "Partition Number"
] = range(
    1,
    len(
        usable_daily_file_audit
    ) + 1
)


usable_daily_file_audit = (
    usable_daily_file_audit[
        [
            "Partition Number",
            "File Number",
            "Filename",
            "File Path",
            "File Size (MB)",
            "Rows",
            "Valid Header",
            "Populated",
            "Usable"
        ]
    ]
)


usable_filenames = [
    file_path.name
    for file_path
    in usable_daily_files
]


unique_usable_filenames = (
    len(
        usable_filenames
    )
    ==
    len(
        set(
            usable_filenames
        )
    )
)


chronological_filename_order = (
    usable_filenames
    ==
    sorted(
        usable_filenames
    )
)


all_usable_rows_positive = bool(
    (
        usable_daily_file_audit[
            "Rows"
        ] > 0
    ).all()
)


all_usable_headers_valid = bool(
    usable_daily_file_audit[
        "Valid Header"
    ].all()
)


print("=" * 80)
print("CELL 9 - CONFIRM USABLE DAILY PARTITIONS")
print("=" * 80)

usable_partition_validation = pd.DataFrame(
    {
        "Check": [
            "Usable daily partitions",
            "Excluded daily files",
            "Unique usable filenames",
            "Chronological filename order",
            "All usable row counts positive",
            "All usable headers valid",
            "Total usable observations"
        ],
        "Result": [
            len(
                usable_daily_files
            ),
            len(
                excluded_daily_file_audit
            ),
            unique_usable_filenames,
            chronological_filename_order,
            all_usable_rows_positive,
            all_usable_headers_valid,
            int(
                usable_daily_file_audit[
                    "Rows"
                ].sum()
            )
        ]
    }
)

display(
    usable_partition_validation
)

print("\nUsable daily partitions:")

display(
    usable_daily_file_audit[
        [
            "Partition Number",
            "Filename",
            "Rows",
            "File Size (MB)"
        ]
    ]
)


if len(
    excluded_daily_file_audit
) > 0:

    print("\nExcluded daily files:")

    display(
        excluded_daily_file_audit[
            [
                "File Number",
                "Filename",
                "Rows",
                "Valid Header",
                "Exclusion Reason"
            ]
        ]
    )


assert len(
    usable_daily_files
) == usable_daily_file_audit.shape[0]

assert unique_usable_filenames

assert chronological_filename_order

assert all_usable_rows_positive

assert all_usable_headers_valid


print(
    "\nThe official usable daily-partition list "
    "was created successfully."
)

CELL 9 - CONFIRM USABLE DAILY PARTITIONS


,Check,Result
0,Usable daily partitions,90
1,Excluded daily files,5
2,Unique usable filenames,True
3,Chronological filename order,True
4,All usable row counts positive,True
5,All usable headers valid,True
6,Total usable observations,119267039



Usable daily partitions:


,Partition Number,Filename,Rows,File Size (MB)
0,1,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1322741,175.760000
1,2,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1325263,176.370000
2,3,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1371738,181.810000
3,4,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1223418,162.210000
4,5,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1294137,171.240000
5,6,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1215695,162.210000
6,7,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1276920,170.290000
7,8,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1267340,169.620000
8,9,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1232265,164.330000
9,10,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1208420,161.150000



Excluded daily files:


,File Number,Filename,Rows,Valid Header,Exclusion Reason
0,6,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,0,True,Header-only file with no observations
1,16,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,0,True,Header-only file with no observations
2,21,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,0,True,Header-only file with no observations
3,94,GEO_Export_Neon_530k_5590k_668k_5650k_2026-07-...,0,True,Header-only file with no observations
4,95,GEO_Export_Neon_530k_5590k_668k_5650k_2026-07-...,0,True,Header-only file with no observations



The official usable daily-partition list was created successfully.


## What Cell 9 Does

This cell creates the official ordered list of usable daily files for the current run.

Every retained file has a valid header and at least one observation. The files are assigned partition numbers and placed in filename order.

All later cleaning and model-dataset stages use this automatically created list. If more valid files are added in the future, they will be included in the list when the notebook is rerun.

# 3. Data Cleaning

This section defines how the supplied BT columns are renamed and converted into a consistent format.

The cleaning process keeps all usable observations and does not replace missing network values or remove unusual measurements.

In [11]:
# ================================================================
# CELL 10 - DEFINE RAW-TO-CLEAN COLUMN RENAMING
# ================================================================

RAW_TO_CLEAN_COLUMN_MAP = {
    raw_column: clean_column
    for raw_column, clean_column
    in zip(
        RAW_BT_COLUMNS,
        ORIGINAL_BT_COLUMNS
    )
}


CLEAN_TO_RAW_COLUMN_MAP = {
    clean_column: raw_column
    for raw_column, clean_column
    in RAW_TO_CLEAN_COLUMN_MAP.items()
}


renaming_table = pd.DataFrame(
    {
        "Column Number": range(
            1,
            len(
                RAW_BT_COLUMNS
            ) + 1
        ),
        "Raw Column": RAW_BT_COLUMNS,
        "Clean Column": ORIGINAL_BT_COLUMNS
    }
)


raw_columns_unique = (
    len(
        RAW_TO_CLEAN_COLUMN_MAP
    )
    ==
    len(
        RAW_BT_COLUMNS
    )
)


clean_columns_unique = (
    len(
        set(
            RAW_TO_CLEAN_COLUMN_MAP.values()
        )
    )
    ==
    len(
        ORIGINAL_BT_COLUMNS
    )
)


prefix_removed_from_all_columns = all(
    not column_name.startswith(
        SOURCE_TABLE_PREFIX
    )
    for column_name
    in RAW_TO_CLEAN_COLUMN_MAP.values()
)


print("=" * 80)
print("CELL 10 - DEFINE RAW-TO-CLEAN COLUMN RENAMING")
print("=" * 80)

display(
    renaming_table
)


renaming_validation = pd.DataFrame(
    {
        "Check": [
            "Raw columns mapped",
            "Clean columns produced",
            "Raw names unique",
            "Clean names unique",
            "BT prefix removed from clean names"
        ],
        "Result": [
            len(
                RAW_TO_CLEAN_COLUMN_MAP
            ),
            len(
                set(
                    RAW_TO_CLEAN_COLUMN_MAP.values()
                )
            ),
            raw_columns_unique,
            clean_columns_unique,
            prefix_removed_from_all_columns
        ]
    }
)

display(
    renaming_validation
)


assert len(
    RAW_TO_CLEAN_COLUMN_MAP
) == 33

assert raw_columns_unique

assert clean_columns_unique

assert prefix_removed_from_all_columns

assert list(
    RAW_TO_CLEAN_COLUMN_MAP.values()
) == ORIGINAL_BT_COLUMNS


print(
    "\nThe raw-to-clean column mapping was "
    "defined successfully."
)

CELL 10 - DEFINE RAW-TO-CLEAN COLUMN RENAMING


,Column Number,Raw Column,Clean Column
0,1,b_geolte_daily_bin.xbin,xbin
1,2,b_geolte_daily_bin.ybin,ybin
2,3,b_geolte_daily_bin.averagedownlinkthroughput,averagedownlinkthroughput
3,4,b_geolte_daily_bin.averageuplinkthroughput,averageuplinkthroughput
4,5,b_geolte_daily_bin.csfallbackattempts,csfallbackattempts
5,6,b_geolte_daily_bin.irathandoverattempts,irathandoverattempts
6,7,b_geolte_daily_bin.linearaveragersrp,linearaveragersrp
7,8,b_geolte_daily_bin.linearaveragersrq,linearaveragersrq
8,9,b_geolte_daily_bin.minutesofuse,minutesofuse
9,10,b_geolte_daily_bin.numberofconnections,numberofconnections


,Check,Result
0,Raw columns mapped,33
1,Clean columns produced,33
2,Raw names unique,True
3,Clean names unique,True
4,BT prefix removed from clean names,True



The raw-to-clean column mapping was defined successfully.


## What Cell 10 Does

This cell defines the mapping between the original BT column names and the shorter names used in the project.

The common source-table prefix is removed, while the meaning and order of all 33 supplied fields are preserved.

A reverse mapping is also kept so that each cleaned column can be linked back to its original source name.

In [12]:
# ================================================================
# CELL 11 - DEFINE CLEANED DATA TYPES
# ================================================================

CLEANED_STRING_COLUMNS = [
    "location_id"
]


CLEANED_DATETIME_COLUMNS = [
    "en_dt"
]


CLEANED_NUMERIC_COLUMNS = [
    column_name
    for column_name
    in ORIGINAL_BT_COLUMNS
    if column_name not in CLEANED_DATETIME_COLUMNS
]


CLEANED_FLOAT_COLUMNS = (
    CLEANED_NUMERIC_COLUMNS.copy()
)


CLEANED_TYPE_ROLES = {
    column_name: (
        "string"
        if column_name in CLEANED_STRING_COLUMNS
        else
        "datetime64[ns]"
        if column_name in CLEANED_DATETIME_COLUMNS
        else
        "float64"
    )
    for column_name
    in CLEANED_BT_COLUMNS
}


cleaned_type_table = pd.DataFrame(
    {
        "Column Number": range(
            1,
            len(
                CLEANED_BT_COLUMNS
            ) + 1
        ),
        "Column": CLEANED_BT_COLUMNS,
        "Target Type": [
            CLEANED_TYPE_ROLES[
                column_name
            ]
            for column_name
            in CLEANED_BT_COLUMNS
        ],
        "Allows Missing Values": [
            True
            for _ in CLEANED_BT_COLUMNS
        ]
    }
)


print("=" * 80)
print("CELL 11 - DEFINE CLEANED DATA TYPES")
print("=" * 80)

display(
    cleaned_type_table
)


cleaned_type_summary = pd.DataFrame(
    {
        "Type Group": [
            "String identifiers",
            "Numeric BT fields",
            "Datetime fields",
            "Total cleaned columns"
        ],
        "Columns": [
            len(
                CLEANED_STRING_COLUMNS
            ),
            len(
                CLEANED_NUMERIC_COLUMNS
            ),
            len(
                CLEANED_DATETIME_COLUMNS
            ),
            len(
                CLEANED_BT_COLUMNS
            )
        ]
    }
)

display(
    cleaned_type_summary
)


assert len(
    CLEANED_STRING_COLUMNS
) == 1

assert len(
    CLEANED_NUMERIC_COLUMNS
) == 32

assert len(
    CLEANED_DATETIME_COLUMNS
) == 1

assert (
    len(
        CLEANED_STRING_COLUMNS
    )
    +
    len(
        CLEANED_NUMERIC_COLUMNS
    )
    +
    len(
        CLEANED_DATETIME_COLUMNS
    )
) == len(
    CLEANED_BT_COLUMNS
)

assert set(
    CLEANED_TYPE_ROLES
) == set(
    CLEANED_BT_COLUMNS
)


print(
    "\nThe cleaned data-type structure was "
    "defined successfully."
)

CELL 11 - DEFINE CLEANED DATA TYPES


,Column Number,Column,Target Type,Allows Missing Values
0,1,location_id,string,True
1,2,xbin,float64,True
2,3,ybin,float64,True
3,4,averagedownlinkthroughput,float64,True
4,5,averageuplinkthroughput,float64,True
5,6,csfallbackattempts,float64,True
6,7,irathandoverattempts,float64,True
7,8,linearaveragersrp,float64,True
8,9,linearaveragersrq,float64,True
9,10,minutesofuse,float64,True


,Type Group,Columns
0,String identifiers,1
1,Numeric BT fields,32
2,Datetime fields,1
3,Total cleaned columns,34



The cleaned data-type structure was defined successfully.


## What Cell 11 Does

This cell defines the required data type for every cleaned column.

`location_id` is stored as text, the date is stored as a datetime value, and the remaining supplied fields are stored as numeric values.

Missing network values remain allowed and are not filled during this notebook.

In [13]:
# ================================================================
# CELL 12 - CREATE THE BT CLEANING FUNCTION
# ================================================================

def create_location_id(
    x_values,
    y_values
):
    """
    Create a stable location identifier from BT grid coordinates.
    """

    x_numeric = pd.to_numeric(
        x_values,
        errors="coerce"
    )

    y_numeric = pd.to_numeric(
        y_values,
        errors="coerce"
    )


    valid_coordinates = (
        x_numeric.notna()
        &
        y_numeric.notna()
        &
        np.isfinite(
            x_numeric
        )
        &
        np.isfinite(
            y_numeric
        )
    )


    location_ids = pd.Series(
        pd.NA,
        index=x_values.index,
        dtype="string"
    )


    location_ids.loc[
        valid_coordinates
    ] = (
        x_numeric.loc[
            valid_coordinates
        ]
        .round()
        .astype(
            "int64"
        )
        .astype(
            "string"
        )
        +
        "_"
        +
        y_numeric.loc[
            valid_coordinates
        ]
        .round()
        .astype(
            "int64"
        )
        .astype(
            "string"
        )
    )


    return location_ids


def clean_bt_daily_dataframe(
    raw_dataframe
):
    """
    Validate and clean one complete BT daily DataFrame.
    """

    if not isinstance(
        raw_dataframe,
        pd.DataFrame
    ):

        raise TypeError(
            "raw_dataframe must be a pandas DataFrame."
        )


    observed_columns = list(
        raw_dataframe.columns
    )


    if observed_columns != RAW_BT_COLUMNS:

        missing_columns = [
            column_name
            for column_name
            in RAW_BT_COLUMNS
            if column_name not in observed_columns
        ]

        unexpected_columns = [
            column_name
            for column_name
            in observed_columns
            if column_name not in RAW_BT_COLUMNS
        ]

        raise ValueError(
            "The raw BT schema does not match the expected "
            f"33-column structure. Missing columns: "
            f"{missing_columns}. Unexpected columns: "
            f"{unexpected_columns}."
        )


    cleaned_dataframe = (
        raw_dataframe
        .rename(
            columns=RAW_TO_CLEAN_COLUMN_MAP
        )
        .copy()
    )


    for column_name in CLEANED_NUMERIC_COLUMNS:

        cleaned_dataframe[
            column_name
        ] = pd.to_numeric(
            cleaned_dataframe[
                column_name
            ],
            errors="coerce"
        )


    cleaned_dataframe[
        CLEANED_NUMERIC_COLUMNS
    ] = cleaned_dataframe[
        CLEANED_NUMERIC_COLUMNS
    ].replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )


    for column_name in CLEANED_FLOAT_COLUMNS:

        cleaned_dataframe[
            column_name
        ] = cleaned_dataframe[
            column_name
        ].astype(
            "float64"
        )


    cleaned_dataframe[
        "en_dt"
    ] = pd.to_datetime(
        cleaned_dataframe[
            "en_dt"
        ],
        errors="coerce"
    )


    cleaned_dataframe.insert(
        0,
        "location_id",
        create_location_id(
            cleaned_dataframe[
                "xbin"
            ],
            cleaned_dataframe[
                "ybin"
            ]
        )
    )


    cleaned_dataframe = cleaned_dataframe[
        CLEANED_BT_COLUMNS
    ]


    if len(
        cleaned_dataframe
    ) != len(
        raw_dataframe
    ):

        raise AssertionError(
            "The cleaning process changed the number of rows."
        )


    if list(
        cleaned_dataframe.columns
    ) != CLEANED_BT_COLUMNS:

        raise AssertionError(
            "The cleaned column order is incorrect."
        )


    if cleaned_dataframe.shape[1] != 34:

        raise AssertionError(
            "The cleaned DataFrame must contain 34 columns."
        )


    return cleaned_dataframe


print("=" * 80)
print("CELL 12 - CREATE THE BT CLEANING FUNCTION")
print("=" * 80)


cleaning_function_summary = pd.DataFrame(
    {
        "Cleaning Rule": [
            "Validate the exact raw BT schema",
            "Remove the common BT table prefix",
            "Convert coordinates and measurements to numeric",
            "Replace numeric infinities with missing values",
            "Convert the date field to datetime",
            "Create location_id from xbin and ybin",
            "Preserve missing network measurements",
            "Preserve the original observation count",
            "Return the official cleaned column order"
        ],
        "Applied": [
            True,
            True,
            True,
            True,
            True,
            True,
            True,
            True,
            True
        ]
    }
)

display(
    cleaning_function_summary
)


assert callable(
    create_location_id
)

assert callable(
    clean_bt_daily_dataframe
)


print(
    "\nThe reusable BT cleaning function was "
    "created successfully."
)

CELL 12 - CREATE THE BT CLEANING FUNCTION


,Cleaning Rule,Applied
0,Validate the exact raw BT schema,True
1,Remove the common BT table prefix,True
2,Convert coordinates and measurements to numeric,True
3,Replace numeric infinities with missing values,True
4,Convert the date field to datetime,True
5,Create location_id from xbin and ybin,True
6,Preserve missing network measurements,True
7,Preserve the original observation count,True
8,Return the official cleaned column order,True



The reusable BT cleaning function was created successfully.


## What Cell 12 Does

This cell creates the cleaning function used for every usable daily file.

The function:

- checks the complete 33-column source structure;
- removes the common prefix from the column names;
- converts the coordinates and measurements to numeric values;
- converts the date to a datetime value;
- replaces infinite numeric values with missing values;
- creates `location_id` from `xbin` and `ybin`;
- returns the columns in the required order.

The function does not remove observations, fill missing network values or create model-based measurements.

# 4. Complete Cleaning Test

This section tests the full cleaning process on one complete populated daily file.

Using a complete file confirms that the process works correctly before it is applied to every usable partition.

In [14]:
# ================================================================
# CELL 13 - SELECT ONE POPULATED BT DAILY FILE
# ================================================================

TEST_PARTITION_NUMBER = 1


test_partition_record = (
    usable_daily_file_audit.loc[
        usable_daily_file_audit[
            "Partition Number"
        ] == TEST_PARTITION_NUMBER
    ]
    .iloc[0]
)


TEST_RAW_FILE = Path(
    test_partition_record[
        "File Path"
    ]
)


TEST_EXPECTED_ROWS = int(
    test_partition_record[
        "Rows"
    ]
)


TEST_FILE_SIZE_MB = float(
    test_partition_record[
        "File Size (MB)"
    ]
)


print("=" * 80)
print("CELL 13 - SELECT ONE POPULATED BT DAILY FILE")
print("=" * 80)


test_file_summary = pd.DataFrame(
    {
        "Property": [
            "Partition number",
            "Filename",
            "Expected rows",
            "File size (MB)",
            "Valid header",
            "Populated",
            "Usable"
        ],
        "Value": [
            TEST_PARTITION_NUMBER,
            TEST_RAW_FILE.name,
            TEST_EXPECTED_ROWS,
            TEST_FILE_SIZE_MB,
            bool(
                test_partition_record[
                    "Valid Header"
                ]
            ),
            bool(
                test_partition_record[
                    "Populated"
                ]
            ),
            bool(
                test_partition_record[
                    "Usable"
                ]
            )
        ]
    }
)

display(
    test_file_summary
)


assert TEST_RAW_FILE.exists()

assert TEST_EXPECTED_ROWS > 0

assert bool(
    test_partition_record[
        "Valid Header"
    ]
)

assert bool(
    test_partition_record[
        "Usable"
    ]
)


print(
    "\nThe complete populated test partition "
    "was selected successfully."
)

CELL 13 - SELECT ONE POPULATED BT DAILY FILE


,Property,Value
0,Partition number,1
1,Filename,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...
2,Expected rows,1322741
3,File size (MB),175.760000
4,Valid header,True
5,Populated,True
6,Usable,True



The complete populated test partition was selected successfully.


## What Cell 13 Does

This cell selects one usable daily file for the cleaning test.

The selected file has already passed the header and row-count checks. Its partition number, filename, expected observations and file size are displayed.

The entire file is used rather than a smaller sample.

In [15]:
# ================================================================
# CELL 14 - LOAD AND INSPECT THE COMPLETE TEST FILE
# ================================================================

test_file_load_start_time = time.time()


test_raw_dataframe = pd.read_csv(
    TEST_RAW_FILE,
    low_memory=False
)


test_file_load_seconds = (
    time.time()
    - test_file_load_start_time
)


test_raw_missing_summary = pd.DataFrame(
    {
        "Raw Column": test_raw_dataframe.columns,
        "Missing Values": [
            int(
                test_raw_dataframe[
                    column_name
                ].isna().sum()
            )
            for column_name
            in test_raw_dataframe.columns
        ]
    }
)


raw_xbin_column = CLEAN_TO_RAW_COLUMN_MAP[
    "xbin"
]

raw_ybin_column = CLEAN_TO_RAW_COLUMN_MAP[
    "ybin"
]

raw_date_column = CLEAN_TO_RAW_COLUMN_MAP[
    "en_dt"
]


raw_test_dates = pd.to_datetime(
    test_raw_dataframe[
        raw_date_column
    ],
    errors="coerce"
)


test_raw_inspection = pd.DataFrame(
    {
        "Check": [
            "Rows loaded",
            "Expected rows",
            "Raw columns",
            "Expected raw columns",
            "Missing date values",
            "Minimum date",
            "Maximum date",
            "Minimum xbin",
            "Maximum xbin",
            "Minimum ybin",
            "Maximum ybin",
            "Memory use (MB)",
            "Loading time (seconds)"
        ],
        "Result": [
            len(
                test_raw_dataframe
            ),
            TEST_EXPECTED_ROWS,
            test_raw_dataframe.shape[1],
            len(
                RAW_BT_COLUMNS
            ),
            int(
                raw_test_dates.isna().sum()
            ),
            raw_test_dates.min(),
            raw_test_dates.max(),
            pd.to_numeric(
                test_raw_dataframe[
                    raw_xbin_column
                ],
                errors="coerce"
            ).min(),
            pd.to_numeric(
                test_raw_dataframe[
                    raw_xbin_column
                ],
                errors="coerce"
            ).max(),
            pd.to_numeric(
                test_raw_dataframe[
                    raw_ybin_column
                ],
                errors="coerce"
            ).min(),
            pd.to_numeric(
                test_raw_dataframe[
                    raw_ybin_column
                ],
                errors="coerce"
            ).max(),
            round(
                test_raw_dataframe.memory_usage(
                    deep=True
                ).sum()
                / 1_000_000,
                2
            ),
            round(
                test_file_load_seconds,
                2
            )
        ]
    }
)


print("=" * 80)
print("CELL 14 - LOAD AND INSPECT THE COMPLETE TEST FILE")
print("=" * 80)

display(
    test_raw_inspection
)

print("\nRaw data types:")

display(
    test_raw_dataframe.dtypes.to_frame(
        name="Raw Data Type"
    )
)

print("\nFirst five raw observations:")

display(
    test_raw_dataframe.head()
)

print("\nRaw missing-value summary:")

display(
    test_raw_missing_summary
)


assert len(
    test_raw_dataframe
) == TEST_EXPECTED_ROWS

assert list(
    test_raw_dataframe.columns
) == RAW_BT_COLUMNS

assert test_raw_dataframe.shape[1] == 33


print(
    "\nThe complete test partition was loaded "
    "and inspected successfully."
)

CELL 14 - LOAD AND INSPECT THE COMPLETE TEST FILE


,Check,Result
0,Rows loaded,1322741
1,Expected rows,1322741
2,Raw columns,33
3,Expected raw columns,33
4,Missing date values,0
5,Minimum date,2026-04-04 00:00:00
6,Maximum date,2026-04-04 00:00:00
7,Minimum xbin,530000
8,Maximum xbin,667950
9,Minimum ybin,5590000



Raw data types:


,Raw Data Type
b_geolte_daily_bin.xbin,int64
b_geolte_daily_bin.ybin,int64
b_geolte_daily_bin.averagedownlinkthroughput,float64
b_geolte_daily_bin.averageuplinkthroughput,float64
b_geolte_daily_bin.csfallbackattempts,int64
b_geolte_daily_bin.irathandoverattempts,float64
b_geolte_daily_bin.linearaveragersrp,float64
b_geolte_daily_bin.linearaveragersrq,float64
b_geolte_daily_bin.minutesofuse,float64
b_geolte_daily_bin.numberofconnections,int64



First five raw observations:


,b_geolte_daily_bin.xbin,b_geolte_daily_bin.ybin,b_geolte_daily_bin.averagedownlinkthroughput,b_geolte_daily_bin.averageuplinkthroughput,b_geolte_daily_bin.csfallbackattempts,b_geolte_daily_bin.irathandoverattempts,b_geolte_daily_bin.linearaveragersrp,b_geolte_daily_bin.linearaveragersrq,b_geolte_daily_bin.minutesofuse,b_geolte_daily_bin.numberofconnections,b_geolte_daily_bin.pedestrianminutesofuse,b_geolte_daily_bin.s1handoverattempts,b_geolte_daily_bin.s1handoverfailures,b_geolte_daily_bin.s1handoversuccesses,b_geolte_daily_bin.stationaryminutesofuse,b_geolte_daily_bin.totalconnectionblocks,b_geolte_daily_bin.totalconnectiondrops,b_geolte_daily_bin.totalconnectionnormalreleases,b_geolte_daily_bin.totaldownlinkdataduration,b_geolte_daily_bin.totaldownlinkvolume,b_geolte_daily_bin.totalerabblocks,b_geolte_daily_bin.totalerabdrops,b_geolte_daily_bin.totalerabnormalreleases,b_geolte_daily_bin.totaluplinkdataduration,b_geolte_daily_bin.totaluplinkvolume,b_geolte_daily_bin.vehicularminutesofuse,b_geolte_daily_bin.voiceminutesofuse,b_geolte_daily_bin.x2handoverattempts,b_geolte_daily_bin.x2handoverfailures,b_geolte_daily_bin.x2handoversuccesses,b_geolte_daily_bin.indoorminutesofuse,b_geolte_daily_bin.outdoorminutesofuse,b_geolte_daily_bin.en_dt
0,530050,5647100,43.280000,11.540000,0,NaN,-101.950000,-13.550000,0.490000,5,0.000000,0,0,0,0.000000,0,0,2,0.070000,7,0,0,4,0.120000,3,0.490000,0.000000,2,1,1,0.000000,0.490000,2026-04-04
1,530100,5606050,33.870000,7.120000,0,NaN,-114.080000,-12.730000,0.780000,7,0.000000,0,0,0,0.000000,0,0,7,0.150000,8,0,0,4,0.510000,6,0.780000,0.000000,0,0,0,0.000000,0.780000,2026-04-04
2,530100,5618850,257.350000,67.570000,0,NaN,NaN,NaN,0.590000,1,0.000000,0,0,0,0.590000,0,0,1,0.140000,35,0,0,1,0.220000,15,0.000000,0.000000,0,0,0,0.000000,0.000000,2026-04-04
3,530100,5631050,232.450000,40.910000,0,NaN,-96.420000,-10.890000,155.590000,556,5.690000,0,0,0,75.550000,0,0,350,110.360000,156520,1,0,593,223.740000,30986,74.360000,0.370000,100,0,100,0.360000,155.230000,2026-04-04
4,530100,5634250,133.700000,18.690000,0,NaN,-101.960000,-11.360000,38.960000,256,1.870000,8,0,8,23.730000,0,0,189,25.910000,12807,0,0,166,97.330000,2481,13.380000,1.350000,36,1,35,0.000000,38.960000,2026-04-04



Raw missing-value summary:


,Raw Column,Missing Values
0,b_geolte_daily_bin.xbin,0
1,b_geolte_daily_bin.ybin,0
2,b_geolte_daily_bin.averagedownlinkthroughput,0
3,b_geolte_daily_bin.averageuplinkthroughput,0
4,b_geolte_daily_bin.csfallbackattempts,0
5,b_geolte_daily_bin.irathandoverattempts,1322741
6,b_geolte_daily_bin.linearaveragersrp,185096
7,b_geolte_daily_bin.linearaveragersrq,185096
8,b_geolte_daily_bin.minutesofuse,0
9,b_geolte_daily_bin.numberofconnections,0



The complete test partition was loaded and inspected successfully.


## What Cell 14 Does

This cell loads and inspects the complete test file.

It reports the number of observations and columns, data types, missing values, date range, coordinate range, memory use and loading time.

The source row count and exact 33-column structure are checked before cleaning begins.

In [16]:
# ================================================================
# CELL 15 - APPLY THE CLEANING PROCESS
# ================================================================

test_cleaning_start_time = time.time()


test_cleaned_dataframe = clean_bt_daily_dataframe(
    test_raw_dataframe
)


test_cleaning_seconds = (
    time.time()
    - test_cleaning_start_time
)


test_cleaning_validation = pd.DataFrame(
    {
        "Check": [
            "Raw rows",
            "Cleaned rows",
            "Row difference",
            "Raw columns",
            "Cleaned columns",
            "Correct cleaned column order",
            "location_id is first column",
            "Missing location identifiers",
            "Missing x coordinates",
            "Missing y coordinates",
            "Invalid dates",
            "Cleaning time (seconds)"
        ],
        "Result": [
            len(
                test_raw_dataframe
            ),
            len(
                test_cleaned_dataframe
            ),
            (
                len(
                    test_cleaned_dataframe
                )
                -
                len(
                    test_raw_dataframe
                )
            ),
            test_raw_dataframe.shape[1],
            test_cleaned_dataframe.shape[1],
            (
                list(
                    test_cleaned_dataframe.columns
                )
                == CLEANED_BT_COLUMNS
            ),
            (
                test_cleaned_dataframe.columns[0]
                == "location_id"
            ),
            int(
                test_cleaned_dataframe[
                    "location_id"
                ].isna().sum()
            ),
            int(
                test_cleaned_dataframe[
                    "xbin"
                ].isna().sum()
            ),
            int(
                test_cleaned_dataframe[
                    "ybin"
                ].isna().sum()
            ),
            int(
                test_cleaned_dataframe[
                    "en_dt"
                ].isna().sum()
            ),
            round(
                test_cleaning_seconds,
                2
            )
        ]
    }
)


print("=" * 80)
print("CELL 15 - APPLY THE CLEANING PROCESS")
print("=" * 80)

display(
    test_cleaning_validation
)

print("\nCleaned data types:")

display(
    test_cleaned_dataframe.dtypes.to_frame(
        name="Cleaned Data Type"
    )
)

print("\nFirst five cleaned observations:")

display(
    test_cleaned_dataframe.head()
)


assert len(
    test_cleaned_dataframe
) == len(
    test_raw_dataframe
)

assert test_cleaned_dataframe.shape[1] == 34

assert list(
    test_cleaned_dataframe.columns
) == CLEANED_BT_COLUMNS

assert test_cleaned_dataframe.columns[0] == (
    "location_id"
)

assert pd.api.types.is_string_dtype(
    test_cleaned_dataframe[
        "location_id"
    ]
)

assert pd.api.types.is_datetime64_any_dtype(
    test_cleaned_dataframe[
        "en_dt"
    ]
)


print(
    "\nThe complete test partition was cleaned "
    "successfully."
)

CELL 15 - APPLY THE CLEANING PROCESS


,Check,Result
0,Raw rows,1322741
1,Cleaned rows,1322741
2,Row difference,0
3,Raw columns,33
4,Cleaned columns,34
5,Correct cleaned column order,True
6,location_id is first column,True
7,Missing location identifiers,0
8,Missing x coordinates,0
9,Missing y coordinates,0



Cleaned data types:


,Cleaned Data Type
location_id,string[python]
xbin,float64
ybin,float64
averagedownlinkthroughput,float64
averageuplinkthroughput,float64
csfallbackattempts,float64
irathandoverattempts,float64
linearaveragersrp,float64
linearaveragersrq,float64
minutesofuse,float64



First five cleaned observations:


,location_id,xbin,ybin,averagedownlinkthroughput,averageuplinkthroughput,csfallbackattempts,irathandoverattempts,linearaveragersrp,linearaveragersrq,minutesofuse,numberofconnections,pedestrianminutesofuse,s1handoverattempts,s1handoverfailures,s1handoversuccesses,stationaryminutesofuse,totalconnectionblocks,totalconnectiondrops,totalconnectionnormalreleases,totaldownlinkdataduration,totaldownlinkvolume,totalerabblocks,totalerabdrops,totalerabnormalreleases,totaluplinkdataduration,totaluplinkvolume,vehicularminutesofuse,voiceminutesofuse,x2handoverattempts,x2handoverfailures,x2handoversuccesses,indoorminutesofuse,outdoorminutesofuse,en_dt
0,530050_5647100,"530,050.000000","5,647,100.000000",43.280000,11.540000,0.000000,NaN,-101.950000,-13.550000,0.490000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.070000,7.000000,0.000000,0.000000,4.000000,0.120000,3.000000,0.490000,0.000000,2.000000,1.000000,1.000000,0.000000,0.490000,2026-04-04
1,530100_5606050,"530,100.000000","5,606,050.000000",33.870000,7.120000,0.000000,NaN,-114.080000,-12.730000,0.780000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.000000,0.150000,8.000000,0.000000,0.000000,4.000000,0.510000,6.000000,0.780000,0.000000,0.000000,0.000000,0.000000,0.000000,0.780000,2026-04-04
2,530100_5618850,"530,100.000000","5,618,850.000000",257.350000,67.570000,0.000000,NaN,NaN,NaN,0.590000,1.000000,0.000000,0.000000,0.000000,0.000000,0.590000,0.000000,0.000000,1.000000,0.140000,35.000000,0.000000,0.000000,1.000000,0.220000,15.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2026-04-04
3,530100_5631050,"530,100.000000","5,631,050.000000",232.450000,40.910000,0.000000,NaN,-96.420000,-10.890000,155.590000,556.000000,5.690000,0.000000,0.000000,0.000000,75.550000,0.000000,0.000000,350.000000,110.360000,"156,520.000000",1.000000,0.000000,593.000000,223.740000,"30,986.000000",74.360000,0.370000,100.000000,0.000000,100.000000,0.360000,155.230000,2026-04-04
4,530100_5634250,"530,100.000000","5,634,250.000000",133.700000,18.690000,0.000000,NaN,-101.960000,-11.360000,38.960000,256.000000,1.870000,8.000000,0.000000,8.000000,23.730000,0.000000,0.000000,189.000000,25.910000,"12,807.000000",0.000000,0.000000,166.000000,97.330000,"2,481.000000",13.380000,1.350000,36.000000,1.000000,35.000000,0.000000,38.960000,2026-04-04



The complete test partition was cleaned successfully.


## What Cell 15 Does

This cell applies the cleaning function to the complete test file.

The result contains the 33 cleaned BT columns and the newly created `location_id`.

The checks confirm that the number of observations has not changed, the column order is correct and the identifier and date use the intended data types.

In [17]:
# ================================================================
# CELL 16 - VALIDATE THE CLEANED TEST FILE
# ================================================================

test_numeric_values = (
    test_cleaned_dataframe[
        CLEANED_NUMERIC_COLUMNS
    ]
    .to_numpy(
        dtype="float64",
        copy=False
    )
)


test_infinite_values = int(
    np.isinf(
        test_numeric_values
    ).sum()
)


test_exact_duplicate_rows = int(
    test_cleaned_dataframe.duplicated(
        keep=False
    ).sum()
)


test_unique_locations = int(
    test_cleaned_dataframe[
        "location_id"
    ].nunique(
        dropna=True
    )
)


test_repeated_location_rows = int(
    test_cleaned_dataframe[
        "location_id"
    ].duplicated(
        keep=False
    ).sum()
)


location_coordinate_pairs = (
    test_cleaned_dataframe[
        [
            "location_id",
            "xbin",
            "ybin"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


conflicting_location_ids = int(
    (
        location_coordinate_pairs
        .groupby(
            "location_id",
            sort=False
        )
        .size()
        > 1
    ).sum()
)


test_cleaned_missing_summary = pd.DataFrame(
    {
        "Column": CLEANED_BT_COLUMNS,
        "Missing Values": [
            int(
                test_cleaned_dataframe[
                    column_name
                ].isna().sum()
            )
            for column_name
            in CLEANED_BT_COLUMNS
        ],
        "Missing Percentage": [
            (
                test_cleaned_dataframe[
                    column_name
                ].isna().mean()
                * 100
            )
            for column_name
            in CLEANED_BT_COLUMNS
        ]
    }
)


test_cleaned_validation = pd.DataFrame(
    {
        "Check": [
            "Rows retained",
            "Cleaned columns",
            "Unique locations",
            "Rows sharing a repeated location",
            "Exact duplicate rows",
            "Conflicting location-coordinate mappings",
            "Missing location identifiers",
            "Missing x coordinates",
            "Missing y coordinates",
            "Invalid dates",
            "Infinite numeric values",
            "Minimum date",
            "Maximum date",
            "Cleaned memory use (MB)"
        ],
        "Result": [
            len(
                test_cleaned_dataframe
            ),
            test_cleaned_dataframe.shape[1],
            test_unique_locations,
            test_repeated_location_rows,
            test_exact_duplicate_rows,
            conflicting_location_ids,
            int(
                test_cleaned_dataframe[
                    "location_id"
                ].isna().sum()
            ),
            int(
                test_cleaned_dataframe[
                    "xbin"
                ].isna().sum()
            ),
            int(
                test_cleaned_dataframe[
                    "ybin"
                ].isna().sum()
            ),
            int(
                test_cleaned_dataframe[
                    "en_dt"
                ].isna().sum()
            ),
            test_infinite_values,
            test_cleaned_dataframe[
                "en_dt"
            ].min(),
            test_cleaned_dataframe[
                "en_dt"
            ].max(),
            round(
                test_cleaned_dataframe.memory_usage(
                    deep=True
                ).sum()
                / 1_000_000,
                2
            )
        ]
    }
)


print("=" * 80)
print("CELL 16 - VALIDATE THE CLEANED TEST FILE")
print("=" * 80)

display(
    test_cleaned_validation
)

print("\nCleaned missing-value summary:")

display(
    test_cleaned_missing_summary
)


assert len(
    test_cleaned_dataframe
) == TEST_EXPECTED_ROWS

assert test_cleaned_dataframe.shape[1] == 34

assert conflicting_location_ids == 0

assert test_infinite_values == 0

assert list(
    test_cleaned_dataframe.columns
) == CLEANED_BT_COLUMNS


print(
    "\nThe complete cleaned test partition "
    "passed all required validation checks."
)

CELL 16 - VALIDATE THE CLEANED TEST FILE


,Check,Result
0,Rows retained,1322741
1,Cleaned columns,34
2,Unique locations,1322741
3,Rows sharing a repeated location,0
4,Exact duplicate rows,0
5,Conflicting location-coordinate mappings,0
6,Missing location identifiers,0
7,Missing x coordinates,0
8,Missing y coordinates,0
9,Invalid dates,0



Cleaned missing-value summary:


,Column,Missing Values,Missing Percentage
0,location_id,0,0.000000
1,xbin,0,0.000000
2,ybin,0,0.000000
3,averagedownlinkthroughput,0,0.000000
4,averageuplinkthroughput,0,0.000000
5,csfallbackattempts,0,0.000000
6,irathandoverattempts,1322741,100.000000
7,linearaveragersrp,185096,13.993367
8,linearaveragersrq,185096,13.993367
9,minutesofuse,0,0.000000



The complete cleaned test partition passed all required validation checks.


## What Cell 16 Does

This cell performs detailed checks on the cleaned test file.

It checks for missing identifiers, invalid coordinates, invalid dates, infinite values and inconsistent relationships between `location_id` and its coordinate pair.

Repeated location identifiers are expected because many observations can refer to the same BT grid location. Exact duplicate observations are reported but are not automatically removed.

# 5. Clean All Daily Partitions

This section applies the tested cleaning process to every usable daily file.

One cleaned Parquet partition is created for each usable input. Valid outputs that already exist are checked and reused rather than being created again.

In [18]:
# ================================================================
# CELL 17 - CONFIGURE CLEANED-PARTITION OUTPUT
# ================================================================

CLEANED_FILE_SUFFIX = (
    "_cleaned_bt.parquet"
)

CLEANED_TEMPORARY_SUFFIX = (
    ".temporary.parquet"
)

CLEANED_PARQUET_COMPRESSION = (
    "snappy"
)

CLEANED_PARQUET_ROW_GROUP_SIZE = (
    250_000
)

OVERWRITE_CLEANED_PARTITIONS = False


def cleaned_output_path(
    source_file
):
    """
    Return the cleaned Parquet path for one source CSV file.
    """

    return (
        DAILY_CLEANED_PARTITIONS_FOLDER
        /
        (
            source_file.stem
            + CLEANED_FILE_SUFFIX
        )
    )


def cleaned_temporary_path(
    source_file
):
    """
    Return the temporary Parquet path used during saving.
    """

    return (
        DAILY_CLEANED_PARTITIONS_FOLDER
        /
        (
            source_file.stem
            + CLEANED_TEMPORARY_SUFFIX
        )
    )


def validate_cleaned_parquet(
    parquet_file,
    expected_rows
):
    """
    Validate the essential metadata of one cleaned partition.
    """

    if not parquet_file.exists():

        return False


    try:

        parquet_metadata = pq.ParquetFile(
            parquet_file
        )

        parquet_columns = (
            parquet_metadata.schema_arrow.names
        )

        return (
            parquet_metadata.metadata.num_rows
            == expected_rows
            and
            parquet_metadata.metadata.num_columns
            == len(
                CLEANED_BT_COLUMNS
            )
            and
            parquet_columns
            == CLEANED_BT_COLUMNS
        )

    except Exception:

        return False


cleaned_output_configuration = pd.DataFrame(
    {
        "Setting": [
            "Usable input partitions",
            "Expected cleaned columns",
            "Output folder",
            "Filename suffix",
            "Compression",
            "Row-group size",
            "Overwrite valid outputs"
        ],
        "Value": [
            len(
                usable_daily_files
            ),
            len(
                CLEANED_BT_COLUMNS
            ),
            str(
                DAILY_CLEANED_PARTITIONS_FOLDER
            ),
            CLEANED_FILE_SUFFIX,
            CLEANED_PARQUET_COMPRESSION,
            CLEANED_PARQUET_ROW_GROUP_SIZE,
            OVERWRITE_CLEANED_PARTITIONS
        ]
    }
)


print("=" * 80)
print("CELL 17 - CONFIGURE CLEANED-PARTITION OUTPUT")
print("=" * 80)

display(
    cleaned_output_configuration
)


expected_cleaned_output_paths = [
    cleaned_output_path(
        source_file
    )
    for source_file
    in usable_daily_files
]


assert len(
    expected_cleaned_output_paths
) == len(
    usable_daily_files
)

assert len(
    set(
        expected_cleaned_output_paths
    )
) == len(
    expected_cleaned_output_paths
)

assert len(
    CLEANED_BT_COLUMNS
) == 34


print(
    "\nThe cleaned-partition output was "
    "configured successfully."
)

CELL 17 - CONFIGURE CLEANED-PARTITION OUTPUT


,Setting,Value
0,Usable input partitions,90
1,Expected cleaned columns,34
2,Output folder,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
3,Filename suffix,_cleaned_bt.parquet
4,Compression,snappy
5,Row-group size,250000
6,Overwrite valid outputs,False



The cleaned-partition output was configured successfully.


## What Cell 17 Does

This cell defines how the cleaned daily partitions will be saved.

It sets the output filenames, compression method, row-group size and overwrite behaviour.

Each file is first written to a temporary location. It becomes a completed output only after its row count and 34-column structure have been verified.

In [19]:
# ================================================================
# CELL 18 - CLEAN AND SAVE ALL DAILY PARTITIONS
# ================================================================

clean_all_start_time = time.time()

cleaned_partition_records = []


for _, partition_row in usable_daily_file_audit.iterrows():

    partition_number = int(
        partition_row[
            "Partition Number"
        ]
    )

    source_file = Path(
        partition_row[
            "File Path"
        ]
    )

    expected_rows = int(
        partition_row[
            "Rows"
        ]
    )

    output_file = cleaned_output_path(
        source_file
    )

    temporary_file = cleaned_temporary_path(
        source_file
    )


    partition_start_time = time.time()


    if (
        output_file.exists()
        and not OVERWRITE_CLEANED_PARTITIONS
        and validate_cleaned_parquet(
            output_file,
            expected_rows
        )
    ):

        processing_status = (
            "Skipped valid output"
        )

        output_metadata = pq.ParquetFile(
            output_file
        )

        cleaned_rows = int(
            output_metadata.metadata.num_rows
        )

        cleaned_columns = int(
            output_metadata.metadata.num_columns
        )


    else:

        if temporary_file.exists():

            temporary_file.unlink()


        if output_file.exists():

            output_file.unlink()


        raw_partition = pd.read_csv(
            source_file,
            low_memory=False
        )


        cleaned_partition = (
            clean_bt_daily_dataframe(
                raw_partition
            )
        )


        assert len(
            cleaned_partition
        ) == expected_rows

        assert list(
            cleaned_partition.columns
        ) == CLEANED_BT_COLUMNS

        assert cleaned_partition.shape[1] == 34


        cleaned_partition.to_parquet(
            temporary_file,
            engine="pyarrow",
            compression=(
                CLEANED_PARQUET_COMPRESSION
            ),
            index=False,
            row_group_size=(
                CLEANED_PARQUET_ROW_GROUP_SIZE
            )
        )


        assert validate_cleaned_parquet(
            temporary_file,
            expected_rows
        )


        temporary_file.replace(
            output_file
        )


        assert validate_cleaned_parquet(
            output_file,
            expected_rows
        )


        cleaned_rows = len(
            cleaned_partition
        )

        cleaned_columns = (
            cleaned_partition.shape[1]
        )

        processing_status = "Created"


        del raw_partition
        del cleaned_partition

        gc.collect()


    partition_seconds = (
        time.time()
        - partition_start_time
    )


    cleaned_partition_records.append(
        {
            "Partition Number": (
                partition_number
            ),
            "Source Filename": (
                source_file.name
            ),
            "Output Filename": (
                output_file.name
            ),
            "Expected Rows": (
                expected_rows
            ),
            "Cleaned Rows": (
                cleaned_rows
            ),
            "Columns": (
                cleaned_columns
            ),
            "Output Size (MB)": round(
                output_file.stat().st_size
                / 1_000_000,
                2
            ),
            "Status": (
                processing_status
            ),
            "Processing Time (Seconds)": round(
                partition_seconds,
                2
            )
        }
    )


    print(
        f"Partition {partition_number:02d}/"
        f"{len(usable_daily_file_audit):02d} | "
        f"{processing_status} | "
        f"{cleaned_rows:,} rows | "
        f"{partition_seconds:,.2f} seconds"
    )


cleaned_partition_processing = pd.DataFrame(
    cleaned_partition_records
)


clean_all_seconds = (
    time.time()
    - clean_all_start_time
)


print("\n" + "=" * 80)
print("CELL 18 - CLEAN AND SAVE ALL DAILY PARTITIONS")
print("=" * 80)


cleaning_process_summary = pd.DataFrame(
    {
        "Result": [
            "Partitions processed",
            "Partitions created",
            "Valid outputs skipped",
            "Rows represented",
            "Expected cleaned columns",
            "Total processing time (minutes)"
        ],
        "Value": [
            len(
                cleaned_partition_processing
            ),
            int(
                (
                    cleaned_partition_processing[
                        "Status"
                    ] == "Created"
                ).sum()
            ),
            int(
                (
                    cleaned_partition_processing[
                        "Status"
                    ]
                    == "Skipped valid output"
                ).sum()
            ),
            int(
                cleaned_partition_processing[
                    "Cleaned Rows"
                ].sum()
            ),
            len(
                CLEANED_BT_COLUMNS
            ),
            round(
                clean_all_seconds / 60,
                2
            )
        ]
    }
)

display(
    cleaning_process_summary
)


print("\nCleaned-partition processing results:")

display(
    cleaned_partition_processing
)


assert len(
    cleaned_partition_processing
) == len(
    usable_daily_files
)

assert (
    cleaned_partition_processing[
        "Expected Rows"
    ]
    ==
    cleaned_partition_processing[
        "Cleaned Rows"
    ]
).all()

assert (
    cleaned_partition_processing[
        "Columns"
    ] == 34
).all()

assert all(
    validate_cleaned_parquet(
        cleaned_output_path(
            source_file
        ),
        int(
            usable_daily_file_audit.loc[
                usable_daily_file_audit[
                    "File Path"
                ] == source_file,
                "Rows"
            ].iloc[0]
        )
    )
    for source_file
    in usable_daily_files
)


print(
    "\nAll usable daily partitions were cleaned "
    "and saved successfully."
)

Partition 01/90 | Created | 1,322,741 rows | 7.06 seconds
Partition 02/90 | Created | 1,325,263 rows | 6.91 seconds
Partition 03/90 | Created | 1,371,738 rows | 7.17 seconds
Partition 04/90 | Created | 1,223,418 rows | 6.57 seconds
Partition 05/90 | Created | 1,294,137 rows | 6.71 seconds
Partition 06/90 | Created | 1,215,695 rows | 6.66 seconds
Partition 07/90 | Created | 1,276,920 rows | 6.78 seconds
Partition 08/90 | Created | 1,267,340 rows | 6.64 seconds
Partition 09/90 | Created | 1,232,265 rows | 6.47 seconds
Partition 10/90 | Created | 1,208,420 rows | 7.41 seconds
Partition 11/90 | Created | 1,165,378 rows | 7.53 seconds
Partition 12/90 | Created | 1,244,816 rows | 6.56 seconds
Partition 13/90 | Created | 1,272,228 rows | 6.66 seconds
Partition 14/90 | Created | 1,368,786 rows | 7.01 seconds
Partition 15/90 | Created | 1,247,121 rows | 6.41 seconds
Partition 16/90 | Created | 1,224,135 rows | 6.61 seconds
Partition 17/90 | Created | 1,235,078 rows | 6.45 seconds
Partition 18/9

,Result,Value
0,Partitions processed,90.000000
1,Partitions created,90.000000
2,Valid outputs skipped,0.000000
3,Rows represented,"119,267,039.000000"
4,Expected cleaned columns,34.000000
5,Total processing time (minutes),10.440000



Cleaned-partition processing results:


,Partition Number,Source Filename,Output Filename,Expected Rows,Cleaned Rows,Columns,Output Size (MB),Status,Processing Time (Seconds)
0,1,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1322741,1322741,34,66.340000,Created,7.060000
1,2,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1325263,1325263,34,66.450000,Created,6.910000
2,3,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1371738,1371738,34,67.750000,Created,7.170000
3,4,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1223418,1223418,34,58.870000,Created,6.570000
4,5,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1294137,1294137,34,63.700000,Created,6.710000
5,6,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1215695,1215695,34,61.000000,Created,6.660000
6,7,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1276920,1276920,34,64.650000,Created,6.780000
7,8,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1267340,1267340,34,61.990000,Created,6.640000
8,9,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1232265,1232265,34,62.220000,Created,6.470000
9,10,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1208420,1208420,34,56.420000,Created,7.410000



All usable daily partitions were cleaned and saved successfully.


## What Cell 18 Does

This cell cleans and saves every usable daily file.

For each input file, the cell checks whether a valid cleaned output already exists. Existing valid outputs are reused, while missing or invalid outputs are created again.

Every completed partition must preserve the original observation count and contain the required 34 cleaned columns.

This allows future daily files to be added without repeating the cleaning work for files that have already been processed successfully.

In [20]:
# ================================================================
# CELL 19 - VERIFY ALL CLEANED PARTITIONS
# ================================================================

expected_cleaned_output_paths = [
    cleaned_output_path(
        source_file
    )
    for source_file
    in usable_daily_files
]


expected_cleaned_output_set = {
    file_path.resolve()
    for file_path
    in expected_cleaned_output_paths
}


observed_cleaned_output_paths = sorted(
    DAILY_CLEANED_PARTITIONS_FOLDER.glob(
        f"*{CLEANED_FILE_SUFFIX}"
    )
)


observed_cleaned_output_set = {
    file_path.resolve()
    for file_path
    in observed_cleaned_output_paths
}


missing_cleaned_output_paths = sorted(
    expected_cleaned_output_set
    -
    observed_cleaned_output_set
)


unexpected_cleaned_output_paths = sorted(
    observed_cleaned_output_set
    -
    expected_cleaned_output_set
)


# Only the outputs expected from the current usable inputs
# are used by the remaining notebook cells.
cleaned_partition_files = [
    file_path
    for file_path
    in expected_cleaned_output_paths
    if file_path.exists()
]


temporary_cleaned_files = sorted(
    DAILY_CLEANED_PARTITIONS_FOLDER.glob(
        f"*{CLEANED_TEMPORARY_SUFFIX}"
    )
)


cleaned_partition_verification_records = []


for _, source_row in usable_daily_file_audit.iterrows():

    partition_number = int(
        source_row[
            "Partition Number"
        ]
    )

    source_file = Path(
        source_row[
            "File Path"
        ]
    )

    expected_rows = int(
        source_row[
            "Rows"
        ]
    )

    output_file = cleaned_output_path(
        source_file
    )


    file_exists = output_file.exists()

    readable = False
    observed_rows = 0
    observed_columns = 0
    observed_row_groups = 0
    correct_column_order = False
    file_size_mb = 0
    verification_error = None


    if file_exists:

        try:

            parquet_metadata = pq.read_metadata(
                output_file
            )

            parquet_schema = pq.read_schema(
                output_file
            )


            observed_rows = int(
                parquet_metadata.num_rows
            )

            observed_columns = int(
                parquet_metadata.num_columns
            )

            observed_row_groups = int(
                parquet_metadata.num_row_groups
            )

            correct_column_order = (
                parquet_schema.names
                ==
                CLEANED_BT_COLUMNS
            )

            file_size_mb = round(
                output_file.stat().st_size
                / 1_000_000,
                2
            )

            readable = True

        except Exception as error:

            verification_error = str(
                error
            )


    valid_partition = (
        file_exists
        and readable
        and observed_rows == expected_rows
        and observed_columns
        == len(
            CLEANED_BT_COLUMNS
        )
        and observed_row_groups > 0
        and correct_column_order
        and file_size_mb > 0
    )


    cleaned_partition_verification_records.append(
        {
            "Partition Number": (
                partition_number
            ),
            "Source Filename": (
                source_file.name
            ),
            "Output Filename": (
                output_file.name
            ),
            "Expected Rows": (
                expected_rows
            ),
            "Observed Rows": (
                observed_rows
            ),
            "Columns": (
                observed_columns
            ),
            "Row Groups": (
                observed_row_groups
            ),
            "File Size (MB)": (
                file_size_mb
            ),
            "Readable": (
                readable
            ),
            "Correct Column Order": (
                correct_column_order
            ),
            "Valid": (
                valid_partition
            ),
            "Verification Error": (
                verification_error
            )
        }
    )


cleaned_partition_verification = pd.DataFrame(
    cleaned_partition_verification_records
)


verified_cleaned_rows = int(
    cleaned_partition_verification[
        "Observed Rows"
    ].sum()
)


expected_cleaned_rows = int(
    usable_daily_file_audit[
        "Rows"
    ].sum()
)


valid_cleaned_partitions = int(
    cleaned_partition_verification[
        "Valid"
    ].sum()
)


invalid_cleaned_partitions = int(
    (
        ~cleaned_partition_verification[
            "Valid"
        ]
    ).sum()
)


combined_cleaned_size_gb = (
    cleaned_partition_verification[
        "File Size (MB)"
    ].sum()
    / 1_000
)


print("=" * 80)
print("CELL 19 - VERIFY ALL CLEANED PARTITIONS")
print("=" * 80)


cleaned_partition_verification_summary = pd.DataFrame(
    {
        "Check": [
            "Usable input files",
            "Expected cleaned outputs",
            "Expected outputs found",
            "Missing expected outputs",
            "Unexpected old outputs",
            "Valid cleaned partitions",
            "Invalid cleaned partitions",
            "Expected rows",
            "Verified rows",
            "Row difference",
            "Columns per partition",
            "Temporary files",
            "Combined size (GB)"
        ],
        "Result": [
            len(
                usable_daily_files
            ),
            len(
                expected_cleaned_output_paths
            ),
            len(
                cleaned_partition_files
            ),
            len(
                missing_cleaned_output_paths
            ),
            len(
                unexpected_cleaned_output_paths
            ),
            valid_cleaned_partitions,
            invalid_cleaned_partitions,
            expected_cleaned_rows,
            verified_cleaned_rows,
            (
                verified_cleaned_rows
                -
                expected_cleaned_rows
            ),
            len(
                CLEANED_BT_COLUMNS
            ),
            len(
                temporary_cleaned_files
            ),
            round(
                combined_cleaned_size_gb,
                2
            )
        ]
    }
)

display(
    cleaned_partition_verification_summary
)


print("\nCleaned-partition verification:")

display(
    cleaned_partition_verification
)


if missing_cleaned_output_paths:

    print("\nMissing expected cleaned outputs:")

    for file_path in missing_cleaned_output_paths:

        print(
            file_path
        )


if unexpected_cleaned_output_paths:

    print(
        "\nUnexpected old cleaned outputs "
        "(reported but not used):"
    )

    for file_path in unexpected_cleaned_output_paths:

        print(
            file_path
        )


assert len(
    missing_cleaned_output_paths
) == 0

assert len(
    cleaned_partition_files
) == len(
    usable_daily_files
)

assert valid_cleaned_partitions == len(
    usable_daily_files
)

assert invalid_cleaned_partitions == 0

assert verified_cleaned_rows == (
    expected_cleaned_rows
)

assert len(
    temporary_cleaned_files
) == 0


print(
    "\nEvery usable daily input has one valid "
    "cleaned partition."
)

CELL 19 - VERIFY ALL CLEANED PARTITIONS


,Check,Result
0,Usable input files,90.000000
1,Expected cleaned outputs,90.000000
2,Expected outputs found,90.000000
3,Missing expected outputs,0.000000
4,Unexpected old outputs,0.000000
5,Valid cleaned partitions,90.000000
6,Invalid cleaned partitions,0.000000
7,Expected rows,"119,267,039.000000"
8,Verified rows,"119,267,039.000000"
9,Row difference,0.000000



Cleaned-partition verification:


,Partition Number,Source Filename,Output Filename,Expected Rows,Observed Rows,Columns,Row Groups,File Size (MB),Readable,Correct Column Order,Valid,Verification Error
0,1,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1322741,1322741,34,6,66.340000,True,True,True,None
1,2,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1325263,1325263,34,6,66.450000,True,True,True,None
2,3,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1371738,1371738,34,6,67.750000,True,True,True,None
3,4,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1223418,1223418,34,5,58.870000,True,True,True,None
4,5,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1294137,1294137,34,6,63.700000,True,True,True,None
5,6,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1215695,1215695,34,5,61.000000,True,True,True,None
6,7,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1276920,1276920,34,6,64.650000,True,True,True,None
7,8,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1267340,1267340,34,6,61.990000,True,True,True,None
8,9,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1232265,1232265,34,5,62.220000,True,True,True,None
9,10,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,GEO_Export_Neon_530k_5590k_668k_5650k_2026-04-...,1208420,1208420,34,5,56.420000,True,True,True,None



Every usable daily input has one valid cleaned partition.


## What Cell 19 Does

This cell verifies the cleaned outputs expected from the current usable input files.

It checks that:

- every usable input has one cleaned output;
- every output has the correct row count;
- every output contains the required 34 columns;
- no incomplete temporary files remain.

Old output files that do not correspond to the current input list are reported but are not used by the remaining notebook stages.

In [21]:
# ================================================================
# CELL 20 - SUMMARISE THE CLEANED BT DATASET
# ================================================================

import pyarrow.compute as pc


cleaned_quality_start_time = time.time()


cleaned_missing_counts = {
    column_name: 0
    for column_name
    in CLEANED_BT_COLUMNS
}


cleaned_minimum_xbin = None
cleaned_maximum_xbin = None

cleaned_minimum_ybin = None
cleaned_maximum_ybin = None

cleaned_minimum_date = None
cleaned_maximum_date = None


for partition_number, parquet_file in enumerate(
    cleaned_partition_files,
    start=1
):

    partition_table = pq.read_table(
        parquet_file,
        columns=CLEANED_BT_COLUMNS
    )


    for column_name in CLEANED_BT_COLUMNS:

        cleaned_missing_counts[
            column_name
        ] += int(
            partition_table[
                column_name
            ].null_count
        )


    partition_minimum_xbin = pc.min(
        partition_table[
            "xbin"
        ]
    ).as_py()

    partition_maximum_xbin = pc.max(
        partition_table[
            "xbin"
        ]
    ).as_py()

    partition_minimum_ybin = pc.min(
        partition_table[
            "ybin"
        ]
    ).as_py()

    partition_maximum_ybin = pc.max(
        partition_table[
            "ybin"
        ]
    ).as_py()

    partition_minimum_date = pc.min(
        partition_table[
            "en_dt"
        ]
    ).as_py()

    partition_maximum_date = pc.max(
        partition_table[
            "en_dt"
        ]
    ).as_py()


    if partition_minimum_xbin is not None:

        cleaned_minimum_xbin = (
            partition_minimum_xbin
            if cleaned_minimum_xbin is None
            else min(
                cleaned_minimum_xbin,
                partition_minimum_xbin
            )
        )


    if partition_maximum_xbin is not None:

        cleaned_maximum_xbin = (
            partition_maximum_xbin
            if cleaned_maximum_xbin is None
            else max(
                cleaned_maximum_xbin,
                partition_maximum_xbin
            )
        )


    if partition_minimum_ybin is not None:

        cleaned_minimum_ybin = (
            partition_minimum_ybin
            if cleaned_minimum_ybin is None
            else min(
                cleaned_minimum_ybin,
                partition_minimum_ybin
            )
        )


    if partition_maximum_ybin is not None:

        cleaned_maximum_ybin = (
            partition_maximum_ybin
            if cleaned_maximum_ybin is None
            else max(
                cleaned_maximum_ybin,
                partition_maximum_ybin
            )
        )


    if partition_minimum_date is not None:

        cleaned_minimum_date = (
            partition_minimum_date
            if cleaned_minimum_date is None
            else min(
                cleaned_minimum_date,
                partition_minimum_date
            )
        )


    if partition_maximum_date is not None:

        cleaned_maximum_date = (
            partition_maximum_date
            if cleaned_maximum_date is None
            else max(
                cleaned_maximum_date,
                partition_maximum_date
            )
        )


    del partition_table

    gc.collect()


    print(
        f"Summarised partition "
        f"{partition_number:02d}/"
        f"{len(cleaned_partition_files):02d}"
    )


cleaned_missing_summary = pd.DataFrame(
    {
        "Column": CLEANED_BT_COLUMNS,
        "Missing Values": [
            cleaned_missing_counts[
                column_name
            ]
            for column_name
            in CLEANED_BT_COLUMNS
        ]
    }
)


cleaned_missing_summary[
    "Missing Percentage"
] = (
    cleaned_missing_summary[
        "Missing Values"
    ]
    / verified_cleaned_rows
    * 100
)


cleaned_dataset_summary = pd.DataFrame(
    {
        "Measure": [
            "Supplied daily files",
            "Usable daily files",
            "Excluded daily files",
            "Hourly files used in daily section",
            "Source observations",
            "Cleaned observations",
            "Row difference",
            "Supplied BT columns",
            "Created location identifiers",
            "Cleaned columns",
            "Minimum date",
            "Maximum date",
            "Minimum xbin",
            "Maximum xbin",
            "Minimum ybin",
            "Maximum ybin",
            "Combined output size (GB)",
            "Summary time (minutes)"
        ],
        "Value": [
            len(
                daily_csv_files
            ),
            len(
                usable_daily_files
            ),
            len(
                excluded_daily_file_audit
            ),
            0,
            expected_cleaned_rows,
            verified_cleaned_rows,
            (
                verified_cleaned_rows
                -
                expected_cleaned_rows
            ),
            len(
                ORIGINAL_BT_COLUMNS
            ),
            1,
            len(
                CLEANED_BT_COLUMNS
            ),
            cleaned_minimum_date,
            cleaned_maximum_date,
            cleaned_minimum_xbin,
            cleaned_maximum_xbin,
            cleaned_minimum_ybin,
            cleaned_maximum_ybin,
            round(
                combined_cleaned_size_gb,
                2
            ),
            round(
                (
                    time.time()
                    - cleaned_quality_start_time
                ) / 60,
                2
            )
        ]
    }
)


print("=" * 80)
print("CELL 20 - SUMMARISE THE CLEANED BT DATASET")
print("=" * 80)

display(
    cleaned_dataset_summary
)

print("\nComplete cleaned-data missingness:")

display(
    cleaned_missing_summary
)


assert verified_cleaned_rows == (
    expected_cleaned_rows
)

assert (
    verified_cleaned_rows
    -
    expected_cleaned_rows
) == 0

assert len(
    CLEANED_BT_COLUMNS
) == 34


print(
    "\nThe complete cleaned BT dataset was "
    "summarised successfully."
)

Summarised partition 01/90
Summarised partition 02/90
Summarised partition 03/90
Summarised partition 04/90
Summarised partition 05/90
Summarised partition 06/90
Summarised partition 07/90
Summarised partition 08/90
Summarised partition 09/90
Summarised partition 10/90
Summarised partition 11/90
Summarised partition 12/90
Summarised partition 13/90
Summarised partition 14/90
Summarised partition 15/90
Summarised partition 16/90
Summarised partition 17/90
Summarised partition 18/90
Summarised partition 19/90
Summarised partition 20/90
Summarised partition 21/90
Summarised partition 22/90
Summarised partition 23/90
Summarised partition 24/90
Summarised partition 25/90
Summarised partition 26/90
Summarised partition 27/90
Summarised partition 28/90
Summarised partition 29/90
Summarised partition 30/90
Summarised partition 31/90
Summarised partition 32/90
Summarised partition 33/90
Summarised partition 34/90
Summarised partition 35/90
Summarised partition 36/90
Summarised partition 37/90
S

,Measure,Value
0,Supplied daily files,95
1,Usable daily files,90
2,Excluded daily files,5
3,Hourly files used in daily section,0
4,Source observations,119267039
5,Cleaned observations,119267039
6,Row difference,0
7,Supplied BT columns,33
8,Created location identifiers,1
9,Cleaned columns,34



Complete cleaned-data missingness:


,Column,Missing Values,Missing Percentage
0,location_id,0,0.000000
1,xbin,0,0.000000
2,ybin,0,0.000000
3,averagedownlinkthroughput,0,0.000000
4,averageuplinkthroughput,0,0.000000
5,csfallbackattempts,0,0.000000
6,irathandoverattempts,119267039,100.000000
7,linearaveragersrp,15465932,12.967482
8,linearaveragersrq,15465940,12.967489
9,minutesofuse,0,0.000000



The complete cleaned BT dataset was summarised successfully.


## What Cell 20 Does

This cell summarises the complete cleaned BT dataset.

It reports the total observations, date range, coordinate range, storage size and missing values for every cleaned column.

The cell reads the saved outputs for reporting only and does not change any data.

# 6. Unique BT Locations

This section identifies every distinct BT grid location represented by the current cleaned files.

The location list is rebuilt from the current inputs so that any locations introduced by newly added daily files can be detected.

In [22]:
# ================================================================
# CELL 21 - EXTRACT ALL UNIQUE BT LOCATIONS
# ================================================================

unique_location_start_time = time.time()


seen_location_ids = set()

new_location_frames = []

unique_location_partition_records = []


for partition_number, parquet_file in enumerate(
    cleaned_partition_files,
    start=1
):

    partition_locations = pd.read_parquet(
        parquet_file,
        columns=[
            "location_id",
            "xbin",
            "ybin"
        ],
        engine="pyarrow"
    )


    partition_rows = len(
        partition_locations
    )


    partition_locations = (
        partition_locations
        .dropna(
            subset=[
                "location_id",
                "xbin",
                "ybin"
            ]
        )
        .drop_duplicates(
            subset=[
                "location_id",
                "xbin",
                "ybin"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    new_location_mask = (
        ~partition_locations[
            "location_id"
        ].isin(
            seen_location_ids
        )
    )


    new_partition_locations = (
        partition_locations.loc[
            new_location_mask
        ]
        .copy()
    )


    if len(
        new_partition_locations
    ) > 0:

        new_location_frames.append(
            new_partition_locations
        )

        seen_location_ids.update(
            new_partition_locations[
                "location_id"
            ].tolist()
        )


    unique_location_partition_records.append(
        {
            "Partition Number": (
                partition_number
            ),
            "Partition Rows": (
                partition_rows
            ),
            "Distinct Locations in Partition": (
                len(
                    partition_locations
                )
            ),
            "New Locations Added": (
                len(
                    new_partition_locations
                )
            ),
            "Cumulative Unique Locations": (
                len(
                    seen_location_ids
                )
            )
        }
    )


    print(
        f"Partition {partition_number:02d}/"
        f"{len(cleaned_partition_files):02d} | "
        f"new locations: "
        f"{len(new_partition_locations):,} | "
        f"cumulative: "
        f"{len(seen_location_ids):,}"
    )


    del partition_locations
    del new_partition_locations

    gc.collect()


unique_bt_locations = (
    pd.concat(
        new_location_frames,
        ignore_index=True
    )
    .reset_index(
        drop=True
    )
)


unique_location_extraction_summary = pd.DataFrame(
    unique_location_partition_records
)


unique_location_seconds = (
    time.time()
    - unique_location_start_time
)


print("\n" + "=" * 80)
print("CELL 21 - EXTRACT ALL UNIQUE BT LOCATIONS")
print("=" * 80)


unique_location_summary = pd.DataFrame(
    {
        "Result": [
            "Cleaned partitions scanned",
            "Cleaned observations represented",
            "Unique BT locations",
            "Location columns",
            "Processing time (minutes)"
        ],
        "Value": [
            len(
                cleaned_partition_files
            ),
            verified_cleaned_rows,
            len(
                unique_bt_locations
            ),
            unique_bt_locations.shape[1],
            round(
                unique_location_seconds / 60,
                2
            )
        ]
    }
)

display(
    unique_location_summary
)

print("\nUnique-location extraction by partition:")

display(
    unique_location_extraction_summary
)

print("\nFirst unique BT locations:")

display(
    unique_bt_locations.head()
)


assert len(
    unique_bt_locations
) > 0

assert list(
    unique_bt_locations.columns
) == [
    "location_id",
    "xbin",
    "ybin"
]

assert unique_bt_locations[
    "location_id"
].is_unique


print(
    "\nAll unique BT grid locations were "
    "extracted successfully."
)

Partition 01/90 | new locations: 1,322,741 | cumulative: 1,322,741
Partition 02/90 | new locations: 328,287 | cumulative: 1,651,028
Partition 03/90 | new locations: 208,661 | cumulative: 1,859,689
Partition 04/90 | new locations: 100,881 | cumulative: 1,960,570
Partition 05/90 | new locations: 95,200 | cumulative: 2,055,770
Partition 06/90 | new locations: 60,962 | cumulative: 2,116,732
Partition 07/90 | new locations: 56,448 | cumulative: 2,173,180
Partition 08/90 | new locations: 50,062 | cumulative: 2,223,242
Partition 09/90 | new locations: 39,071 | cumulative: 2,262,313
Partition 10/90 | new locations: 29,422 | cumulative: 2,291,735
Partition 11/90 | new locations: 25,562 | cumulative: 2,317,297
Partition 12/90 | new locations: 29,675 | cumulative: 2,346,972
Partition 13/90 | new locations: 27,199 | cumulative: 2,374,171
Partition 14/90 | new locations: 35,146 | cumulative: 2,409,317
Partition 15/90 | new locations: 22,061 | cumulative: 2,431,378
Partition 16/90 | new locations: 1

,Result,Value
0,Cleaned partitions scanned,90.000000
1,Cleaned observations represented,"119,267,039.000000"
2,Unique BT locations,"2,942,185.000000"
3,Location columns,3.000000
4,Processing time (minutes),5.430000



Unique-location extraction by partition:


,Partition Number,Partition Rows,Distinct Locations in Partition,New Locations Added,Cumulative Unique Locations
0,1,1322741,1322741,1322741,1322741
1,2,1325263,1325263,328287,1651028
2,3,1371738,1371738,208661,1859689
3,4,1223418,1223418,100881,1960570
4,5,1294137,1294137,95200,2055770
5,6,1215695,1215695,60962,2116732
6,7,1276920,1276920,56448,2173180
7,8,1267340,1267340,50062,2223242
8,9,1232265,1232265,39071,2262313
9,10,1208420,1208420,29422,2291735



First unique BT locations:


,location_id,xbin,ybin
0,530050_5647100,"530,050.000000","5,647,100.000000"
1,530100_5606050,"530,100.000000","5,606,050.000000"
2,530100_5618850,"530,100.000000","5,618,850.000000"
3,530100_5631050,"530,100.000000","5,631,050.000000"
4,530100_5634250,"530,100.000000","5,634,250.000000"



All unique BT grid locations were extracted successfully.


## What Cell 21 Does

This cell creates one record for every unique BT grid location.

It reads only `location_id`, `xbin` and `ybin` from each cleaned partition. It does not reload all of the network measurements and does not recalculate any POI distances.

The current location list allows the notebook to identify whether newly added daily files contain grid locations that are not already stored in the POI location dataset.

In [23]:
# ================================================================
# CELL 22 - VALIDATE THE UNIQUE BT LOCATIONS
# ================================================================

recreated_location_ids = create_location_id(
    unique_bt_locations[
        "xbin"
    ],
    unique_bt_locations[
        "ybin"
    ]
)


location_id_mismatches = int(
    (
        recreated_location_ids
        !=
        unique_bt_locations[
            "location_id"
        ]
    )
    .fillna(
        True
    )
    .sum()
)


duplicate_location_ids = int(
    unique_bt_locations[
        "location_id"
    ].duplicated().sum()
)


duplicate_coordinate_pairs = int(
    unique_bt_locations[
        [
            "xbin",
            "ybin"
        ]
    ].duplicated().sum()
)


missing_location_ids = int(
    unique_bt_locations[
        "location_id"
    ].isna().sum()
)

missing_x_coordinates = int(
    unique_bt_locations[
        "xbin"
    ].isna().sum()
)

missing_y_coordinates = int(
    unique_bt_locations[
        "ybin"
    ].isna().sum()
)


invalid_x_coordinates = int(
    (
        ~np.isfinite(
            unique_bt_locations[
                "xbin"
            ]
        )
    ).sum()
)


invalid_y_coordinates = int(
    (
        ~np.isfinite(
            unique_bt_locations[
                "ybin"
            ]
        )
    ).sum()
)


unique_location_validation = pd.DataFrame(
    {
        "Check": [
            "Unique location records",
            "Location columns",
            "Duplicate location identifiers",
            "Duplicate coordinate pairs",
            "Missing location identifiers",
            "Missing x coordinates",
            "Missing y coordinates",
            "Non-finite x coordinates",
            "Non-finite y coordinates",
            "Identifier-coordinate mismatches",
            "Minimum xbin",
            "Maximum xbin",
            "Minimum ybin",
            "Maximum ybin"
        ],
        "Result": [
            len(
                unique_bt_locations
            ),
            unique_bt_locations.shape[1],
            duplicate_location_ids,
            duplicate_coordinate_pairs,
            missing_location_ids,
            missing_x_coordinates,
            missing_y_coordinates,
            invalid_x_coordinates,
            invalid_y_coordinates,
            location_id_mismatches,
            unique_bt_locations[
                "xbin"
            ].min(),
            unique_bt_locations[
                "xbin"
            ].max(),
            unique_bt_locations[
                "ybin"
            ].min(),
            unique_bt_locations[
                "ybin"
            ].max()
        ]
    }
)


print("=" * 80)
print("CELL 22 - VALIDATE THE UNIQUE BT LOCATIONS")
print("=" * 80)

display(
    unique_location_validation
)


assert duplicate_location_ids == 0

assert duplicate_coordinate_pairs == 0

assert missing_location_ids == 0

assert missing_x_coordinates == 0

assert missing_y_coordinates == 0

assert invalid_x_coordinates == 0

assert invalid_y_coordinates == 0

assert location_id_mismatches == 0


print(
    "\nThe unique BT location table passed "
    "all validation checks."
)

CELL 22 - VALIDATE THE UNIQUE BT LOCATIONS


,Check,Result
0,Unique location records,"2,942,185.000000"
1,Location columns,3.000000
2,Duplicate location identifiers,0.000000
3,Duplicate coordinate pairs,0.000000
4,Missing location identifiers,0.000000
5,Missing x coordinates,0.000000
6,Missing y coordinates,0.000000
7,Non-finite x coordinates,0.000000
8,Non-finite y coordinates,0.000000
9,Identifier-coordinate mismatches,0.000000



The unique BT location table passed all validation checks.


## What Cell 22 Does

This cell checks the completed unique-location table.

Every location must have:

- one unique `location_id`;
- one unique coordinate pair;
- complete and finite coordinate values;
- an identifier that matches its `xbin` and `ybin` values.

These checks confirm that the locations are suitable for joining with the POI information.

# 7. POI Preparation

This section loads and validates the processed POI dataset created in Notebook 1.

The available POI categories are discovered dynamically from the current file. The notebook therefore does not require the number of POI categories to remain fixed between runs.

A fingerprint of the processed POI source is also created so that later cells can determine whether the geographic reference data has changed since the previous successful run.

In [24]:
# ================================================================
# CELL 23 - IMPORT GEOGRAPHIC PACKAGES
# ================================================================

import fiona
import geopandas as gpd
import shapely

from shapely.geometry import Point


print("=" * 80)
print("CELL 23 - IMPORT GEOGRAPHIC PACKAGES")
print("=" * 80)


geographic_software_versions = pd.DataFrame(
    {
        "Software": [
            "GeoPandas",
            "Fiona",
            "Shapely"
        ],
        "Version": [
            gpd.__version__,
            fiona.__version__,
            shapely.__version__
        ]
    }
)

display(
    geographic_software_versions
)


print(
    "\nThe geographic packages were imported "
    "successfully."
)

CELL 23 - IMPORT GEOGRAPHIC PACKAGES


,Software,Version
0,GeoPandas,1.1.4
1,Fiona,1.10.1
2,Shapely,2.1.2



The geographic packages were imported successfully.


## What Cell 23 Does

This cell identifies the processed POI dataset produced by Notebook 1 and prepares the libraries required for geographic processing.

The processed GeoPackage is used as the geographic reference source for the location features created later in the notebook.

The POI categories themselves are discovered from the current file in the following cells rather than being manually fixed in Notebook 2.

In [25]:
# ================================================================
# CELL 24 - LOAD AND VERIFY THE PROCESSED POI DATASET
# ================================================================

BT_CRS = "EPSG:32630"

POI_CATEGORY_COLUMN = (
    "poi_category"
)

POI_FINGERPRINT_METHOD = (
    "canonical_poi_content_v1"
)

# ------------------------------------------------
# Discover the appropriate POI layer
# ------------------------------------------------

available_poi_layers = list(
    fiona.listlayers(
        POI_INPUT_FILE
    )
)

preferred_poi_layers = [
    "combined_pois",
    "BT_POI_Dataset",
    "poi_dataset"
]

selected_poi_layer = next(
    (
        layer_name
        for layer_name
        in preferred_poi_layers
        if layer_name
        in available_poi_layers
    ),
    None
)

if selected_poi_layer is None:

    if len(
        available_poi_layers
    ) == 1:

        selected_poi_layer = (
            available_poi_layers[
                0
            ]
        )

    else:

        raise ValueError(
            "The POI GeoPackage contains multiple layers "
            "and the combined POI layer could not be "
            "identified automatically."
        )

# ------------------------------------------------
# Load current POI dataset
# ------------------------------------------------

poi_gdf = gpd.read_file(
    POI_INPUT_FILE,
    layer=selected_poi_layer
)

required_poi_source_columns = [
    POI_CATEGORY_COLUMN,
    "poi_name",
    "geometry"
]

missing_poi_source_columns = [
    column_name
    for column_name
    in required_poi_source_columns
    if column_name
    not in poi_gdf.columns
]

if missing_poi_source_columns:

    raise ValueError(
        "The processed POI dataset is missing required "
        f"columns: {missing_poi_source_columns}"
    )

# ------------------------------------------------
# Discover current POI categories
# ------------------------------------------------

observed_poi_categories = sorted(
    poi_gdf[
        POI_CATEGORY_COLUMN
    ]
    .dropna()
    .astype(
        "string"
    )
    .unique()
    .tolist()
)

missing_poi_category_values = int(
    poi_gdf[
        POI_CATEGORY_COLUMN
    ]
    .isna()
    .sum()
)

missing_poi_geometries = int(
    poi_gdf.geometry
    .isna()
    .sum()
)

empty_poi_geometries = int(
    poi_gdf.geometry
    .is_empty
    .sum()
)

invalid_poi_geometries = int(
    (
        ~poi_gdf.geometry.is_valid
    ).sum()
)

poi_category_summary = (
    poi_gdf
    .groupby(
        POI_CATEGORY_COLUMN
    )
    .size()
    .reset_index(
        name="POI Features"
    )
    .sort_values(
        POI_CATEGORY_COLUMN
    )
    .reset_index(
        drop=True
    )
)

# ------------------------------------------------
# Create a legacy fingerprint of the physical
# GeoPackage file.
#
# This is retained only so existing saved state files
# created by earlier versions of Notebook 2 can be
# migrated without forcing another full rebuild.
# ------------------------------------------------

poi_source_file_hasher = (
    hashlib.sha256()
)

with POI_INPUT_FILE.open(
    "rb"
) as poi_source_handle:

    while True:

        poi_source_chunk = (
            poi_source_handle.read(
                8
                *
                1024
                *
                1024
            )
        )

        if not poi_source_chunk:

            break

        poi_source_file_hasher.update(
            poi_source_chunk
        )

POI_SOURCE_FILE_FINGERPRINT = (
    poi_source_file_hasher.hexdigest()
)

# ------------------------------------------------
# Create a stable fingerprint of the actual POI
# contents.
#
# Unlike the physical-file fingerprint above, this
# remains unchanged if the same POI records are simply
# rewritten into a new GeoPackage.
# ------------------------------------------------

preferred_fingerprint_columns = [
    "poi_id",
    "poi_category",
    "poi_name",
    "osm_element_type",
    "osm_id",
    "poi_source",
    "download_buffer_metres"
]

poi_fingerprint_columns = [
    column_name
    for column_name
    in preferred_fingerprint_columns
    if column_name
    in poi_gdf.columns
]

poi_fingerprint_table = (
    poi_gdf[
        poi_fingerprint_columns
    ]
    .copy()
)

for column_name in poi_fingerprint_columns:

    if column_name == "download_buffer_metres":

        numeric_values = pd.to_numeric(
            poi_fingerprint_table[
                column_name
            ],
            errors="coerce"
        )

        poi_fingerprint_table[
            column_name
        ] = numeric_values.map(
            lambda value: (
                "<NA>"
                if pd.isna(
                    value
                )
                else format(
                    float(
                        value
                    ),
                    ".12g"
                )
            )
        )

    else:

        poi_fingerprint_table[
            column_name
        ] = (
            poi_fingerprint_table[
                column_name
            ]
            .astype(
                "string"
            )
            .fillna(
                "<NA>"
            )
        )

poi_fingerprint_table[
    "geometry_wkb"
] = (
    poi_gdf.geometry.map(
        lambda geometry: (
            "<NA>"
            if geometry is None
            else geometry.wkb_hex
        )
    )
)

poi_fingerprint_sort_columns = [
    *poi_fingerprint_columns,
    "geometry_wkb"
]

poi_fingerprint_table = (
    poi_fingerprint_table
    .sort_values(
        poi_fingerprint_sort_columns,
        kind="mergesort"
    )
    .reset_index(
        drop=True
    )
)

poi_content_hasher = (
    hashlib.sha256()
)

fingerprint_header = {
    "fingerprint_method": (
        POI_FINGERPRINT_METHOD
    ),
    "crs": (
        str(
            poi_gdf.crs
        )
    ),
    "columns": (
        poi_fingerprint_sort_columns
    )
}

poi_content_hasher.update(
    json.dumps(
        fingerprint_header,
        sort_keys=True,
        separators=(
            ",",
            ":"
        )
    )
    .encode(
        "utf-8"
    )
)

for fingerprint_row in (
    poi_fingerprint_table
    .itertuples(
        index=False,
        name=None
    )
):

    poi_content_hasher.update(
        json.dumps(
            list(
                fingerprint_row
            ),
            ensure_ascii=False,
            separators=(
                ",",
                ":"
            )
        )
        .encode(
            "utf-8"
        )
    )

    poi_content_hasher.update(
        b"\n"
    )

POI_SOURCE_FINGERPRINT = (
    poi_content_hasher.hexdigest()
)

del poi_fingerprint_table

# ------------------------------------------------
# Record current POI source state
# ------------------------------------------------

CURRENT_POI_SOURCE_STATE = {
    "source_filename": (
        POI_INPUT_FILE.name
    ),
    "selected_layer": (
        selected_poi_layer
    ),
    "fingerprint_method": (
        POI_FINGERPRINT_METHOD
    ),
    "sha256": (
        POI_SOURCE_FINGERPRINT
    ),
    "file_sha256": (
        POI_SOURCE_FILE_FINGERPRINT
    ),
    "file_size_bytes": int(
        POI_INPUT_FILE.stat().st_size
    ),
    "feature_count": int(
        len(
            poi_gdf
        )
    ),
    "category_count": int(
        len(
            observed_poi_categories
        )
    ),
    "categories": (
        observed_poi_categories
    )
}

poi_validation_summary = pd.DataFrame(
    {
        "Check": [
            "GeoPackage layers",
            "Selected layer",
            "POI features",
            "POI columns",
            "Observed POI categories",
            "Missing category values",
            "Missing geometries",
            "Empty geometries",
            "Invalid geometries",
            "Coordinate reference system",
            "Distance units",
            "Fingerprint method",
            "POI content SHA-256",
            "GeoPackage file SHA-256"
        ],
        "Result": [
            len(
                available_poi_layers
            ),
            selected_poi_layer,
            len(
                poi_gdf
            ),
            len(
                poi_gdf.columns
            ),
            len(
                observed_poi_categories
            ),
            missing_poi_category_values,
            missing_poi_geometries,
            empty_poi_geometries,
            invalid_poi_geometries,
            str(
                poi_gdf.crs
            ),
            "metres",
            POI_FINGERPRINT_METHOD,
            POI_SOURCE_FINGERPRINT,
            POI_SOURCE_FILE_FINGERPRINT
        ]
    }
)

print("=" * 80)
print("CELL 24 - LOAD AND VERIFY THE PROCESSED POI DATASET")
print("=" * 80)

print(
    "\nAvailable GeoPackage layers:"
)

for layer_name in (
    available_poi_layers
):

    print(
        f"  - {layer_name}"
    )

print(
    "\nPOI category summary:"
)

display(
    poi_category_summary
)

print(
    "\nPOI validation summary:"
)

display(
    poi_validation_summary
)

assert len(
    observed_poi_categories
) > 0

assert missing_poi_category_values == 0

assert missing_poi_geometries == 0

assert empty_poi_geometries == 0

assert invalid_poi_geometries == 0

assert poi_gdf.crs is not None

assert (
    poi_gdf.crs.to_epsg()
    ==
    32630
)

print(
    "\nThe processed POI dataset was loaded "
    "and verified successfully."
)

print(
    f"\nCurrent POI categories discovered: "
    f"{len(observed_poi_categories):,}"
)

print(
    "\nThe stable POI content fingerprint will "
    "be used for downstream change detection."
)

CELL 24 - LOAD AND VERIFY THE PROCESSED POI DATASET

Available GeoPackage layers:
  - combined_pois

POI category summary:


,poi_category,POI Features
0,airports,27
1,attractions,302
2,beaches,481
3,bus_stations,23
4,bus_stops,11117
5,cafes,1704
6,fast_foods,1632
7,gyms,136
8,holiday_parks,195
9,hospitals,78



POI validation summary:


,Check,Result
0,GeoPackage layers,1
1,Selected layer,combined_pois
2,POI features,23948
3,POI columns,8
4,Observed POI categories,24
5,Missing category values,0
6,Missing geometries,0
7,Empty geometries,0
8,Invalid geometries,0
9,Coordinate reference system,EPSG:32630



The processed POI dataset was loaded and verified successfully.

Current POI categories discovered: 24

The stable POI content fingerprint will be used for downstream change detection.


## What Cell 24 Does

This cell loads the POI dataset created in `1_POI_Dataset_Overpass` (Notebook 1).

The dataset is checked to make sure:

- the required POI information is available;
- all POI categories are identified;
- the locations have valid geometries;
- the coordinate system is UTM Zone 30N (EPSG:32630), so distances can be calculated in metres.

The cell also creates a fingerprint of the POI dataset. This acts as a unique identifier for its current contents.

Later cells use this fingerprint to determine whether the POI dataset has changed since the previous run.

If the POI dataset has not changed, existing POI distance calculations can be reused. If POIs have been added, removed or changed in `1_POI_Dataset_Overpass` (Notebook 1), the notebook detects this and updates the required POI information.

This allows the notebook to remain up to date while avoiding unnecessary recalculation of POI distances for the BT grid locations.

In [26]:
# ================================================================
# CELL 25 - DEFINE THE POI FEATURE STRUCTURE
# ================================================================

# ------------------------------------------------
# Use every POI category discovered in Cell 24
# ------------------------------------------------

POI_CATEGORIES = list(
    observed_poi_categories
)


# Retain the previous variable name for compatibility
# with later notebook cells.
EXPECTED_POI_CATEGORIES = (
    POI_CATEGORIES
)


# ------------------------------------------------
# Convert category names into safe column-name keys
# ------------------------------------------------

def make_poi_category_key(
    category_name
):

    category_key = (
        re.sub(
            r"[^a-z0-9]+",
            "_",
            str(
                category_name
            )
            .strip()
            .lower()
        )
        .strip(
            "_"
        )
    )


    if not category_key:

        raise ValueError(
            f"POI category {category_name!r} cannot "
            "be converted into a valid column name."
        )


    return category_key


POI_CATEGORY_KEYS = {
    category_name: (
        make_poi_category_key(
            category_name
        )
    )
    for category_name
    in POI_CATEGORIES
}


if len(
    set(
        POI_CATEGORY_KEYS.values()
    )
) != len(
    POI_CATEGORY_KEYS
):

    raise ValueError(
        "Two or more POI categories become the same "
        "column name after normalisation."
    )


POI_DISTANCE_COLUMN_BY_CATEGORY = {
    category_name: (
        f"nearest_"
        f"{POI_CATEGORY_KEYS[category_name]}"
        f"_distance"
    )
    for category_name
    in POI_CATEGORIES
}


POI_DISTANCE_COLUMNS = [
    POI_DISTANCE_COLUMN_BY_CATEGORY[
        category_name
    ]
    for category_name
    in POI_CATEGORIES
]


# ------------------------------------------------
# Existing transport-hub definition
# ------------------------------------------------

BASE_TRANSPORT_HUB_CATEGORIES = [
    "railway_stations",
    "bus_stations",
    "bus_stops",
    "airports"
]


TRANSPORT_HUB_CATEGORIES = [
    category_name
    for category_name
    in BASE_TRANSPORT_HUB_CATEGORIES
    if category_name
    in POI_CATEGORIES
]


if len(
    TRANSPORT_HUB_CATEGORIES
) == 0:

    raise ValueError(
        "None of the configured transport-hub POI "
        "categories are present in the current POI dataset."
    )


# ------------------------------------------------
# Additional overall POI fields
# ------------------------------------------------

ADDITIONAL_POI_COLUMNS = [
    "nearest_transport_hub_distance",
    "nearest_poi_distance",
    "nearest_poi_type",
    "nearest_poi_name"
]


POI_FEATURE_COLUMNS = (
    POI_DISTANCE_COLUMNS
    +
    ADDITIONAL_POI_COLUMNS
)


POI_LOOKUP_COLUMNS = [
    "location_id",
    "xbin",
    "ybin",
    *POI_FEATURE_COLUMNS
]


EXPECTED_POI_CATEGORY_DISTANCE_COUNT = (
    len(
        POI_CATEGORIES
    )
)


EXPECTED_POI_COLUMN_COUNT = (
    len(
        POI_FEATURE_COLUMNS
    )
)


EXPECTED_LOCATION_AND_POI_COLUMN_COUNT = (
    1
    +
    EXPECTED_POI_COLUMN_COUNT
)


EXPECTED_FINAL_MODEL_COLUMN_COUNT = (
    len(
        CLEANED_BT_COLUMNS
    )
    +
    len(
        POI_FEATURE_COLUMNS
    )
)


named_source_pois = int(
    poi_gdf[
        "poi_name"
    ]
    .notna()
    .sum()
)


unnamed_source_pois = int(
    poi_gdf[
        "poi_name"
    ]
    .isna()
    .sum()
)


poi_feature_structure = pd.DataFrame(
    {
        "Feature Group": [
            "Current POI categories",
            "Category-specific nearest distances",
            "Nearest transport-hub distance",
            "Overall nearest-POI distance",
            "Overall nearest-POI type",
            "Overall nearest-POI name",
            "Total POI context fields",
            "Location lookup columns",
            "Final model dataset columns"
        ],
        "Columns": [
            len(
                POI_CATEGORIES
            ),
            len(
                POI_DISTANCE_COLUMNS
            ),
            1,
            1,
            1,
            1,
            len(
                POI_FEATURE_COLUMNS
            ),
            len(
                POI_LOOKUP_COLUMNS
            ),
            EXPECTED_FINAL_MODEL_COLUMN_COUNT
        ]
    }
)


print("=" * 80)
print("CELL 25 - DEFINE THE POI FEATURE STRUCTURE")
print("=" * 80)


display(
    poi_feature_structure
)


print(
    "\nPOI categories and distance columns:"
)


display(
    pd.DataFrame(
        {
            "POI Category": (
                POI_CATEGORIES
            ),
            "Distance Column": [
                POI_DISTANCE_COLUMN_BY_CATEGORY[
                    category_name
                ]
                for category_name
                in POI_CATEGORIES
            ]
        }
    )
)


print(
    "\nPOI naming coverage:"
)


display(
    pd.DataFrame(
        {
            "POI Name Status": [
                "Named source POIs",
                "Unnamed source POIs",
                "Total source POIs"
            ],
            "POI Features": [
                named_source_pois,
                unnamed_source_pois,
                len(
                    poi_gdf
                )
            ]
        }
    )
)


print(
    "\nTransport-hub categories:"
)


for category_name in (
    TRANSPORT_HUB_CATEGORIES
):

    print(
        f"  - {category_name}"
    )


assert len(
    POI_CATEGORIES
) > 0


assert len(
    POI_DISTANCE_COLUMNS
) == len(
    POI_CATEGORIES
)


assert len(
    POI_FEATURE_COLUMNS
) == (
    len(
        POI_CATEGORIES
    )
    +
    4
)


assert len(
    POI_LOOKUP_COLUMNS
) == (
    len(
        POI_CATEGORIES
    )
    +
    7
)


assert EXPECTED_FINAL_MODEL_COLUMN_COUNT == (
    len(
        CLEANED_BT_COLUMNS
    )
    +
    len(
        POI_FEATURE_COLUMNS
    )
)


print(
    f"\nThe dynamic POI feature structure was "
    f"defined successfully using "
    f"{len(POI_CATEGORIES):,} categories."
)

CELL 25 - DEFINE THE POI FEATURE STRUCTURE


,Feature Group,Columns
0,Current POI categories,24
1,Category-specific nearest distances,24
2,Nearest transport-hub distance,1
3,Overall nearest-POI distance,1
4,Overall nearest-POI type,1
5,Overall nearest-POI name,1
6,Total POI context fields,28
7,Location lookup columns,31
8,Final model dataset columns,62



POI categories and distance columns:


,POI Category,Distance Column
0,airports,nearest_airports_distance
1,attractions,nearest_attractions_distance
2,beaches,nearest_beaches_distance
3,bus_stations,nearest_bus_stations_distance
4,bus_stops,nearest_bus_stops_distance
5,cafes,nearest_cafes_distance
6,fast_foods,nearest_fast_foods_distance
7,gyms,nearest_gyms_distance
8,holiday_parks,nearest_holiday_parks_distance
9,hospitals,nearest_hospitals_distance



POI naming coverage:


,POI Name Status,POI Features
0,Named source POIs,23948
1,Unnamed source POIs,0
2,Total source POIs,23948



Transport-hub categories:
  - railway_stations
  - bus_stations
  - bus_stops
  - airports

The dynamic POI feature structure was defined successfully using 24 categories.


## What Cell 25 Does

This cell dynamically defines the POI feature structure from the categories discovered in the current processed POI dataset.

One nearest-distance column is created for every available POI category.

Four additional context fields are also defined:

- nearest transport-hub distance;
- overall nearest-POI distance;
- overall nearest-POI type;
- overall nearest-POI name.

The number of POI context fields and the number of columns in the final model-ready dataset therefore update automatically when the POI categories change.

POIs without a recorded name can still provide their type and distance.

No fixed-distance yes-or-no columns are created because distance thresholds can be calculated later from the stored distances.

# 8. POI Feature Creation

This section creates the POI information required by the current BT grid locations.

The notebook first compares the current POI source with its saved state.

If the POI dataset is unchanged, previously calculated location information can be reused and distances are calculated only for genuinely new BT grid locations.

If the POI dataset has changed, the previous lookup is treated as outdated and POI information is recalculated for all BT locations required by the current data.

This ensures that additions or changes to the POI dataset are reflected throughout the model-ready data.

In [27]:
# ================================================================
# CELL 26 - CONFIGURE THE POI CALCULATION
# ================================================================

from shapely.strtree import STRtree


POI_CALCULATION_BATCH_SIZE = (
    250_000
)


POI_SOURCE_STATE_FILE = (
    POI_LOCATION_FEATURES_FOLDER
    /
    "BT_Grid_POI_Source_State.json"
)


# ------------------------------------------------
# Inspect the existing POI lookup
# ------------------------------------------------

lookup_file_exists = (
    POI_LOCATION_FEATURES_FILE.exists()
)


existing_lookup_schema_valid = False
existing_lookup_rows_on_disk = 0


if lookup_file_exists:

    try:

        existing_lookup_metadata = (
            pq.read_metadata(
                POI_LOCATION_FEATURES_FILE
            )
        )


        existing_lookup_schema = (
            pq.read_schema(
                POI_LOCATION_FEATURES_FILE
            )
        )


        existing_lookup_rows_on_disk = int(
            existing_lookup_metadata.num_rows
        )


        existing_lookup_schema_valid = (
            existing_lookup_schema.names
            ==
            POI_LOOKUP_COLUMNS
        )


    except Exception:

        existing_lookup_schema_valid = False


# ------------------------------------------------
# Load the saved POI source state
# ------------------------------------------------

saved_poi_source_state = None


if POI_SOURCE_STATE_FILE.exists():

    try:

        with POI_SOURCE_STATE_FILE.open(
            "r",
            encoding="utf-8"
        ) as state_handle:

            saved_poi_source_state = (
                json.load(
                    state_handle
                )
            )

    except Exception:

        saved_poi_source_state = None


saved_poi_source_fingerprint = (
    saved_poi_source_state.get(
        "sha256"
    )
    if isinstance(
        saved_poi_source_state,
        dict
    )
    else None
)


# ------------------------------------------------
# Compare the saved fingerprint with both the new
# stable POI-content fingerprint and the legacy
# physical GeoPackage fingerprint.
#
# This allows an existing state created before the
# content-fingerprint change to be migrated without
# forcing another unnecessary full POI recalculation.
# ------------------------------------------------

poi_source_state_matches = (
    saved_poi_source_fingerprint
    in {
        POI_SOURCE_FINGERPRINT,
        POI_SOURCE_FILE_FINGERPRINT
    }
)


# ------------------------------------------------
# Legacy compatibility
# ------------------------------------------------
# Existing projects created before this fingerprint system
# do not yet have a state JSON file. If the existing lookup
# already has exactly the structure required by the current
# POI dataset, accept it once and create the state file later.

poi_source_state_bootstrap = (
    saved_poi_source_state is None
    and
    lookup_file_exists
    and
    existing_lookup_schema_valid
)


existing_lookup_reusable = (
    lookup_file_exists
    and
    existing_lookup_schema_valid
    and
    (
        poi_source_state_matches
        or
        poi_source_state_bootstrap
    )
)


POI_RECALCULATE_ALL_LOCATIONS = (
    not existing_lookup_reusable
)


poi_source_changed = (
    saved_poi_source_state is not None
    and
    not poi_source_state_matches
)


# ------------------------------------------------
# Load reusable lookup or start a fresh one
# ------------------------------------------------

if existing_lookup_reusable:

    existing_poi_lookup = (
        pd.read_parquet(
            POI_LOCATION_FEATURES_FILE,
            engine="pyarrow"
        )
    )


    if not existing_poi_lookup[
        "location_id"
    ].is_unique:

        raise ValueError(
            "The existing POI lookup contains duplicate "
            "location identifiers."
        )


else:

    existing_poi_lookup = (
        pd.DataFrame(
            columns=POI_LOOKUP_COLUMNS
        )
    )


# ------------------------------------------------
# Determine BT locations requiring calculation
# ------------------------------------------------

if POI_RECALCULATE_ALL_LOCATIONS:

    new_bt_locations = (
        unique_bt_locations
        .copy()
        .reset_index(
            drop=True
        )
    )


else:

    existing_poi_location_ids = set(
        existing_poi_lookup[
            "location_id"
        ]
        .dropna()
    )


    new_bt_locations = (
        unique_bt_locations.loc[
            ~unique_bt_locations[
                "location_id"
            ]
            .isin(
                existing_poi_location_ids
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------
# Identify old stored locations not currently required
# ------------------------------------------------

if existing_lookup_reusable:

    existing_locations_not_in_current_data = (
        existing_poi_lookup.loc[
            ~existing_poi_lookup[
                "location_id"
            ]
            .isin(
                unique_bt_locations[
                    "location_id"
                ]
            ),
            "location_id"
        ]
        .tolist()
    )


else:

    existing_locations_not_in_current_data = []


# ------------------------------------------------
# Create geometry for locations requiring calculation
# ------------------------------------------------

new_grid_locations_gdf = (
    gpd.GeoDataFrame(
        new_bt_locations.copy(),
        geometry=gpd.points_from_xy(
            new_bt_locations[
                "xbin"
            ],
            new_bt_locations[
                "ybin"
            ]
        ),
        crs=BT_CRS
    )
)


# ------------------------------------------------
# Prepare current POI geometry collections
# ------------------------------------------------

poi_calculation_source = (
    poi_gdf[
        [
            "poi_category",
            "poi_name",
            "geometry"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


poi_calculation_source[
    "poi_category"
] = (
    poi_calculation_source[
        "poi_category"
    ]
    .astype(
        "string"
    )
)


poi_calculation_source[
    "poi_name"
] = (
    poi_calculation_source[
        "poi_name"
    ]
    .astype(
        "string"
    )
)


poi_category_tables = {
    category_name: (
        poi_calculation_source.loc[
            poi_calculation_source[
                "poi_category"
            ]
            ==
            category_name
        ]
        .reset_index(
            drop=True
        )
    )
    for category_name
    in POI_CATEGORIES
}


poi_category_geometries = {
    category_name: (
        category_table[
            "geometry"
        ]
        .to_numpy()
    )
    for category_name, category_table
    in poi_category_tables.items()
}


empty_poi_categories = [
    category_name
    for category_name, category_geometries
    in poi_category_geometries.items()
    if len(
        category_geometries
    ) == 0
]


overall_poi_geometries = (
    poi_calculation_source[
        "geometry"
    ]
    .to_numpy()
)


overall_poi_categories = (
    poi_calculation_source[
        "poi_category"
    ]
    .to_numpy(
        dtype=object
    )
)


overall_poi_names = (
    poi_calculation_source[
        "poi_name"
    ]
    .to_numpy(
        dtype=object
    )
)


if POI_RECALCULATE_ALL_LOCATIONS:

    calculation_mode = (
        "Recalculate all current BT locations"
    )


elif len(
    new_bt_locations
) > 0:

    calculation_mode = (
        "Calculate new BT locations only"
    )


else:

    calculation_mode = (
        "Reuse complete existing POI lookup"
    )


poi_calculation_configuration = (
    pd.DataFrame(
        {
            "Setting": [
                "Current unique BT locations",
                "Existing lookup file",
                "Existing lookup rows",
                "Existing lookup schema valid",
                "Saved POI source state available",
                "POI source fingerprint matches",
                "Legacy state bootstrap",
                "POI source changed",
                "Calculation mode",
                "Locations requiring calculation",
                "POI categories",
                "POI context fields",
                "Calculation batch size",
                "Distance units"
            ],
            "Value": [
                len(
                    unique_bt_locations
                ),
                lookup_file_exists,
                existing_lookup_rows_on_disk,
                existing_lookup_schema_valid,
                (
                    saved_poi_source_state
                    is not None
                ),
                poi_source_state_matches,
                poi_source_state_bootstrap,
                poi_source_changed,
                calculation_mode,
                len(
                    new_bt_locations
                ),
                len(
                    POI_CATEGORIES
                ),
                len(
                    POI_FEATURE_COLUMNS
                ),
                POI_CALCULATION_BATCH_SIZE,
                "metres"
            ]
        }
    )
)


print("=" * 80)
print("CELL 26 - CONFIGURE THE POI CALCULATION")
print("=" * 80)


display(
    poi_calculation_configuration
)


if poi_source_changed:

    print(
        "\nThe processed POI dataset has changed."
    )

    print(
        "All current BT locations will therefore "
        "receive newly calculated POI information."
    )


elif poi_source_state_bootstrap:

    print(
        "\nThe existing POI lookup predates the "
        "fingerprint system."
    )

    print(
        "Its structure matches the current POI "
        "dataset, so it will be reused and a state "
        "record will be created."
    )


elif len(
    new_bt_locations
) > 0:

    print(
        "\nThe POI source is unchanged."
    )

    print(
        "Only newly discovered BT grid locations "
        "require POI calculation."
    )


else:

    print(
        "\nThe POI source and required BT locations "
        "are unchanged."
    )

    print(
        "The existing POI lookup will be reused."
    )


assert (
    poi_gdf.crs.to_epsg()
    ==
    32630
)


assert len(
    empty_poi_categories
) == 0


assert new_bt_locations[
    "location_id"
].is_unique


assert len(
    overall_poi_geometries
) == len(
    poi_calculation_source
)


print(
    "\nThe POI calculation requirements were "
    "configured successfully."
)

CELL 26 - CONFIGURE THE POI CALCULATION


,Setting,Value
0,Current unique BT locations,2942185
1,Existing lookup file,False
2,Existing lookup rows,0
3,Existing lookup schema valid,False
4,Saved POI source state available,False
5,POI source fingerprint matches,False
6,Legacy state bootstrap,False
7,POI source changed,False
8,Calculation mode,Recalculate all current BT locations
9,Locations requiring calculation,2942185



The POI source is unchanged.
Only newly discovered BT grid locations require POI calculation.

The POI calculation requirements were configured successfully.


## What Cell 26 Does

This cell decides whether existing POI calculations can be reused.

The current POI fingerprint and required feature structure are compared with the saved POI state and existing location lookup.

Three outcomes are possible:

- reuse the complete existing lookup when both the POI source and BT locations are unchanged;
- calculate only newly appearing BT grid locations when the POI source is unchanged;
- recalculate all current BT grid locations when the processed POI source or required POI structure has changed.

The current POI geometries are also prepared for the nearest-distance calculations performed in the next cell.

In [28]:
# ================================================================
# CELL 27 - CALCULATE REQUIRED POI FEATURES
# ================================================================

poi_calculation_start_time = (
    time.time()
)


def query_nearest_in_batches(
    source_geometries,
    target_geometries,
    batch_size,
    return_target_indices=False
):
    """
    Find the nearest target geometry for every source geometry.
    """

    target_tree = STRtree(
        target_geometries
    )


    distance_values = np.empty(
        len(
            source_geometries
        ),
        dtype="float64"
    )


    if return_target_indices:

        target_index_values = (
            np.empty(
                len(
                    source_geometries
                ),
                dtype="int64"
            )
        )

    else:

        target_index_values = None


    for batch_start in range(
        0,
        len(
            source_geometries
        ),
        batch_size
    ):

        batch_end = min(
            batch_start
            +
            batch_size,
            len(
                source_geometries
            )
        )


        batch_geometries = (
            source_geometries[
                batch_start:batch_end
            ]
        )


        (
            nearest_indices,
            nearest_distances
        ) = target_tree.query_nearest(
            batch_geometries,
            all_matches=False,
            return_distance=True
        )


        batch_source_positions = (
            nearest_indices[
                0
            ]
        )


        batch_target_positions = (
            nearest_indices[
                1
            ]
        )


        batch_distance_values = (
            np.empty(
                len(
                    batch_geometries
                ),
                dtype="float64"
            )
        )


        batch_distance_values[
            batch_source_positions
        ] = nearest_distances


        distance_values[
            batch_start:batch_end
        ] = batch_distance_values


        if return_target_indices:

            batch_target_values = (
                np.empty(
                    len(
                        batch_geometries
                    ),
                    dtype="int64"
                )
            )


            batch_target_values[
                batch_source_positions
            ] = batch_target_positions


            target_index_values[
                batch_start:batch_end
            ] = batch_target_values


    if return_target_indices:

        return (
            distance_values,
            target_index_values
        )


    return distance_values


print("=" * 80)
print("CELL 27 - CALCULATE REQUIRED POI FEATURES")
print("=" * 80)


if len(
    new_bt_locations
) == 0:

    new_poi_location_features = (
        pd.DataFrame(
            columns=(
                POI_LOOKUP_COLUMNS
            )
        )
    )


    poi_distance_calculation_summary = (
        pd.DataFrame(
            columns=[
                "POI Category",
                "Locations",
                "Processing Time (Seconds)"
            ]
        )
    )


    print(
        "\nNo POI distance calculations are required."
    )


    print(
        "The existing POI lookup will be reused."
    )


else:

    new_poi_location_features = (
        new_bt_locations[
            [
                "location_id",
                "xbin",
                "ybin"
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    new_grid_geometry_array = (
        new_grid_locations_gdf
        .geometry
        .to_numpy()
    )


    poi_distance_records = []


    print(
        f"\nBT locations requiring POI calculation: "
        f"{len(new_grid_geometry_array):,}"
    )


    print(
        f"POI categories to calculate: "
        f"{len(POI_CATEGORIES):,}"
    )


    for (
        category_number,
        category_name
    ) in enumerate(
        POI_CATEGORIES,
        start=1
    ):

        category_start_time = (
            time.time()
        )


        distance_column = (
            POI_DISTANCE_COLUMN_BY_CATEGORY[
                category_name
            ]
        )


        category_distances = (
            query_nearest_in_batches(
                source_geometries=(
                    new_grid_geometry_array
                ),
                target_geometries=(
                    poi_category_geometries[
                        category_name
                    ]
                ),
                batch_size=(
                    POI_CALCULATION_BATCH_SIZE
                ),
                return_target_indices=False
            )
        )


        new_poi_location_features[
            distance_column
        ] = category_distances


        category_seconds = (
            time.time()
            -
            category_start_time
        )


        poi_distance_records.append(
            {
                "Category Number": (
                    category_number
                ),
                "POI Category": (
                    category_name
                ),
                "Locations": len(
                    new_grid_geometry_array
                ),
                "Minimum Distance": float(
                    np.min(
                        category_distances
                    )
                ),
                "Median Distance": float(
                    np.median(
                        category_distances
                    )
                ),
                "Maximum Distance": float(
                    np.max(
                        category_distances
                    )
                ),
                "Processing Time (Seconds)": round(
                    category_seconds,
                    2
                )
            }
        )


        print(
            f"[{category_number:02d}/"
            f"{len(POI_CATEGORIES):02d}] "
            f"{category_name} | "
            f"{category_seconds:,.2f} seconds"
        )


        del category_distances

        gc.collect()


    transport_distance_columns = [
        POI_DISTANCE_COLUMN_BY_CATEGORY[
            category_name
        ]
        for category_name
        in TRANSPORT_HUB_CATEGORIES
    ]


    new_poi_location_features[
        "nearest_transport_hub_distance"
    ] = (
        new_poi_location_features[
            transport_distance_columns
        ]
        .min(
            axis=1
        )
    )


    (
        overall_nearest_distances,
        overall_nearest_target_positions
    ) = query_nearest_in_batches(
        source_geometries=(
            new_grid_geometry_array
        ),
        target_geometries=(
            overall_poi_geometries
        ),
        batch_size=(
            POI_CALCULATION_BATCH_SIZE
        ),
        return_target_indices=True
    )


    new_poi_location_features[
        "nearest_poi_distance"
    ] = (
        overall_nearest_distances
    )


    new_poi_location_features[
        "nearest_poi_type"
    ] = pd.Series(
        overall_poi_categories[
            overall_nearest_target_positions
        ],
        dtype="string"
    )


    new_poi_location_features[
        "nearest_poi_name"
    ] = pd.Series(
        overall_poi_names[
            overall_nearest_target_positions
        ],
        dtype="string"
    )


    new_poi_location_features = (
        new_poi_location_features[
            POI_LOOKUP_COLUMNS
        ]
    )


    poi_distance_calculation_summary = (
        pd.DataFrame(
            poi_distance_records
        )
    )


    print(
        "\nPOI distance calculation summary:"
    )


    display(
        poi_distance_calculation_summary
    )


poi_calculation_seconds = (
    time.time()
    -
    poi_calculation_start_time
)


print(
    f"\nPOI lookup rows calculated: "
    f"{len(new_poi_location_features):,}"
)


print(
    f"Processing time: "
    f"{poi_calculation_seconds / 60:,.2f} minutes"
)


assert len(
    new_poi_location_features
) == len(
    new_bt_locations
)


assert list(
    new_poi_location_features.columns
) == POI_LOOKUP_COLUMNS


print(
    "\nThe required POI information was "
    "calculated successfully."
)

CELL 27 - CALCULATE REQUIRED POI FEATURES

BT locations requiring POI calculation: 2,942,185
POI categories to calculate: 24
[01/24] airports | 4.12 seconds
[02/24] attractions | 5.99 seconds
[03/24] beaches | 6.46 seconds
[04/24] bus_stations | 3.96 seconds
[05/24] bus_stops | 10.47 seconds
[06/24] cafes | 7.60 seconds
[07/24] fast_foods | 6.96 seconds
[08/24] gyms | 5.45 seconds
[09/24] holiday_parks | 5.74 seconds
[10/24] hospitals | 5.15 seconds
[11/24] hotels | 6.33 seconds
[12/24] offices | 7.56 seconds
[13/24] parks | 7.03 seconds
[14/24] pubs | 7.06 seconds
[15/24] railway_stations | 5.07 seconds
[16/24] restaurants | 6.97 seconds
[17/24] schools | 6.56 seconds
[18/24] shopping_malls | 3.81 seconds
[19/24] sport_centres | 5.66 seconds
[20/24] stadiums | 2.85 seconds
[21/24] supermarkets | 6.05 seconds
[22/24] theme_parks | 4.02 seconds
[23/24] universities | 3.90 seconds
[24/24] zoos | 3.66 seconds

POI distance calculation summary:


,Category Number,POI Category,Locations,Minimum Distance,Median Distance,Maximum Distance,Processing Time (Seconds)
0,1,airports,2942185,4.489850,"9,635.045026","40,373.107832",4.120000
1,2,attractions,2942185,0.698328,"3,756.760844","34,641.953314",5.990000
2,3,beaches,2942185,2.085036,"5,183.456882","33,922.390315",6.460000
3,4,bus_stations,2942185,9.327939,"12,530.265792","51,123.522884",3.960000
4,5,bus_stops,2942185,0.299443,"1,937.571130","33,056.658263",10.470000
5,6,cafes,2942185,0.647783,"2,577.427879","33,423.235158",7.600000
6,7,fast_foods,2942185,1.101064,"4,307.428302","33,403.469886",6.960000
7,8,gyms,2942185,2.329526,"7,387.635483","33,890.615944",5.450000
8,9,holiday_parks,2942185,3.939995,"4,407.348444","34,369.189272",5.740000
9,10,hospitals,2942185,6.311279,"7,095.991789","41,278.650174",5.150000



POI lookup rows calculated: 2,942,185
Processing time: 2.69 minutes

The required POI information was calculated successfully.


## What Cell 27 Does

This cell calculates POI information for the BT locations selected by Cell 26.

For every required BT location, it calculates the nearest distance to each currently available POI category.

It also calculates the nearest transport-hub distance and identifies the overall nearest POI, including its distance, type and available name.

If the existing lookup already covers every required location and the POI source is unchanged, no expensive distance calculation is required.

The calculations are processed in batches so that large numbers of BT locations can be handled without loading unnecessary intermediate data into memory.

In [29]:
# ================================================================
# CELL 28 - BUILD AND VALIDATE THE COMPLETE POI LOOKUP
# ================================================================

if POI_RECALCULATE_ALL_LOCATIONS:

    poi_location_features = (
        new_poi_location_features
        .copy()
        .reset_index(
            drop=True
        )
    )


elif len(
    new_poi_location_features
) > 0:

    poi_location_features = (
        pd.concat(
            [
                existing_poi_lookup,
                new_poi_location_features
            ],
            ignore_index=True
        )
        .drop_duplicates(
            subset=[
                "location_id"
            ],
            keep="last"
        )
        .reset_index(
            drop=True
        )
    )


else:

    poi_location_features = (
        existing_poi_lookup
        .copy()
        .reset_index(
            drop=True
        )
    )


poi_location_features = (
    poi_location_features[
        POI_LOOKUP_COLUMNS
    ]
)


# ------------------------------------------------
# Select records required by current BT files
# ------------------------------------------------

current_poi_location_features = (
    unique_bt_locations[
        [
            "location_id",
            "xbin",
            "ybin"
        ]
    ]
    .merge(
        poi_location_features,
        how="left",
        on=[
            "location_id",
            "xbin",
            "ybin"
        ],
        validate="one_to_one",
        sort=False
    )
)


current_poi_location_features = (
    current_poi_location_features[
        POI_LOOKUP_COLUMNS
    ]
)


poi_numeric_columns = [
    *POI_DISTANCE_COLUMNS,
    "nearest_transport_hub_distance",
    "nearest_poi_distance"
]


current_poi_numeric_matrix = (
    current_poi_location_features[
        poi_numeric_columns
    ]
    .to_numpy(
        dtype="float64",
        copy=False
    )
)


missing_poi_numeric_values = int(
    np.isnan(
        current_poi_numeric_matrix
    ).sum()
)


infinite_poi_numeric_values = int(
    np.isinf(
        current_poi_numeric_matrix
    ).sum()
)


negative_poi_distances = int(
    (
        current_poi_numeric_matrix
        <
        0
    ).sum()
)


missing_nearest_poi_types = int(
    current_poi_location_features[
        "nearest_poi_type"
    ]
    .isna()
    .sum()
)


invalid_nearest_poi_types = int(
    (
        ~current_poi_location_features[
            "nearest_poi_type"
        ]
        .isin(
            POI_CATEGORIES
        )
    ).sum()
)


duplicate_complete_lookup_ids = int(
    poi_location_features[
        "location_id"
    ]
    .duplicated()
    .sum()
)


missing_current_locations = int(
    current_poi_location_features[
        "nearest_poi_distance"
    ]
    .isna()
    .sum()
)


recalculated_nearest_distance = (
    current_poi_location_features[
        POI_DISTANCE_COLUMNS
    ]
    .min(
        axis=1
    )
)


nearest_distance_mismatches = int(
    (
        ~np.isclose(
            recalculated_nearest_distance,
            current_poi_location_features[
                "nearest_poi_distance"
            ],
            rtol=0,
            atol=1e-9
        )
    ).sum()
)


transport_distance_columns = [
    POI_DISTANCE_COLUMN_BY_CATEGORY[
        category_name
    ]
    for category_name
    in TRANSPORT_HUB_CATEGORIES
]


recalculated_transport_distance = (
    current_poi_location_features[
        transport_distance_columns
    ]
    .min(
        axis=1
    )
)


transport_distance_mismatches = int(
    (
        ~np.isclose(
            recalculated_transport_distance,
            current_poi_location_features[
                "nearest_transport_hub_distance"
            ],
            rtol=0,
            atol=1e-9
        )
    ).sum()
)


poi_feature_validation = pd.DataFrame(
    {
        "Check": [
            "Calculation mode",
            "Existing lookup rows reused",
            "POI rows calculated",
            "Complete lookup rows",
            "Current required locations",
            "Current locations matched",
            "POI categories",
            "Lookup columns",
            "POI context fields",
            "Duplicate lookup identifiers",
            "Missing current locations",
            "Missing numeric distances",
            "Infinite numeric distances",
            "Negative numeric distances",
            "Missing nearest-POI types",
            "Invalid nearest-POI types",
            "Nearest-distance mismatches",
            "Transport-distance mismatches"
        ],
        "Result": [
            calculation_mode,
            len(
                existing_poi_lookup
            ),
            len(
                new_poi_location_features
            ),
            len(
                poi_location_features
            ),
            len(
                unique_bt_locations
            ),
            len(
                current_poi_location_features
            ),
            len(
                POI_CATEGORIES
            ),
            poi_location_features.shape[
                1
            ],
            len(
                POI_FEATURE_COLUMNS
            ),
            duplicate_complete_lookup_ids,
            missing_current_locations,
            missing_poi_numeric_values,
            infinite_poi_numeric_values,
            negative_poi_distances,
            missing_nearest_poi_types,
            invalid_nearest_poi_types,
            nearest_distance_mismatches,
            transport_distance_mismatches
        ]
    }
)


print("=" * 80)
print("CELL 28 - BUILD AND VALIDATE THE COMPLETE POI LOOKUP")
print("=" * 80)


display(
    poi_feature_validation
)


assert list(
    poi_location_features.columns
) == POI_LOOKUP_COLUMNS


assert poi_location_features.shape[
    1
] == len(
    POI_LOOKUP_COLUMNS
)


assert duplicate_complete_lookup_ids == 0


assert len(
    current_poi_location_features
) == len(
    unique_bt_locations
)


assert missing_current_locations == 0


assert missing_poi_numeric_values == 0


assert infinite_poi_numeric_values == 0


assert negative_poi_distances == 0


assert missing_nearest_poi_types == 0


assert invalid_nearest_poi_types == 0


assert nearest_distance_mismatches == 0


assert transport_distance_mismatches == 0


print(
    "\nThe complete reusable POI lookup contains "
    "every location required by the current data."
)

CELL 28 - BUILD AND VALIDATE THE COMPLETE POI LOOKUP


,Check,Result
0,Calculation mode,Recalculate all current BT locations
1,Existing lookup rows reused,0
2,POI rows calculated,2942185
3,Complete lookup rows,2942185
4,Current required locations,2942185
5,Current locations matched,2942185
6,POI categories,24
7,Lookup columns,31
8,POI context fields,28
9,Duplicate lookup identifiers,0



The complete reusable POI lookup contains every location required by the current data.


## What Cell 28 Does

This cell builds and validates the complete reusable POI location lookup.

If the POI source changed, the newly calculated information becomes the complete current lookup.

If the POI source is unchanged and only new BT locations were found, their results are added to the existing lookup.

If nothing changed, the existing lookup is reused.

The checks confirm that every BT location required by the current daily data has valid POI distances and a valid nearest-POI type.

The overall nearest-POI and transport-hub distances are also independently checked against the category-specific distance columns.

In [30]:
# ================================================================
# CELL 29 - SAVE OR REUSE THE POI FEATURE LOOKUP
# ================================================================

POI_LOOKUP_TEMPORARY_FILE = (
    POI_LOCATION_FEATURES_FOLDER
    /
    "BT_Grid_POI_Features.temporary.parquet"
)


POI_SOURCE_STATE_TEMPORARY_FILE = (
    POI_LOCATION_FEATURES_FOLDER
    /
    "BT_Grid_POI_Source_State.temporary.json"
)


POI_LOOKUP_REQUIRES_UPDATE = (
    not POI_LOCATION_FEATURES_FILE.exists()
    or
    POI_RECALCULATE_ALL_LOCATIONS
    or
    len(
        new_poi_location_features
    ) > 0
)


# ------------------------------------------------
# Remove stale temporary files only
# ------------------------------------------------

if POI_LOOKUP_TEMPORARY_FILE.exists():

    POI_LOOKUP_TEMPORARY_FILE.unlink()


if POI_SOURCE_STATE_TEMPORARY_FILE.exists():

    POI_SOURCE_STATE_TEMPORARY_FILE.unlink()


# ------------------------------------------------
# Safely create updated lookup when required
# ------------------------------------------------

if POI_LOOKUP_REQUIRES_UPDATE:

    poi_location_features.to_parquet(
        POI_LOOKUP_TEMPORARY_FILE,
        engine="pyarrow",
        compression="snappy",
        index=False,
        row_group_size=250_000
    )


    temporary_poi_metadata = (
        pq.read_metadata(
            POI_LOOKUP_TEMPORARY_FILE
        )
    )


    temporary_poi_schema = (
        pq.read_schema(
            POI_LOOKUP_TEMPORARY_FILE
        )
    )


    assert (
        temporary_poi_metadata.num_rows
        ==
        len(
            poi_location_features
        )
    )


    assert (
        temporary_poi_metadata.num_columns
        ==
        len(
            POI_LOOKUP_COLUMNS
        )
    )


    assert (
        temporary_poi_schema.names
        ==
        POI_LOOKUP_COLUMNS
    )


    del temporary_poi_metadata
    del temporary_poi_schema

    gc.collect()


    # Atomic replacement keeps the previous valid lookup
    # until the new file has been created successfully.
    POI_LOOKUP_TEMPORARY_FILE.replace(
        POI_LOCATION_FEATURES_FILE
    )


    if POI_RECALCULATE_ALL_LOCATIONS:

        poi_lookup_status = (
            "Rebuilt from current POI source"
        )

    else:

        poi_lookup_status = (
            "Extended with new BT locations"
        )


else:

    poi_lookup_status = (
        "Reused without changes"
    )


# ------------------------------------------------
# Save the current POI source state
# ------------------------------------------------

state_to_save = {
    **CURRENT_POI_SOURCE_STATE,
    "poi_feature_columns": (
        POI_FEATURE_COLUMNS
    ),
    "poi_lookup_columns": (
        POI_LOOKUP_COLUMNS
    ),
    "final_model_column_count": (
        EXPECTED_FINAL_MODEL_COLUMN_COUNT
    )
}


with POI_SOURCE_STATE_TEMPORARY_FILE.open(
    "w",
    encoding="utf-8"
) as state_handle:

    json.dump(
        state_to_save,
        state_handle,
        indent=2
    )


POI_SOURCE_STATE_TEMPORARY_FILE.replace(
    POI_SOURCE_STATE_FILE
)


# ------------------------------------------------
# Reload and independently verify saved lookup
# ------------------------------------------------

saved_poi_metadata = pq.read_metadata(
    POI_LOCATION_FEATURES_FILE
)


saved_poi_schema = pq.read_schema(
    POI_LOCATION_FEATURES_FILE
)


saved_poi_lookup = pd.read_parquet(
    POI_LOCATION_FEATURES_FILE,
    engine="pyarrow"
)


saved_current_location_matches = int(
    unique_bt_locations[
        "location_id"
    ]
    .isin(
        saved_poi_lookup[
            "location_id"
        ]
    )
    .sum()
)


saved_poi_lookup_summary = pd.DataFrame(
    {
        "Check": [
            "Lookup status",
            "Output file exists",
            "POI source state exists",
            "POI source fingerprint saved",
            "POI categories",
            "Rows saved",
            "Current locations required",
            "Current locations matched",
            "Columns saved",
            "Expected columns",
            "Correct column order",
            "File size (MB)",
            "Temporary lookup remains",
            "Temporary state remains"
        ],
        "Result": [
            poi_lookup_status,
            POI_LOCATION_FEATURES_FILE.exists(),
            POI_SOURCE_STATE_FILE.exists(),
            POI_SOURCE_FINGERPRINT,
            len(
                POI_CATEGORIES
            ),
            int(
                saved_poi_metadata.num_rows
            ),
            len(
                unique_bt_locations
            ),
            saved_current_location_matches,
            int(
                saved_poi_metadata.num_columns
            ),
            len(
                POI_LOOKUP_COLUMNS
            ),
            (
                saved_poi_schema.names
                ==
                POI_LOOKUP_COLUMNS
            ),
            round(
                POI_LOCATION_FEATURES_FILE
                .stat()
                .st_size
                /
                1_000_000,
                2
            ),
            POI_LOOKUP_TEMPORARY_FILE.exists(),
            POI_SOURCE_STATE_TEMPORARY_FILE.exists()
        ]
    }
)


print("=" * 80)
print("CELL 29 - SAVE OR REUSE THE POI FEATURE LOOKUP")
print("=" * 80)


display(
    saved_poi_lookup_summary
)


assert POI_LOCATION_FEATURES_FILE.exists()


assert POI_SOURCE_STATE_FILE.exists()


assert saved_poi_metadata.num_columns == len(
    POI_LOOKUP_COLUMNS
)


assert saved_poi_schema.names == (
    POI_LOOKUP_COLUMNS
)


assert saved_poi_lookup[
    "location_id"
].is_unique


assert saved_current_location_matches == len(
    unique_bt_locations
)


assert (
    POI_LOCATION_FEATURES_FILE
    .stat()
    .st_size
    >
    0
)


assert not POI_LOOKUP_TEMPORARY_FILE.exists()


assert not POI_SOURCE_STATE_TEMPORARY_FILE.exists()


del saved_poi_metadata
del saved_poi_schema

gc.collect()


print(
    "\nThe reusable POI lookup is ready for "
    "the current daily dataset."
)

CELL 29 - SAVE OR REUSE THE POI FEATURE LOOKUP


,Check,Result
0,Lookup status,Rebuilt from current POI source
1,Output file exists,True
2,POI source state exists,True
3,POI source fingerprint saved,ce81ba194854cffdb292c583e34b5e94a4103f8e7336f0...
4,POI categories,24
5,Rows saved,2942185
6,Current locations required,2942185
7,Current locations matched,2942185
8,Columns saved,31
9,Expected columns,31



The reusable POI lookup is ready for the current daily dataset.


## What Cell 29 Does

This cell saves or reuses the complete POI location lookup.

When the lookup has changed, a temporary Parquet file is written and validated before it replaces the previous saved lookup.

This protects the existing valid output from being removed before the replacement has been created successfully.

The cell also saves a small POI source-state file containing the current source fingerprint, categories and expected feature structure.

If neither the POI source nor the required BT locations have changed, the existing lookup is reused without being rewritten.

These saved resources allow later notebook runs to determine whether expensive POI calculations are actually required.

# 9. Complete POI Join Test

This section tests the current POI feature structure on one complete cleaned BT daily file.

The test confirms that the dynamic POI lookup can be joined to the BT observations without changing the number of observations or introducing additional duplicates.

In [31]:
# ================================================================
# CELL 30 - JOIN POIS TO THE COMPLETE CLEANED TEST FILE
# ================================================================

saved_poi_lookup = pd.read_parquet(
    POI_LOCATION_FEATURES_FILE,
    engine="pyarrow"
)


poi_join_fields = [
    "location_id",
    *POI_FEATURE_COLUMNS
]


test_join_start_time = (
    time.time()
)


test_model_ready_dataframe = (
    test_cleaned_dataframe
    .merge(
        saved_poi_lookup[
            poi_join_fields
        ],
        how="left",
        on="location_id",
        validate="many_to_one",
        sort=False
    )
)


test_join_seconds = (
    time.time()
    -
    test_join_start_time
)


test_unmatched_poi_rows = int(
    test_model_ready_dataframe[
        "nearest_poi_distance"
    ]
    .isna()
    .sum()
)


test_missing_poi_type_rows = int(
    test_model_ready_dataframe[
        "nearest_poi_type"
    ]
    .isna()
    .sum()
)


test_named_nearest_poi_rows = int(
    test_model_ready_dataframe[
        "nearest_poi_name"
    ]
    .notna()
    .sum()
)


test_unnamed_nearest_poi_rows = int(
    test_model_ready_dataframe[
        "nearest_poi_name"
    ]
    .isna()
    .sum()
)


test_duplicate_change = (
    int(
        test_model_ready_dataframe
        .duplicated(
            keep=False
        )
        .sum()
    )
    -
    int(
        test_cleaned_dataframe
        .duplicated(
            keep=False
        )
        .sum()
    )
)


test_poi_join_validation = pd.DataFrame(
    {
        "Check": [
            "Cleaned test rows",
            "Joined test rows",
            "Row difference",
            "Cleaned columns",
            "POI categories",
            "POI fields joined",
            "Final columns",
            "Expected final columns",
            "Unmatched POI rows",
            "Missing nearest-POI types",
            "Rows with a nearest POI name",
            "Rows without a nearest POI name",
            "Change in exact duplicate rows",
            "Join time (seconds)"
        ],
        "Result": [
            len(
                test_cleaned_dataframe
            ),
            len(
                test_model_ready_dataframe
            ),
            (
                len(
                    test_model_ready_dataframe
                )
                -
                len(
                    test_cleaned_dataframe
                )
            ),
            test_cleaned_dataframe.shape[
                1
            ],
            len(
                POI_CATEGORIES
            ),
            len(
                POI_FEATURE_COLUMNS
            ),
            test_model_ready_dataframe.shape[
                1
            ],
            EXPECTED_FINAL_MODEL_COLUMN_COUNT,
            test_unmatched_poi_rows,
            test_missing_poi_type_rows,
            test_named_nearest_poi_rows,
            test_unnamed_nearest_poi_rows,
            test_duplicate_change,
            round(
                test_join_seconds,
                2
            )
        ]
    }
)


print("=" * 80)
print("CELL 30 - JOIN POIS TO THE COMPLETE CLEANED TEST FILE")
print("=" * 80)


display(
    test_poi_join_validation
)


print(
    "\nFirst joined observations:"
)


display(
    test_model_ready_dataframe.head()
)


assert len(
    test_model_ready_dataframe
) == len(
    test_cleaned_dataframe
)


assert (
    test_model_ready_dataframe.shape[
        1
    ]
    ==
    EXPECTED_FINAL_MODEL_COLUMN_COUNT
)


assert test_unmatched_poi_rows == 0


assert test_missing_poi_type_rows == 0


assert test_duplicate_change == 0


assert all(
    column_name
    in test_model_ready_dataframe.columns
    for column_name
    in POI_FEATURE_COLUMNS
)


assert (
    test_named_nearest_poi_rows
    +
    test_unnamed_nearest_poi_rows
) == len(
    test_model_ready_dataframe
)


print(
    "\nThe complete POI join test passed "
    "all validation checks."
)

CELL 30 - JOIN POIS TO THE COMPLETE CLEANED TEST FILE


,Check,Result
0,Cleaned test rows,"1,322,741.000000"
1,Joined test rows,"1,322,741.000000"
2,Row difference,0.000000
3,Cleaned columns,34.000000
4,POI categories,24.000000
5,POI fields joined,28.000000
6,Final columns,62.000000
7,Expected final columns,62.000000
8,Unmatched POI rows,0.000000
9,Missing nearest-POI types,0.000000



First joined observations:


,location_id,xbin,ybin,averagedownlinkthroughput,averageuplinkthroughput,csfallbackattempts,irathandoverattempts,linearaveragersrp,linearaveragersrq,minutesofuse,numberofconnections,pedestrianminutesofuse,s1handoverattempts,s1handoverfailures,s1handoversuccesses,stationaryminutesofuse,totalconnectionblocks,totalconnectiondrops,totalconnectionnormalreleases,totaldownlinkdataduration,totaldownlinkvolume,totalerabblocks,totalerabdrops,totalerabnormalreleases,totaluplinkdataduration,totaluplinkvolume,vehicularminutesofuse,voiceminutesofuse,x2handoverattempts,x2handoverfailures,x2handoversuccesses,indoorminutesofuse,outdoorminutesofuse,en_dt,nearest_airports_distance,nearest_attractions_distance,nearest_beaches_distance,nearest_bus_stations_distance,nearest_bus_stops_distance,nearest_cafes_distance,nearest_fast_foods_distance,nearest_gyms_distance,nearest_holiday_parks_distance,nearest_hospitals_distance,nearest_hotels_distance,nearest_offices_distance,nearest_parks_distance,nearest_pubs_distance,nearest_railway_stations_distance,nearest_restaurants_distance,nearest_schools_distance,nearest_shopping_malls_distance,nearest_sport_centres_distance,nearest_stadiums_distance,nearest_supermarkets_distance,nearest_theme_parks_distance,nearest_universities_distance,nearest_zoos_distance,nearest_transport_hub_distance,nearest_poi_distance,nearest_poi_type,nearest_poi_name
0,530050_5647100,"530,050.000000","5,647,100.000000",43.280000,11.540000,0.000000,NaN,-101.950000,-13.550000,0.490000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.070000,7.000000,0.000000,0.000000,4.000000,0.120000,3.000000,0.490000,0.000000,2.000000,1.000000,1.000000,0.000000,0.490000,2026-04-04,"5,852.298235","7,507.425191","32,637.116985","5,316.089762","2,526.435748","4,090.198512","4,464.492054","5,289.221618","16,743.747031","4,636.681762","4,081.908135","3,974.237272","2,180.663443","1,046.213673","4,411.682890","4,536.791783","1,365.400501","45,630.515686","4,204.840737","31,754.062538","4,164.598692","12,064.751151","54,076.887257","39,811.628217","2,526.435748","1,046.213673",pubs,White Post
1,530100_5606050,"530,100.000000","5,606,050.000000",33.870000,7.120000,0.000000,NaN,-114.080000,-12.730000,0.780000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.000000,0.150000,8.000000,0.000000,0.000000,4.000000,0.510000,6.000000,0.780000,0.000000,0.000000,0.000000,0.000000,0.000000,0.780000,2026-04-04,"9,796.917784","5,750.359144","2,948.378280","37,603.971782","4,774.559023","5,497.671327","5,549.068417","7,380.981363","5,751.151734","6,721.402149","3,065.376696","5,632.575173","5,601.908825","4,376.027738","8,549.297980","6,320.110605","5,576.719252","43,684.241662","7,372.068123","6,505.521056","5,570.405672","9,331.006741","50,136.613635","9,491.516380","4,774.559023","2,948.378280",beaches,Chesil Beach
2,530100_5618850,"530,100.000000","5,618,850.000000",257.350000,67.570000,0.000000,NaN,NaN,NaN,0.590000,1.000000,0.000000,0.000000,0.000000,0.000000,0.590000,0.000000,0.000000,1.000000,0.140000,35.000000,0.000000,0.000000,1.000000,0.220000,15.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2026-04-04,"4,285.178814","3,997.990450","7,713.928336","24,909.187200","1,602.192490","6,301.023919","2,656.668962","7,635.231693","4,370.696155","8,931.586875","9,424.105348","7,270.141122","4,582.697178","2,210.269483","6,628.173671","7,086.004183","2,562.143934","41,726.556433","8,296.640489","9,306.159100","9,052.668643","13,836.639006","47,735.037153","13,940.086238","1,602.192490","1,602.192490",bus_stops,Kingston Russell Turn
3,530100_5631050,"530,100.000000","5,631,050.000000",232.450000,40.910000,0.000000,NaN,-96.420000,-10.890000,155.590000,556.000000,5.690000,0.000000,0.000000,0.000000,75.550000,0.000000,0.000000,350.000000,110.360000,"156,520.000000",1.000000,0.000000,593.000000,223.740000,"30,986.000000",74.360000,0.370000,100.000000,0.000000,100.000000,0.360000,155.230000,2026-04-04


The complete POI join test passed all validation checks.


## What Cell 30 Does

This cell joins the saved POI location lookup to the complete cleaned test file using `location_id`.

Many BT observations can share the same location, while each location appears only once in the POI lookup.

The checks confirm that all observations are retained, no additional duplicate observations are introduced and every row receives the required POI distance and type information.

The number of POI fields and final columns is checked against the dynamic structure created from the current POI dataset.

A nearest-POI name is included where the source POI has an available name.

# 10. Final Model Dataset

This section creates the final daily partitions used by the anomaly-detection notebook.

One final model-ready partition is required for every usable daily BT input file found during the current run.

Valid existing outputs are reused when their structure and saved POI state still match the current data.

If new BT daily files are added, only the additional required outputs need to be created when the POI source is unchanged.

If the processed POI source changes, the existing final partitions are rebuilt so that every observation uses the updated POI information.

In [32]:
# ================================================================
# CELL 31 - CONFIGURE THE FINAL MODEL DATASET
# ================================================================

FINAL_MODEL_FILE_SUFFIX = (
    "_model_ready.parquet"
)


FINAL_MODEL_TEMPORARY_SUFFIX = (
    ".temporary.parquet"
)


FINAL_MODEL_COMPRESSION = (
    "snappy"
)


FINAL_MODEL_ROW_GROUP_SIZE = (
    250_000
)


FINAL_MODEL_BATCH_SIZE = (
    250_000
)


FINAL_MODEL_STATE_FILE = (
    FINAL_MODEL_DATASET_FOLDER
    /
    "_final_model_state.json"
)


FINAL_MODEL_STATE_TEMPORARY_FILE = (
    FINAL_MODEL_DATASET_FOLDER
    /
    "_final_model_state.temporary.json"
)


FINAL_MODEL_COLUMNS = [
    *CLEANED_BT_COLUMNS,
    *POI_FEATURE_COLUMNS
]


FINAL_POI_JOIN_COLUMNS = [
    "location_id",
    *POI_FEATURE_COLUMNS
]


# ------------------------------------------------
# Load previous final-dataset state
# ------------------------------------------------

saved_final_model_state = None


if FINAL_MODEL_STATE_FILE.exists():

    try:

        with FINAL_MODEL_STATE_FILE.open(
            "r",
            encoding="utf-8"
        ) as state_handle:

            saved_final_model_state = (
                json.load(
                    state_handle
                )
            )

    except Exception:

        saved_final_model_state = None


# ------------------------------------------------
# Compare the saved final-model state with both the
# new stable POI-content fingerprint and the legacy
# physical GeoPackage fingerprint.
# ------------------------------------------------

final_model_state_matches = (
    isinstance(
        saved_final_model_state,
        dict
    )
    and
    saved_final_model_state.get(
        "poi_source_sha256"
    )
    in {
        POI_SOURCE_FINGERPRINT,
        POI_SOURCE_FILE_FINGERPRINT
    }
    and
    saved_final_model_state.get(
        "final_model_columns"
    )
    ==
    FINAL_MODEL_COLUMNS
)


# Existing notebooks have no final state file yet.
# When the POI lookup itself was accepted through the
# legacy bootstrap, allow the current final outputs to
# be validated and reused once.

final_model_legacy_bootstrap = (
    saved_final_model_state is None
    and
    poi_source_state_bootstrap
    and
    not POI_RECALCULATE_ALL_LOCATIONS
)


OVERWRITE_FINAL_MODEL_PARTITIONS = (
    not final_model_state_matches
    and
    not final_model_legacy_bootstrap
)


def final_model_output_path(
    cleaned_file
):

    cleaned_stem = (
        cleaned_file.stem
    )


    if cleaned_stem.endswith(
        "_cleaned_bt"
    ):

        cleaned_stem = cleaned_stem[
            :-len(
                "_cleaned_bt"
            )
        ]


    return (
        FINAL_MODEL_DATASET_FOLDER
        /
        (
            cleaned_stem
            +
            FINAL_MODEL_FILE_SUFFIX
        )
    )


def final_model_temporary_path(
    cleaned_file
):

    cleaned_stem = (
        cleaned_file.stem
    )


    if cleaned_stem.endswith(
        "_cleaned_bt"
    ):

        cleaned_stem = cleaned_stem[
            :-len(
                "_cleaned_bt"
            )
        ]


    return (
        FINAL_MODEL_DATASET_FOLDER
        /
        (
            cleaned_stem
            +
            FINAL_MODEL_TEMPORARY_SUFFIX
        )
    )


def validate_final_model_parquet(
    parquet_file,
    expected_rows
):

    if not parquet_file.exists():

        return False


    try:

        parquet_metadata = (
            pq.read_metadata(
                parquet_file
            )
        )


        parquet_schema = (
            pq.read_schema(
                parquet_file
            )
        )


        return (
            parquet_metadata.num_rows
            ==
            expected_rows
            and
            parquet_metadata.num_columns
            ==
            len(
                FINAL_MODEL_COLUMNS
            )
            and
            parquet_schema.names
            ==
            FINAL_MODEL_COLUMNS
        )


    except Exception:

        return False


saved_poi_merge_lookup = (
    saved_poi_lookup[
        FINAL_POI_JOIN_COLUMNS
    ]
    .copy()
    .set_index(
        "location_id",
        verify_integrity=True
    )
)


expected_final_output_paths = [
    final_model_output_path(
        cleaned_file
    )
    for cleaned_file
    in cleaned_partition_files
]


final_model_configuration = pd.DataFrame(
    {
        "Setting": [
            "Current usable input files",
            "Cleaned input partitions",
            "POI source changed",
            "Final state matches POI source",
            "Legacy final-state bootstrap",
            "Existing POI lookup locations",
            "POI categories",
            "POI context fields",
            "Expected final outputs",
            "Expected final columns",
            "Batch size",
            "Rebuild existing final outputs"
        ],
        "Value": [
            len(
                usable_daily_files
            ),
            len(
                cleaned_partition_files
            ),
            poi_source_changed,
            final_model_state_matches,
            final_model_legacy_bootstrap,
            len(
                saved_poi_merge_lookup
            ),
            len(
                POI_CATEGORIES
            ),
            len(
                POI_FEATURE_COLUMNS
            ),
            len(
                expected_final_output_paths
            ),
            len(
                FINAL_MODEL_COLUMNS
            ),
            FINAL_MODEL_BATCH_SIZE,
            OVERWRITE_FINAL_MODEL_PARTITIONS
        ]
    }
)


print("=" * 80)
print("CELL 31 - CONFIGURE THE FINAL MODEL DATASET")
print("=" * 80)


display(
    final_model_configuration
)


assert len(
    FINAL_MODEL_COLUMNS
) == (
    len(
        CLEANED_BT_COLUMNS
    )
    +
    len(
        POI_FEATURE_COLUMNS
    )
)


assert saved_poi_merge_lookup.index.is_unique


assert len(
    expected_final_output_paths
) == len(
    usable_daily_files
)


assert len(
    set(
        expected_final_output_paths
    )
) == len(
    expected_final_output_paths
)


if OVERWRITE_FINAL_MODEL_PARTITIONS:

    print(
        "\nExisting final partitions will be "
        "rebuilt because the saved final-dataset "
        "state does not match the current POI source."
    )


else:

    print(
        "\nValid existing final partitions may be reused."
    )


print(
    "\nOne final output is expected for every "
    "usable daily input in the current run."
)

CELL 31 - CONFIGURE THE FINAL MODEL DATASET


,Setting,Value
0,Current usable input files,90
1,Cleaned input partitions,90
2,POI source changed,False
3,Final state matches POI source,False
4,Legacy final-state bootstrap,False
5,Existing POI lookup locations,2942185
6,POI categories,24
7,POI context fields,28
8,Expected final outputs,90
9,Expected final columns,62



Existing final partitions will be rebuilt because the saved final-dataset state does not match the current POI source.

One final output is expected for every usable daily input in the current run.


## What Cell 31 Does

This cell configures the final model-ready dataset.

Each final partition contains the cleaned BT fields together with all POI context fields dynamically defined from the current POI dataset.

A saved final-dataset state is compared with the current POI fingerprint and required column structure.

If the saved state matches, valid existing final partitions may be reused.

If the state does not match, the current final partitions are rebuilt so that they contain the correct POI information and column structure.

This prevents final files created from an older POI dataset from being silently reused.

In [33]:
# ================================================================
# CELL 32 - CREATE ALL REQUIRED FINAL MODEL PARTITIONS
# ================================================================

final_model_creation_start_time = (
    time.time()
)


final_model_partition_records = []


print("=" * 80)
print("CELL 32 - CREATE ALL REQUIRED FINAL MODEL PARTITIONS")
print("=" * 80)


print(
    f"\nCurrent final partitions required: "
    f"{len(cleaned_partition_files):,}"
)


for (
    partition_number,
    cleaned_file
) in enumerate(
    cleaned_partition_files,
    start=1
):

    partition_start_time = (
        time.time()
    )


    cleaned_metadata = (
        pq.read_metadata(
            cleaned_file
        )
    )


    expected_rows = int(
        cleaned_metadata.num_rows
    )


    output_file = (
        final_model_output_path(
            cleaned_file
        )
    )


    temporary_file = (
        final_model_temporary_path(
            cleaned_file
        )
    )


    if (
        output_file.exists()
        and
        not OVERWRITE_FINAL_MODEL_PARTITIONS
        and
        validate_final_model_parquet(
            output_file,
            expected_rows
        )
    ):

        output_metadata = (
            pq.read_metadata(
                output_file
            )
        )


        processing_status = (
            "Skipped valid output"
        )


        output_rows = int(
            output_metadata.num_rows
        )


        output_columns = int(
            output_metadata.num_columns
        )


        matched_rows = (
            output_rows
        )

        unmatched_rows = 0

        batches_processed = 0


    else:

        if temporary_file.exists():

            temporary_file.unlink()


        parquet_source = (
            pq.ParquetFile(
                cleaned_file
            )
        )


        parquet_writer = None

        output_rows = 0
        matched_rows = 0
        unmatched_rows = 0
        batches_processed = 0


        try:

            for record_batch in (
                parquet_source
                .iter_batches(
                    batch_size=(
                        FINAL_MODEL_BATCH_SIZE
                    )
                )
            ):

                cleaned_batch = (
                    record_batch
                    .to_pandas()
                )


                batch_rows = len(
                    cleaned_batch
                )


                joined_batch = (
                    cleaned_batch
                    .join(
                        saved_poi_merge_lookup,
                        on="location_id",
                        how="left",
                        validate="many_to_one"
                    )
                )


                batch_unmatched_rows = int(
                    joined_batch[
                        "nearest_poi_distance"
                    ]
                    .isna()
                    .sum()
                )


                batch_missing_types = int(
                    joined_batch[
                        "nearest_poi_type"
                    ]
                    .isna()
                    .sum()
                )


                if batch_unmatched_rows > 0:

                    raise ValueError(
                        f"{batch_unmatched_rows:,} rows in "
                        f"{cleaned_file.name} did not match "
                        "the POI lookup."
                    )


                if batch_missing_types > 0:

                    raise ValueError(
                        f"{batch_missing_types:,} rows in "
                        f"{cleaned_file.name} have no "
                        "nearest POI type."
                    )


                if len(
                    joined_batch
                ) != batch_rows:

                    raise AssertionError(
                        "The location join changed the "
                        "batch row count."
                    )


                joined_batch = (
                    joined_batch[
                        FINAL_MODEL_COLUMNS
                    ]
                )


                joined_table = (
                    pa.Table.from_pandas(
                        joined_batch,
                        preserve_index=False
                    )
                )


                if parquet_writer is None:

                    parquet_writer = (
                        pq.ParquetWriter(
                            temporary_file,
                            joined_table.schema,
                            compression=(
                                FINAL_MODEL_COMPRESSION
                            )
                        )
                    )


                parquet_writer.write_table(
                    joined_table,
                    row_group_size=(
                        FINAL_MODEL_ROW_GROUP_SIZE
                    )
                )


                output_rows += (
                    batch_rows
                )


                matched_rows += (
                    batch_rows
                    -
                    batch_unmatched_rows
                )


                unmatched_rows += (
                    batch_unmatched_rows
                )


                batches_processed += 1


                del cleaned_batch
                del joined_batch
                del joined_table

                gc.collect()


            if parquet_writer is not None:

                parquet_writer.close()

                parquet_writer = None


            del parquet_source

            gc.collect()


            assert output_rows == expected_rows


            assert unmatched_rows == 0


            assert validate_final_model_parquet(
                temporary_file,
                expected_rows
            )


            # Replace the previous final file only after
            # the complete temporary file has passed validation.
            temporary_file.replace(
                output_file
            )


            assert validate_final_model_parquet(
                output_file,
                expected_rows
            )


            output_columns = len(
                FINAL_MODEL_COLUMNS
            )


            processing_status = (
                "Created"
            )


        except Exception:

            if parquet_writer is not None:

                parquet_writer.close()


            if "parquet_source" in locals():

                del parquet_source


            gc.collect()


            if temporary_file.exists():

                temporary_file.unlink()


            raise


    partition_seconds = (
        time.time()
        -
        partition_start_time
    )


    final_model_partition_records.append(
        {
            "Partition Number": (
                partition_number
            ),
            "Input Filename": (
                cleaned_file.name
            ),
            "Output Filename": (
                output_file.name
            ),
            "Expected Rows": (
                expected_rows
            ),
            "Output Rows": (
                output_rows
            ),
            "Matched Rows": (
                matched_rows
            ),
            "Unmatched Rows": (
                unmatched_rows
            ),
            "Columns": (
                output_columns
            ),
            "Batches": (
                batches_processed
            ),
            "Status": (
                processing_status
            ),
            "Processing Time (Seconds)": round(
                partition_seconds,
                2
            )
        }
    )


    print(
        f"Partition {partition_number:03d}/"
        f"{len(cleaned_partition_files):03d} | "
        f"{processing_status} | "
        f"{output_rows:,} rows | "
        f"{partition_seconds:,.2f} seconds"
    )


final_model_partition_processing = (
    pd.DataFrame(
        final_model_partition_records
    )
)


final_model_creation_seconds = (
    time.time()
    -
    final_model_creation_start_time
)


print(
    "\nFinal model-dataset creation summary:"
)


display(
    pd.DataFrame(
        {
            "Result": [
                "Required partitions",
                "New or rebuilt partitions",
                "Valid existing outputs reused",
                "Output rows",
                "Unmatched rows",
                "POI categories",
                "POI context fields",
                "Final columns",
                "Processing time (minutes)"
            ],
            "Value": [
                len(
                    final_model_partition_processing
                ),
                int(
                    (
                        final_model_partition_processing[
                            "Status"
                        ]
                        ==
                        "Created"
                    ).sum()
                ),
                int(
                    (
                        final_model_partition_processing[
                            "Status"
                        ]
                        ==
                        "Skipped valid output"
                    ).sum()
                ),
                int(
                    final_model_partition_processing[
                        "Output Rows"
                    ].sum()
                ),
                int(
                    final_model_partition_processing[
                        "Unmatched Rows"
                    ].sum()
                ),
                len(
                    POI_CATEGORIES
                ),
                len(
                    POI_FEATURE_COLUMNS
                ),
                len(
                    FINAL_MODEL_COLUMNS
                ),
                round(
                    final_model_creation_seconds
                    /
                    60,
                    2
                )
            ]
        }
    )
)


assert len(
    final_model_partition_processing
) == len(
    usable_daily_files
)


assert (
    final_model_partition_processing[
        "Expected Rows"
    ]
    ==
    final_model_partition_processing[
        "Output Rows"
    ]
).all()


assert (
    final_model_partition_processing[
        "Unmatched Rows"
    ]
    ==
    0
).all()


assert (
    final_model_partition_processing[
        "Columns"
    ]
    ==
    len(
        FINAL_MODEL_COLUMNS
    )
).all()


print(
    "\nEvery usable daily input now has a "
    "valid final model-ready partition."
)

CELL 32 - CREATE ALL REQUIRED FINAL MODEL PARTITIONS

Current final partitions required: 90
Partition 001/090 | Created | 1,322,741 rows | 21.21 seconds
Partition 002/090 | Created | 1,325,263 rows | 23.09 seconds
Partition 003/090 | Created | 1,371,738 rows | 24.96 seconds
Partition 004/090 | Created | 1,223,418 rows | 20.73 seconds
Partition 005/090 | Created | 1,294,137 rows | 23.62 seconds
Partition 006/090 | Created | 1,215,695 rows | 20.90 seconds
Partition 007/090 | Created | 1,276,920 rows | 23.55 seconds
Partition 008/090 | Created | 1,267,340 rows | 23.33 seconds
Partition 009/090 | Created | 1,232,265 rows | 20.97 seconds
Partition 010/090 | Created | 1,208,420 rows | 20.86 seconds
Partition 011/090 | Created | 1,165,378 rows | 21.25 seconds
Partition 012/090 | Created | 1,244,816 rows | 21.35 seconds
Partition 013/090 | Created | 1,272,228 rows | 24.35 seconds
Partition 014/090 | Created | 1,368,786 rows | 24.80 seconds
Partition 015/090 | Created | 1,247,121 rows | 20.95 s

,Result,Value
0,Required partitions,90.000000
1,New or rebuilt partitions,90.000000
2,Valid existing outputs reused,0.000000
3,Output rows,"119,267,039.000000"
4,Unmatched rows,0.000000
5,POI categories,24.000000
6,POI context fields,28.000000
7,Final columns,62.000000
8,Processing time (minutes),35.760000



Every usable daily input now has a valid final model-ready partition.


## What Cell 32 Does

This cell creates every final model-ready partition required by the current usable BT files.

Each cleaned daily partition is read in manageable batches and joined to the saved POI lookup using `location_id`.

When the saved final-dataset state is current, existing valid outputs are reused and only missing or invalid partitions are created.

When the POI source has changed, the partitions are rebuilt using the updated POI information.

New files are first written to temporary Parquet files and validated before replacing an existing output.

Every completed partition must preserve its original observation count, contain no unmatched locations and use the dynamically defined final column structure.

# 11. Final Verification

This section verifies every model-ready output required by the current BT input data.

The final checks confirm that all usable observations have been preserved, every required partition is valid and the saved POI and final-dataset states are ready for future reuse.

In [34]:
# ================================================================
# CELL 33 - VERIFY ALL REQUIRED FINAL MODEL PARTITIONS
# ================================================================

expected_final_output_set = {
    file_path.resolve()
    for file_path
    in expected_final_output_paths
}


observed_final_output_paths = sorted(
    FINAL_MODEL_DATASET_FOLDER.glob(
        f"*{FINAL_MODEL_FILE_SUFFIX}"
    )
)


observed_final_output_set = {
    file_path.resolve()
    for file_path
    in observed_final_output_paths
}


missing_final_output_paths = sorted(
    expected_final_output_set
    -
    observed_final_output_set
)


unexpected_final_output_paths = sorted(
    observed_final_output_set
    -
    expected_final_output_set
)


final_model_partition_files = [
    file_path
    for file_path
    in expected_final_output_paths
    if file_path.exists()
]


temporary_final_model_files = sorted(
    FINAL_MODEL_DATASET_FOLDER.glob(
        f"*{FINAL_MODEL_TEMPORARY_SUFFIX}"
    )
)


final_partition_verification_records = []


for (
    partition_number,
    cleaned_file
) in enumerate(
    cleaned_partition_files,
    start=1
):

    expected_rows = int(
        pq.read_metadata(
            cleaned_file
        ).num_rows
    )


    final_file = (
        final_model_output_path(
            cleaned_file
        )
    )


    file_exists = (
        final_file.exists()
    )


    readable = False
    observed_rows = 0
    observed_columns = 0
    observed_row_groups = 0
    correct_column_order = False
    file_size_mb = 0
    verification_error = None


    if file_exists:

        try:

            final_metadata = (
                pq.read_metadata(
                    final_file
                )
            )


            final_schema = (
                pq.read_schema(
                    final_file
                )
            )


            observed_rows = int(
                final_metadata.num_rows
            )


            observed_columns = int(
                final_metadata.num_columns
            )


            observed_row_groups = int(
                final_metadata.num_row_groups
            )


            correct_column_order = (
                final_schema.names
                ==
                FINAL_MODEL_COLUMNS
            )


            file_size_mb = round(
                final_file.stat().st_size
                /
                1_000_000,
                2
            )


            readable = True


        except Exception as error:

            verification_error = str(
                error
            )


    valid_partition = (
        file_exists
        and
        readable
        and
        observed_rows
        ==
        expected_rows
        and
        observed_columns
        ==
        len(
            FINAL_MODEL_COLUMNS
        )
        and
        observed_row_groups
        >
        0
        and
        correct_column_order
        and
        file_size_mb
        >
        0
    )


    final_partition_verification_records.append(
        {
            "Partition Number": (
                partition_number
            ),
            "Filename": (
                final_file.name
            ),
            "Expected Rows": (
                expected_rows
            ),
            "Observed Rows": (
                observed_rows
            ),
            "Columns": (
                observed_columns
            ),
            "Row Groups": (
                observed_row_groups
            ),
            "File Size (MB)": (
                file_size_mb
            ),
            "Readable": (
                readable
            ),
            "Correct Column Order": (
                correct_column_order
            ),
            "Valid": (
                valid_partition
            ),
            "Verification Error": (
                verification_error
            )
        }
    )


final_partition_verification = (
    pd.DataFrame(
        final_partition_verification_records
    )
)


verified_final_rows = int(
    final_partition_verification[
        "Observed Rows"
    ].sum()
)


expected_final_rows = int(
    cleaned_partition_verification[
        "Observed Rows"
    ].sum()
)


valid_final_partitions = int(
    final_partition_verification[
        "Valid"
    ].sum()
)


invalid_final_partitions = int(
    (
        ~final_partition_verification[
            "Valid"
        ]
    ).sum()
)


verified_final_size_gb = (
    final_partition_verification[
        "File Size (MB)"
    ].sum()
    /
    1_000
)


print("=" * 80)
print("CELL 33 - VERIFY ALL REQUIRED FINAL MODEL PARTITIONS")
print("=" * 80)


final_verification_summary = pd.DataFrame(
    {
        "Check": [
            "Usable input files",
            "Expected final outputs",
            "Expected outputs found",
            "Missing expected outputs",
            "Unexpected old outputs",
            "Valid final partitions",
            "Invalid final partitions",
            "Expected observations",
            "Verified observations",
            "Row difference",
            "POI categories",
            "POI context fields",
            "Columns per partition",
            "Temporary files",
            "Combined size (GB)"
        ],
        "Result": [
            len(
                usable_daily_files
            ),
            len(
                expected_final_output_paths
            ),
            len(
                final_model_partition_files
            ),
            len(
                missing_final_output_paths
            ),
            len(
                unexpected_final_output_paths
            ),
            valid_final_partitions,
            invalid_final_partitions,
            expected_final_rows,
            verified_final_rows,
            (
                verified_final_rows
                -
                expected_final_rows
            ),
            len(
                POI_CATEGORIES
            ),
            len(
                POI_FEATURE_COLUMNS
            ),
            len(
                FINAL_MODEL_COLUMNS
            ),
            len(
                temporary_final_model_files
            ),
            round(
                verified_final_size_gb,
                2
            )
        ]
    }
)


display(
    final_verification_summary
)


if unexpected_final_output_paths:

    print(
        "\nUnexpected old final outputs "
        "(reported but not used):"
    )


    for file_path in (
        unexpected_final_output_paths
    ):

        print(
            file_path
        )


assert len(
    missing_final_output_paths
) == 0


assert len(
    final_model_partition_files
) == len(
    usable_daily_files
)


assert valid_final_partitions == len(
    usable_daily_files
)


assert invalid_final_partitions == 0


assert verified_final_rows == (
    expected_final_rows
)


assert len(
    temporary_final_model_files
) == 0


# ------------------------------------------------
# Save current final-dataset state only after
# every required final partition passed validation.
# ------------------------------------------------

final_model_state_to_save = {
    "poi_source_sha256": (
        POI_SOURCE_FINGERPRINT
    ),
    "poi_category_count": (
        len(
            POI_CATEGORIES
        )
    ),
    "poi_categories": (
        POI_CATEGORIES
    ),
    "poi_feature_columns": (
        POI_FEATURE_COLUMNS
    ),
    "final_model_columns": (
        FINAL_MODEL_COLUMNS
    ),
    "usable_partition_count": (
        len(
            usable_daily_files
        )
    )
}


if FINAL_MODEL_STATE_TEMPORARY_FILE.exists():

    FINAL_MODEL_STATE_TEMPORARY_FILE.unlink()


with FINAL_MODEL_STATE_TEMPORARY_FILE.open(
    "w",
    encoding="utf-8"
) as state_handle:

    json.dump(
        final_model_state_to_save,
        state_handle,
        indent=2
    )


FINAL_MODEL_STATE_TEMPORARY_FILE.replace(
    FINAL_MODEL_STATE_FILE
)


assert FINAL_MODEL_STATE_FILE.exists()


assert not FINAL_MODEL_STATE_TEMPORARY_FILE.exists()


print(
    "\nEvery usable input has one verified "
    "final model-ready output."
)


print(
    "The final-dataset state now matches "
    "the current POI source."
)

CELL 33 - VERIFY ALL REQUIRED FINAL MODEL PARTITIONS


,Check,Result
0,Usable input files,90.000000
1,Expected final outputs,90.000000
2,Expected outputs found,90.000000
3,Missing expected outputs,0.000000
4,Unexpected old outputs,0.000000
5,Valid final partitions,90.000000
6,Invalid final partitions,0.000000
7,Expected observations,"119,267,039.000000"
8,Verified observations,"119,267,039.000000"
9,Row difference,0.000000



Every usable input has one verified final model-ready output.
The final-dataset state now matches the current POI source.


## What Cell 33 Does

This cell verifies every final output expected from the current usable input files.

It confirms that every usable input has one readable final partition, the row counts match the cleaned source files and every partition contains the complete dynamically defined column structure in the correct order.

Missing, invalid and unexpected old outputs are also checked, and no incomplete temporary files are allowed to remain.

Only after all required final partitions pass verification is the final-dataset state saved for future reuse.

This state records that the completed outputs correspond to the current POI source and feature structure.

In [35]:
# ================================================================
# CELL 34 - DISPLAY THE CURRENT DATASET SUMMARY
# ================================================================

current_poi_lookup = (
    unique_bt_locations[
        [
            "location_id"
        ]
    ]
    .merge(
        saved_poi_lookup[
            [
                "location_id",
                "nearest_poi_type",
                "nearest_poi_name"
            ]
        ],
        how="left",
        on="location_id",
        validate="one_to_one"
    )
)


nearest_poi_type_counts = (
    current_poi_lookup[
        "nearest_poi_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "Nearest POI Type"
    )
    .reset_index(
        name="Unique BT Locations"
    )
)


nearest_poi_type_counts[
    "Percentage of Locations"
] = (
    nearest_poi_type_counts[
        "Unique BT Locations"
    ]
    /
    len(
        current_poi_lookup
    )
    *
    100
)


final_dataset_summary = pd.DataFrame(
    {
        "Measure": [
            "Daily files discovered",
            "Usable daily files",
            "Excluded daily files",
            "Source observations",
            "Final observations",
            "Rows lost or added",
            "Unique BT grid locations",
            "POI source changed this run",
            "POI calculation mode",
            "POI categories",
            "Existing POI rows reused",
            "POI rows calculated",
            "Saved POI lookup rows",
            "Supplied BT columns",
            "Location identifier",
            "POI context fields",
            "Final model columns",
            "Expected final partitions",
            "Verified final partitions",
            "Unexpected old final outputs",
            "Invalid final partitions",
            "Temporary files remaining",
            "Nearest-POI name coverage (%)",
            "Current final dataset size (GB)"
        ],
        "Value": [
            len(
                daily_csv_files
            ),
            len(
                usable_daily_files
            ),
            len(
                excluded_daily_file_audit
            ),
            expected_cleaned_rows,
            verified_final_rows,
            (
                verified_final_rows
                -
                expected_cleaned_rows
            ),
            len(
                unique_bt_locations
            ),
            poi_source_changed,
            calculation_mode,
            len(
                POI_CATEGORIES
            ),
            len(
                existing_poi_lookup
            ),
            len(
                new_poi_location_features
            ),
            len(
                saved_poi_lookup
            ),
            len(
                ORIGINAL_BT_COLUMNS
            ),
            1,
            len(
                POI_FEATURE_COLUMNS
            ),
            len(
                FINAL_MODEL_COLUMNS
            ),
            len(
                expected_final_output_paths
            ),
            len(
                final_model_partition_files
            ),
            len(
                unexpected_final_output_paths
            ),
            invalid_final_partitions,
            (
                len(
                    temporary_final_model_files
                )
                +
                int(
                    POI_LOOKUP_TEMPORARY_FILE.exists()
                )
                +
                int(
                    POI_SOURCE_STATE_TEMPORARY_FILE.exists()
                )
                +
                int(
                    FINAL_MODEL_STATE_TEMPORARY_FILE.exists()
                )
            ),
            round(
                current_poi_lookup[
                    "nearest_poi_name"
                ]
                .notna()
                .mean()
                *
                100,
                2
            ),
            round(
                verified_final_size_gb,
                2
            )
        ]
    }
)


output_location_summary = pd.DataFrame(
    {
        "Output": [
            "Cleaned daily partitions",
            "Reusable POI lookup",
            "POI source state",
            "Final model-ready partitions",
            "Final model state"
        ],
        "Location": [
            str(
                DAILY_CLEANED_PARTITIONS_FOLDER
            ),
            str(
                POI_LOCATION_FEATURES_FILE
            ),
            str(
                POI_SOURCE_STATE_FILE
            ),
            str(
                FINAL_MODEL_DATASET_FOLDER
            ),
            str(
                FINAL_MODEL_STATE_FILE
            )
        ]
    }
)


print("=" * 80)
print("CELL 34 - DISPLAY THE CURRENT DATASET SUMMARY")
print("=" * 80)


display(
    final_dataset_summary
)


print(
    "\nNearest POI types for current locations:"
)


display(
    nearest_poi_type_counts
)


print(
    "\nOutput locations:"
)


display(
    output_location_summary
)


assert verified_final_rows == (
    expected_cleaned_rows
)


assert len(
    FINAL_MODEL_COLUMNS
) == (
    len(
        CLEANED_BT_COLUMNS
    )
    +
    len(
        POI_FEATURE_COLUMNS
    )
)


assert len(
    final_model_partition_files
) == len(
    usable_daily_files
)


assert invalid_final_partitions == 0


print(
    "\nThe current model-ready dataset was "
    "summarised successfully."
)

CELL 34 - DISPLAY THE CURRENT DATASET SUMMARY


,Measure,Value
0,Daily files discovered,95
1,Usable daily files,90
2,Excluded daily files,5
3,Source observations,119267039
4,Final observations,119267039
5,Rows lost or added,0
6,Unique BT grid locations,2942185
7,POI source changed this run,False
8,POI calculation mode,Recalculate all current BT locations
9,POI categories,24



Nearest POI types for current locations:


,Nearest POI Type,Unique BT Locations,Percentage of Locations
0,bus_stops,705894,23.992169
1,beaches,598849,20.353887
2,pubs,302071,10.266893
3,attractions,295405,10.040327
4,cafes,280332,9.528021
5,parks,211888,7.201723
6,schools,171403,5.825704
7,holiday_parks,83897,2.851520
8,offices,56027,1.904265
9,hotels,49850,1.694319



Output locations:


,Output,Location
0,Cleaned daily partitions,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
1,Reusable POI lookup,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
2,POI source state,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
3,Final model-ready partitions,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
4,Final model state,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...



The current model-ready dataset was summarised successfully.


## What Cell 34 Does

This cell summarises the completed dataset for the current run.

It reports the number of discovered and usable BT files, observation totals, unique BT locations, POI calculation mode, current POI categories, POI context fields and final model-ready partitions.

It also reports whether the POI source changed, how many POI location rows were reused or calculated and whether any invalid or temporary outputs remain.

The nearest-POI type distribution and locations of the main reusable outputs are displayed for reference.

The values shown describe the current run and can change automatically when new BT files or an updated POI dataset are supplied.

In [36]:
# ================================================================
# CELL 35 - COMPLETE THE DAILY PIPELINE
# ================================================================

current_usable_partition_count = len(
    usable_daily_files
)


complete_poi_lookup = (
    POI_LOCATION_FEATURES_FILE.exists()
    and
    saved_poi_lookup.shape[
        1
    ]
    ==
    len(
        POI_LOOKUP_COLUMNS
    )
    and
    list(
        saved_poi_lookup.columns
    )
    ==
    POI_LOOKUP_COLUMNS
    and
    saved_poi_lookup[
        "location_id"
    ]
    .is_unique
    and
    unique_bt_locations[
        "location_id"
    ]
    .isin(
        saved_poi_lookup[
            "location_id"
        ]
    )
    .all()
)


complete_final_dataset = (
    len(
        final_model_partition_files
    )
    ==
    current_usable_partition_count
    and
    valid_final_partitions
    ==
    current_usable_partition_count
)


essential_output_checks = pd.DataFrame(
    {
        "Output": [
            "Cleaned daily partitions",
            "POI location-feature lookup",
            "POI source state",
            "Final model-ready partitions",
            "Final model state"
        ],
        "Expected": [
            current_usable_partition_count,
            1,
            1,
            current_usable_partition_count,
            1
        ],
        "Observed": [
            len(
                cleaned_partition_files
            ),
            int(
                POI_LOCATION_FEATURES_FILE.exists()
            ),
            int(
                POI_SOURCE_STATE_FILE.exists()
            ),
            len(
                final_model_partition_files
            ),
            int(
                FINAL_MODEL_STATE_FILE.exists()
            )
        ],
        "Complete": [
            (
                len(
                    cleaned_partition_files
                )
                ==
                current_usable_partition_count
            ),
            complete_poi_lookup,
            POI_SOURCE_STATE_FILE.exists(),
            complete_final_dataset,
            FINAL_MODEL_STATE_FILE.exists()
        ]
    }
)


print("=" * 80)
print("CELL 35 - COMPLETE THE DAILY PIPELINE")
print("=" * 80)


display(
    essential_output_checks
)


assert essential_output_checks[
    "Complete"
].all()


assert expected_cleaned_rows == (
    verified_final_rows
)


assert len(
    ORIGINAL_BT_COLUMNS
) == 33


assert len(
    POI_DISTANCE_COLUMNS
) == len(
    POI_CATEGORIES
)


assert len(
    POI_FEATURE_COLUMNS
) == (
    len(
        POI_CATEGORIES
    )
    +
    4
)


assert len(
    FINAL_MODEL_COLUMNS
) == (
    len(
        CLEANED_BT_COLUMNS
    )
    +
    len(
        POI_FEATURE_COLUMNS
    )
)


assert saved_poi_lookup[
    "location_id"
].is_unique


assert invalid_final_partitions == 0


assert len(
    temporary_cleaned_files
) == 0


assert len(
    temporary_final_model_files
) == 0


assert not POI_LOOKUP_TEMPORARY_FILE.exists()


assert not POI_SOURCE_STATE_TEMPORARY_FILE.exists()


assert not FINAL_MODEL_STATE_TEMPORARY_FILE.exists()


print(
    "\nThe daily preprocessing pipeline completed successfully."
)


print(
    f"\nUsable daily files processed: "
    f"{current_usable_partition_count:,}"
)


print(
    f"POI categories used: "
    f"{len(POI_CATEGORIES):,}"
)


print(
    f"POI context fields: "
    f"{len(POI_FEATURE_COLUMNS):,}"
)


print(
    f"Final model columns: "
    f"{len(FINAL_MODEL_COLUMNS):,}"
)


print(
    f"Final observations: "
    f"{verified_final_rows:,}"
)


print(
    "\nThe final daily Model 1 input is available at:"
)


print(
    FINAL_MODEL_DATASET_FOLDER
)

CELL 35 - COMPLETE THE DAILY PIPELINE


,Output,Expected,Observed,Complete
0,Cleaned daily partitions,90,90,True
1,POI location-feature lookup,1,1,True
2,POI source state,1,1,True
3,Final model-ready partitions,90,90,True
4,Final model state,1,1,True



The daily preprocessing pipeline completed successfully.

Usable daily files processed: 90
POI categories used: 24
POI context fields: 28
Final model columns: 62
Final observations: 119,267,039

The final daily Model 1 input is available at:
C:\Users\adaml\OneDrive\Desktop\BT_Dissertation_AL\data\cleaned_model_data\bt_final_model_dataset


## What Cell 35 Does

This cell performs the final completion checks for the daily preprocessing pipeline.

It confirms that the number of cleaned and final daily outputs matches the number of usable daily input files discovered during the current run.

It also confirms that the shared POI lookup contains every required daily BT grid location, the saved POI and daily final-dataset state files exist and no invalid or incomplete temporary daily outputs remain.

The current number of POI categories, POI context fields, final daily columns and daily observations is displayed.

The location of the completed daily model-ready dataset for Notebook 3 / Model 1 is also shown.

After these checks pass, the notebook continues to the separate hourly preprocessing section.

# Section 2 - Hourly Data

The second part of this notebook prepares the hourly BT dataset used by Model 2.

The hourly workflow is kept separate from the completed daily workflow because the hourly source data contain an additional `time_interval` field and represent observations at a finer temporal resolution.

The workflow dynamically discovers all supported files in `data/bt_hourly_data`, validates their structure, prepares the hourly date, time and location fields, converts the network measurements to numeric form and creates one reusable cleaned Parquet partition for every usable hourly source file.

A stable `location_id` is created using the same BT grid-coordinate convention as the daily dataset. The hourly workflow additionally creates `hourly_datetime` and `hour` so that Model 2 can analyse within-day anomaly behaviour.

Later cells will enrich these cleaned hourly partitions with the same dynamically prepared POI context used by the daily workflow and create the final hourly model-ready dataset.

Existing valid hourly outputs are reused when their corresponding source files have not changed. New or changed hourly files are processed automatically.

In [37]:
# ================================================================
# CELL 36 - CONFIGURE HOURLY PREPROCESSING
# ================================================================

import duckdb


HOURLY_CLEANED_PARTITIONS_FOLDER = (
    CLEANED_MODEL_DATA_FOLDER
    / "hourly_cleaned_partitions"
)

FINAL_HOURLY_MODEL_DATASET_FOLDER = (
    CLEANED_MODEL_DATA_FOLDER
    / "bt_final_hourly_model_dataset"
)

HOURLY_CLEANING_STATE_FILE = (
    HOURLY_CLEANED_PARTITIONS_FOLDER
    / "hourly_cleaning_state.json"
)


HOURLY_CLEANED_PARTITIONS_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_HOURLY_MODEL_DATASET_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


HOURLY_SUPPORTED_EXTENSIONS = {
    ".csv",
    ".parquet"
}

HOURLY_CLEANED_FILE_SUFFIX = (
    "_cleaned_hourly.parquet"
)

HOURLY_TEMPORARY_SUFFIX = (
    ".temporary.parquet"
)

HOURLY_PARQUET_COMPRESSION = (
    "SNAPPY"
)

OVERWRITE_HOURLY_CLEANED_PARTITIONS = False


print("=" * 80)
print("CELL 36 - CONFIGURE HOURLY PREPROCESSING")
print("=" * 80)


hourly_preprocessing_configuration = pd.DataFrame(
    {
        "Setting": [
            "Hourly source folder",
            "Hourly cleaned partitions",
            "Final hourly model dataset",
            "Cleaning state file",
            "Supported source formats",
            "Parquet compression",
            "Overwrite valid outputs"
        ],
        "Value": [
            str(
                HOURLY_INPUT_FOLDER
            ),
            str(
                HOURLY_CLEANED_PARTITIONS_FOLDER
            ),
            str(
                FINAL_HOURLY_MODEL_DATASET_FOLDER
            ),
            str(
                HOURLY_CLEANING_STATE_FILE
            ),
            ", ".join(
                sorted(
                    HOURLY_SUPPORTED_EXTENSIONS
                )
            ),
            HOURLY_PARQUET_COMPRESSION,
            OVERWRITE_HOURLY_CLEANED_PARTITIONS
        ]
    }
)


display(
    hourly_preprocessing_configuration
)


assert HOURLY_INPUT_FOLDER.exists()

assert HOURLY_CLEANED_PARTITIONS_FOLDER.exists()

assert FINAL_HOURLY_MODEL_DATASET_FOLDER.exists()


print(
    "\nThe hourly preprocessing folders and "
    "settings were configured successfully."
)

CELL 36 - CONFIGURE HOURLY PREPROCESSING


,Setting,Value
0,Hourly source folder,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
1,Hourly cleaned partitions,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
2,Final hourly model dataset,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
3,Cleaning state file,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
4,Supported source formats,".csv, .parquet"
5,Parquet compression,SNAPPY
6,Overwrite valid outputs,False



The hourly preprocessing folders and settings were configured successfully.


## What Cell 36 Does

This cell configures the hourly preprocessing workflow without changing any part of the completed daily pipeline.

Two hourly output folders are defined inside `cleaned_model_data`:

- `hourly_cleaned_partitions` stores the cleaned hourly BT partitions;
- `bt_final_hourly_model_dataset` will later contain the POI-enriched hourly partitions supplied to Model 2.

A cleaning-state file is also defined so that previously processed hourly files can be reused when their source files have not changed.

DuckDB is imported here because it provides an efficient way to inspect and transform the much larger hourly source files without loading each complete dataset into pandas memory.

In [38]:
# ================================================================
# CELL 37 - DISCOVER ALL HOURLY SOURCE FILES
# ================================================================

discovered_hourly_files = sorted(
    [
        file_path.resolve()
        for file_path
        in HOURLY_INPUT_FOLDER.iterdir()
        if (
            file_path.is_file()
            and
            file_path.suffix.lower()
            in HOURLY_SUPPORTED_EXTENSIONS
        )
    ],
    key=lambda file_path: (
        file_path.name.lower()
    )
)


HOURLY_SOURCE_FILE_COUNT = len(
    discovered_hourly_files
)


print("=" * 80)
print("CELL 37 - DISCOVER ALL HOURLY SOURCE FILES")
print("=" * 80)


print(
    f"\nHourly source files discovered: "
    f"{HOURLY_SOURCE_FILE_COUNT:,}"
)


if HOURLY_SOURCE_FILE_COUNT == 0:

    raise FileNotFoundError(
        "No supported hourly source files were found in "
        f"{HOURLY_INPUT_FOLDER}."
    )


hourly_file_discovery = pd.DataFrame(
    {
        "File Number": range(
            1,
            HOURLY_SOURCE_FILE_COUNT + 1
        ),
        "Filename": [
            file_path.name
            for file_path
            in discovered_hourly_files
        ],
        "Extension": [
            file_path.suffix.lower()
            for file_path
            in discovered_hourly_files
        ],
        "File Size (MB)": [
            round(
                file_path.stat().st_size
                / (1024 ** 2),
                3
            )
            for file_path
            in discovered_hourly_files
        ],
        "File Path": [
            str(
                file_path
            )
            for file_path
            in discovered_hourly_files
        ]
    }
)


display(
    hourly_file_discovery
)


assert HOURLY_SOURCE_FILE_COUNT > 0

assert (
    hourly_file_discovery[
        "File Size (MB)"
    ]
    >
    0
).all()


print(
    "\nAll currently available hourly source "
    "files were discovered successfully."
)

CELL 37 - DISCOVER ALL HOURLY SOURCE FILES

Hourly source files discovered: 4


,File Number,Filename,Extension,File Size (MB),File Path
0,1,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,.csv,631.979000,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
1,2,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,.csv,615.162000,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
2,3,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,.csv,557.566000,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
3,4,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,.csv,662.761000,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...



All currently available hourly source files were discovered successfully.


## What Cell 37 Does

This cell dynamically discovers every supported hourly BT source file currently available in `data/bt_hourly_data`.

Both CSV and Parquet sources are supported. No fixed number of hourly files is assumed, so additional hourly files supplied in the future will automatically enter the preprocessing workflow.

The discovered filename, format, size and complete path are displayed for verification.

In [39]:
# ================================================================
# CELL 38 - INSPECT HOURLY FILE METADATA AND SCHEMAS
# ================================================================

hourly_file_metadata_records = []


for file_number, file_path in enumerate(
    discovered_hourly_files,
    start=1
):

    inspection_start_time = time.time()

    readable = True
    inspection_error = None
    column_names = None


    try:

        if file_path.suffix.lower() == ".csv":

            header_data = pd.read_csv(
                file_path,
                nrows=0
            )

            column_names = list(
                header_data.columns
            )


        elif file_path.suffix.lower() == ".parquet":

            parquet_file = pq.ParquetFile(
                file_path
            )

            column_names = (
                parquet_file
                .schema_arrow
                .names
            )


    except Exception as error:

        readable = False

        inspection_error = str(
            error
        )


    inspection_seconds = (
        time.time()
        -
        inspection_start_time
    )


    hourly_file_metadata_records.append(
        {
            "File Number": file_number,
            "Filename": file_path.name,
            "File Path": str(
                file_path
            ),
            "Extension": (
                file_path.suffix.lower()
            ),
            "File Size (MB)": round(
                file_path.stat().st_size
                / (1024 ** 2),
                3
            ),
            "Readable": readable,
            "Columns": (
                len(
                    column_names
                )
                if column_names is not None
                else np.nan
            ),
            "Duplicate Column Names": (
                (
                    len(
                        column_names
                    )
                    -
                    len(
                        set(
                            column_names
                        )
                    )
                )
                if column_names is not None
                else np.nan
            ),
            "Inspection Seconds": round(
                inspection_seconds,
                3
            ),
            "Error": inspection_error,
            "Column Names": column_names
        }
    )


hourly_file_metadata = pd.DataFrame(
    hourly_file_metadata_records
)


print("=" * 80)
print("CELL 38 - INSPECT HOURLY FILE METADATA AND SCHEMAS")
print("=" * 80)


display(
    hourly_file_metadata[
        [
            "File Number",
            "Filename",
            "Extension",
            "File Size (MB)",
            "Readable",
            "Columns",
            "Duplicate Column Names",
            "Inspection Seconds",
            "Error"
        ]
    ]
)


HOURLY_READABLE_FILE_COUNT = int(
    hourly_file_metadata[
        "Readable"
    ].sum()
)


print(
    f"\nReadable hourly files: "
    f"{HOURLY_READABLE_FILE_COUNT:,}"
    f"/"
    f"{HOURLY_SOURCE_FILE_COUNT:,}"
)


if HOURLY_READABLE_FILE_COUNT == 0:

    raise ValueError(
        "None of the discovered hourly source "
        "files could be read."
    )


assert (
    hourly_file_metadata.loc[
        hourly_file_metadata[
            "Readable"
        ],
        "Duplicate Column Names"
    ]
    ==
    0
).all()


print(
    "\nHourly file metadata and schemas were "
    "inspected successfully."
)

CELL 38 - INSPECT HOURLY FILE METADATA AND SCHEMAS


,File Number,Filename,Extension,File Size (MB),Readable,Columns,Duplicate Column Names,Inspection Seconds,Error
0,1,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,.csv,631.979000,True,34,0,0.021000,None
1,2,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,.csv,615.162000,True,34,0,0.014000,None
2,3,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,.csv,557.566000,True,34,0,0.015000,None
3,4,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,.csv,662.761000,True,34,0,0.015000,None



Readable hourly files: 4/4

Hourly file metadata and schemas were inspected successfully.


## What Cell 38 Does

This cell inspects the structure of every discovered hourly source file without loading the complete datasets into memory.

For each file it records whether the file is readable, the number of columns, whether any column names are duplicated and any error encountered during inspection.

This establishes which hourly files can safely continue to the structural-validation stage.

In [40]:
# ================================================================
# CELL 39 - DEFINE AND VALIDATE THE HOURLY DATASET STRUCTURE
# ================================================================

readable_hourly_metadata = (
    hourly_file_metadata.loc[
        hourly_file_metadata[
            "Readable"
        ]
        &
        hourly_file_metadata[
            "Column Names"
        ].notna()
    ]
    .copy()
)


if len(
    readable_hourly_metadata
) == 0:

    raise ValueError(
        "No readable hourly schemas are available."
    )


reference_hourly_columns = list(
    readable_hourly_metadata.iloc[
        0
    ][
        "Column Names"
    ]
)


reference_hourly_filename = (
    readable_hourly_metadata.iloc[
        0
    ][
        "Filename"
    ]
)


HOURLY_RAW_COLUMN_PREFIX = (
    "b_geolte_hourly_bin."
)


HOURLY_RAW_COLUMNS = list(
    reference_hourly_columns
)


HOURLY_RENAME_MAP = {
    column_name: (
        column_name[
            len(
                HOURLY_RAW_COLUMN_PREFIX
            ):
        ]
        if column_name.startswith(
            HOURLY_RAW_COLUMN_PREFIX
        )
        else column_name
    )
    for column_name
    in HOURLY_RAW_COLUMNS
}


HOURLY_SOURCE_COLUMNS = [
    HOURLY_RENAME_MAP[
        column_name
    ]
    for column_name
    in HOURLY_RAW_COLUMNS
]


HOURLY_TIME_COLUMN = (
    "time_interval"
)

HOURLY_X_COLUMN = (
    "xbin"
)

HOURLY_Y_COLUMN = (
    "ybin"
)

HOURLY_DATE_COLUMN = (
    "en_dt"
)


HOURLY_REQUIRED_STRUCTURAL_COLUMNS = [
    HOURLY_TIME_COLUMN,
    HOURLY_X_COLUMN,
    HOURLY_Y_COLUMN,
    HOURLY_DATE_COLUMN
]


missing_hourly_structural_columns = [
    column_name
    for column_name
    in HOURLY_REQUIRED_STRUCTURAL_COLUMNS
    if column_name
    not in HOURLY_SOURCE_COLUMNS
]


HOURLY_MEASUREMENT_COLUMNS = [
    column_name
    for column_name
    in HOURLY_SOURCE_COLUMNS
    if column_name
    not in HOURLY_REQUIRED_STRUCTURAL_COLUMNS
]


hourly_schema_comparison_records = []


for _, metadata_row in (
    readable_hourly_metadata.iterrows()
):

    current_columns = list(
        metadata_row[
            "Column Names"
        ]
    )


    hourly_schema_comparison_records.append(
        {
            "Filename": (
                metadata_row[
                    "Filename"
                ]
            ),
            "Columns": len(
                current_columns
            ),
            "Same Column Names": (
                set(
                    current_columns
                )
                ==
                set(
                    reference_hourly_columns
                )
            ),
            "Same Column Order": (
                current_columns
                ==
                reference_hourly_columns
            )
        }
    )


hourly_schema_comparison = pd.DataFrame(
    hourly_schema_comparison_records
)


print("=" * 80)
print("CELL 39 - DEFINE AND VALIDATE THE HOURLY DATASET STRUCTURE")
print("=" * 80)


print(
    f"\nReference schema: "
    f"{reference_hourly_filename}"
)

print(
    f"Raw hourly columns: "
    f"{len(HOURLY_RAW_COLUMNS):,}"
)

print(
    f"Structural columns: "
    f"{len(HOURLY_REQUIRED_STRUCTURAL_COLUMNS):,}"
)

print(
    f"Measurement columns: "
    f"{len(HOURLY_MEASUREMENT_COLUMNS):,}"
)


display(
    hourly_schema_comparison
)


print(
    "\nClean hourly column structure:"
)


display(
    pd.DataFrame(
        {
            "Column Number": range(
                1,
                len(
                    HOURLY_SOURCE_COLUMNS
                )
                +
                1
            ),
            "Raw Column": (
                HOURLY_RAW_COLUMNS
            ),
            "Clean Column": (
                HOURLY_SOURCE_COLUMNS
            )
        }
    )
)


assert len(
    missing_hourly_structural_columns
) == 0

assert (
    hourly_schema_comparison[
        "Same Column Names"
    ]
).all()

assert (
    hourly_schema_comparison[
        "Same Column Order"
    ]
).all()

assert len(
    HOURLY_MEASUREMENT_COLUMNS
) > 0


print(
    "\nEvery readable hourly file uses the "
    "same validated schema."
)

CELL 39 - DEFINE AND VALIDATE THE HOURLY DATASET STRUCTURE

Reference schema: GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2026-07-14_2026-07-15_Neon_v94_csv.csv
Raw hourly columns: 34
Structural columns: 4
Measurement columns: 30


,Filename,Columns,Same Column Names,Same Column Order
0,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,34,True,True
1,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,34,True,True
2,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,34,True,True
3,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,34,True,True



Clean hourly column structure:


,Column Number,Raw Column,Clean Column
0,1,b_geolte_hourly_bin.time_interval,time_interval
1,2,b_geolte_hourly_bin.xbin,xbin
2,3,b_geolte_hourly_bin.ybin,ybin
3,4,b_geolte_hourly_bin.averagedownlinkthroughput,averagedownlinkthroughput
4,5,b_geolte_hourly_bin.averageuplinkthroughput,averageuplinkthroughput
5,6,b_geolte_hourly_bin.csfallbackattempts,csfallbackattempts
6,7,b_geolte_hourly_bin.irathandoverattempts,irathandoverattempts
7,8,b_geolte_hourly_bin.linearaveragersrp,linearaveragersrp
8,9,b_geolte_hourly_bin.linearaveragersrq,linearaveragersrq
9,10,b_geolte_hourly_bin.minutesofuse,minutesofuse



Every readable hourly file uses the same validated schema.


## What Cell 39 Does

This cell defines the official hourly BT structure from the files that are actually present.

The common `b_geolte_hourly_bin.` prefix is removed from the raw column names. Four fields are treated as structural rather than network measurements: `time_interval`, `xbin`, `ybin` and `en_dt`.

Every readable hourly file must contain the same column names in the same order. This prevents files with a different hourly structure from silently entering the cleaning pipeline.

The remaining fields are retained as candidate network measurements. Model 2 will later decide which of those measurements are suitable Isolation Forest features.

In [41]:
# ================================================================
# CELL 40 - AUDIT EVERY COMPLETE HOURLY SOURCE FILE
# ================================================================

def hourly_sql_path(
    file_path
):
    """
    Return a safely escaped path for DuckDB SQL.
    """

    return (
        Path(
            file_path
        )
        .resolve()
        .as_posix()
        .replace(
            "'",
            "''"
        )
    )


def hourly_source_relation_sql(
    file_path
):
    """
    Return the DuckDB source relation for one hourly file.
    """

    safe_path = hourly_sql_path(
        file_path
    )


    if file_path.suffix.lower() == ".csv":

        return (
            "read_csv_auto("
            f"'{safe_path}', "
            "header = true, "
            "all_varchar = true"
            ")"
        )


    if file_path.suffix.lower() == ".parquet":

        return (
            "read_parquet("
            f"'{safe_path}'"
            ")"
        )


    raise ValueError(
        f"Unsupported hourly source format: "
        f"{file_path.suffix}"
    )


def raw_hourly_column(
    clean_column_name
):
    """
    Return the raw source name corresponding to a clean column.
    """

    return next(
        raw_column_name
        for raw_column_name
        in HOURLY_RAW_COLUMNS
        if HOURLY_RENAME_MAP[
            raw_column_name
        ]
        ==
        clean_column_name
    )


HOURLY_RAW_X_COLUMN = raw_hourly_column(
    HOURLY_X_COLUMN
)

HOURLY_RAW_Y_COLUMN = raw_hourly_column(
    HOURLY_Y_COLUMN
)

HOURLY_RAW_DATE_COLUMN = raw_hourly_column(
    HOURLY_DATE_COLUMN
)

HOURLY_RAW_TIME_COLUMN = raw_hourly_column(
    HOURLY_TIME_COLUMN
)


hourly_source_audit_records = []


for file_number, file_path in enumerate(
    discovered_hourly_files,
    start=1
):

    audit_start_time = time.time()

    source_relation = (
        hourly_source_relation_sql(
            file_path
        )
    )


    file_audit = (
        duckdb.sql(
            f"""
            SELECT
                COUNT(*) AS observations,

                COUNT_IF(
                    TRY_CAST(
                        "{HOURLY_RAW_X_COLUMN}"
                        AS DOUBLE
                    )
                    IS NULL
                ) AS invalid_x,

                COUNT_IF(
                    TRY_CAST(
                        "{HOURLY_RAW_Y_COLUMN}"
                        AS DOUBLE
                    )
                    IS NULL
                ) AS invalid_y,

                COUNT_IF(
                    TRY_CAST(
                        "{HOURLY_RAW_DATE_COLUMN}"
                        AS DATE
                    )
                    IS NULL
                ) AS invalid_dates,

                COUNT_IF(
                    TRY_CAST(
                        "{HOURLY_RAW_TIME_COLUMN}"
                        AS TIME
                    )
                    IS NULL
                ) AS invalid_times,

                COUNT(
                    DISTINCT TRY_CAST(
                        "{HOURLY_RAW_DATE_COLUMN}"
                        AS DATE
                    )
                ) AS unique_dates,

                COUNT(
                    DISTINCT TRY_CAST(
                        "{HOURLY_RAW_TIME_COLUMN}"
                        AS TIME
                    )
                ) AS unique_times,

                MIN(
                    TRY_CAST(
                        "{HOURLY_RAW_DATE_COLUMN}"
                        AS DATE
                    )
                ) AS minimum_date,

                MAX(
                    TRY_CAST(
                        "{HOURLY_RAW_DATE_COLUMN}"
                        AS DATE
                    )
                ) AS maximum_date

            FROM {source_relation}
            """
        )
        .df()
    )


    audit_seconds = (
        time.time()
        -
        audit_start_time
    )


    hourly_source_audit_records.append(
        {
            "File Number": file_number,
            "Filename": file_path.name,
            "File Path": str(
                file_path
            ),
            "Observations": int(
                file_audit.loc[
                    0,
                    "observations"
                ]
            ),
            "Unique Dates": int(
                file_audit.loc[
                    0,
                    "unique_dates"
                ]
            ),
            "Unique Times": int(
                file_audit.loc[
                    0,
                    "unique_times"
                ]
            ),
            "Minimum Date": (
                file_audit.loc[
                    0,
                    "minimum_date"
                ]
            ),
            "Maximum Date": (
                file_audit.loc[
                    0,
                    "maximum_date"
                ]
            ),
            "Invalid X": int(
                file_audit.loc[
                    0,
                    "invalid_x"
                ]
            ),
            "Invalid Y": int(
                file_audit.loc[
                    0,
                    "invalid_y"
                ]
            ),
            "Invalid Dates": int(
                file_audit.loc[
                    0,
                    "invalid_dates"
                ]
            ),
            "Invalid Times": int(
                file_audit.loc[
                    0,
                    "invalid_times"
                ]
            ),
            "Audit Seconds": round(
                audit_seconds,
                2
            )
        }
    )


hourly_source_audit = pd.DataFrame(
    hourly_source_audit_records
)


print("=" * 80)
print("CELL 40 - AUDIT EVERY COMPLETE HOURLY SOURCE FILE")
print("=" * 80)


display(
    hourly_source_audit
)


assert (
    hourly_source_audit[
        "Observations"
    ]
    >
    0
).all()

assert (
    hourly_source_audit[
        [
            "Invalid X",
            "Invalid Y",
            "Invalid Dates",
            "Invalid Times"
        ]
    ]
    ==
    0
).all().all()


print(
    "\nEvery current hourly source file "
    "passed the complete structural audit."
)

CELL 40 - AUDIT EVERY COMPLETE HOURLY SOURCE FILE


,File Number,Filename,File Path,Observations,Unique Dates,Unique Times,Minimum Date,Maximum Date,Invalid X,Invalid Y,Invalid Dates,Invalid Times,Audit Seconds
0,1,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...,4896986,1,96,2026-07-14,2026-07-14,0,0,0,0,1.090000
1,2,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...,4771010,1,95,2026-07-15,2026-07-15,0,0,0,0,1.070000
2,3,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...,4329552,1,85,2026-07-16,2026-07-16,0,0,0,0,1.020000
3,4,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...,5131255,1,96,2026-07-17,2026-07-17,0,0,0,0,1.130000



Every current hourly source file passed the complete structural audit.


## What Cell 40 Does

This cell audits every complete hourly source file rather than relying only on small samples.

DuckDB scans the files efficiently and verifies that every observation has valid grid coordinates, a valid date and a valid hourly time interval.

The number of observations, unique dates, unique time intervals and date range are also recorded separately for every source file.

No complete hourly file is allowed to continue if its required spatial or temporal fields cannot be interpreted.

In [42]:
# ================================================================
# CELL 41 - SUMMARISE THE COMPLETE HOURLY SOURCE DATASET
# ================================================================

HOURLY_TOTAL_ROWS = int(
    hourly_source_audit[
        "Observations"
    ].sum()
)


HOURLY_TOTAL_SIZE_GB = (
    sum(
        file_path.stat().st_size
        for file_path
        in discovered_hourly_files
    )
    /
    (1024 ** 3)
)


HOURLY_DISCOVERED_DATES = sorted(
    set(
        hourly_source_audit[
            "Minimum Date"
        ].dropna()
    )
    |
    set(
        hourly_source_audit[
            "Maximum Date"
        ].dropna()
    )
)


hourly_dataset_summary = pd.DataFrame(
    {
        "Measure": [
            "Hourly source files",
            "Readable hourly files",
            "Total observations",
            "Raw source columns",
            "Structural columns",
            "Measurement candidates",
            "Earliest date",
            "Latest date",
            "Maximum unique times in one file",
            "Combined source size (GB)"
        ],
        "Value": [
            HOURLY_SOURCE_FILE_COUNT,
            HOURLY_READABLE_FILE_COUNT,
            HOURLY_TOTAL_ROWS,
            len(
                HOURLY_RAW_COLUMNS
            ),
            len(
                HOURLY_REQUIRED_STRUCTURAL_COLUMNS
            ),
            len(
                HOURLY_MEASUREMENT_COLUMNS
            ),
            hourly_source_audit[
                "Minimum Date"
            ].min(),
            hourly_source_audit[
                "Maximum Date"
            ].max(),
            int(
                hourly_source_audit[
                    "Unique Times"
                ].max()
            ),
            round(
                HOURLY_TOTAL_SIZE_GB,
                3
            )
        ]
    }
)


print("=" * 80)
print("CELL 41 - SUMMARISE THE COMPLETE HOURLY SOURCE DATASET")
print("=" * 80)


display(
    hourly_dataset_summary
)


print(
    "\nObservations by hourly source file:"
)


display(
    hourly_source_audit[
        [
            "File Number",
            "Filename",
            "Observations",
            "Unique Dates",
            "Unique Times",
            "Minimum Date",
            "Maximum Date"
        ]
    ]
)


assert HOURLY_TOTAL_ROWS > 0

assert HOURLY_READABLE_FILE_COUNT == (
    HOURLY_SOURCE_FILE_COUNT
)


print(
    "\nThe complete hourly source dataset was "
    "summarised successfully."
)

CELL 41 - SUMMARISE THE COMPLETE HOURLY SOURCE DATASET


,Measure,Value
0,Hourly source files,4
1,Readable hourly files,4
2,Total observations,19128803
3,Raw source columns,34
4,Structural columns,4
5,Measurement candidates,30
6,Earliest date,2026-07-14 00:00:00
7,Latest date,2026-07-17 00:00:00
8,Maximum unique times in one file,96
9,Combined source size (GB),2.410000



Observations by hourly source file:


,File Number,Filename,Observations,Unique Dates,Unique Times,Minimum Date,Maximum Date
0,1,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4896986,1,96,2026-07-14,2026-07-14
1,2,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4771010,1,95,2026-07-15,2026-07-15
2,3,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4329552,1,85,2026-07-16,2026-07-16
3,4,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,5131255,1,96,2026-07-17,2026-07-17



The complete hourly source dataset was summarised successfully.


## What Cell 41 Does

This cell creates a single summary of the complete hourly source dataset after all files have passed structural validation.

It reports the number of files and observations, the number of structural and candidate measurement fields, the overall date range, the available hourly time structure and the combined source-data size.

These values provide the baseline against which the cleaned hourly outputs will later be validated.

In [43]:
# ================================================================
# CELL 42 - DEFINE THE HOURLY CLEANING PROCESS
# ================================================================

HOURLY_LOCATION_COLUMN = (
    "location_id"
)

HOURLY_DATETIME_COLUMN = (
    "hourly_datetime"
)

HOURLY_HOUR_COLUMN = (
    "hour"
)


HOURLY_ADDED_COLUMNS = [
    HOURLY_LOCATION_COLUMN,
    HOURLY_DATETIME_COLUMN,
    HOURLY_HOUR_COLUMN
]


HOURLY_CLEAN_COLUMNS = [
    *HOURLY_SOURCE_COLUMNS,
    *HOURLY_ADDED_COLUMNS
]


def clean_bt_hourly_dataframe(
    raw_dataframe
):
    """
    Validate and clean one hourly BT DataFrame.
    """

    if not isinstance(
        raw_dataframe,
        pd.DataFrame
    ):

        raise TypeError(
            "raw_dataframe must be a pandas DataFrame."
        )


    if list(
        raw_dataframe.columns
    ) != HOURLY_RAW_COLUMNS:

        raise ValueError(
            "The hourly source schema does not match "
            "the validated hourly structure."
        )


    cleaned_dataframe = (
        raw_dataframe
        .rename(
            columns=HOURLY_RENAME_MAP
        )
        .copy()
    )


    cleaned_dataframe[
        HOURLY_X_COLUMN
    ] = pd.to_numeric(
        cleaned_dataframe[
            HOURLY_X_COLUMN
        ],
        errors="coerce"
    )


    cleaned_dataframe[
        HOURLY_Y_COLUMN
    ] = pd.to_numeric(
        cleaned_dataframe[
            HOURLY_Y_COLUMN
        ],
        errors="coerce"
    )


    for column_name in (
        HOURLY_MEASUREMENT_COLUMNS
    ):

        cleaned_dataframe[
            column_name
        ] = pd.to_numeric(
            cleaned_dataframe[
                column_name
            ],
            errors="coerce"
        )


    cleaned_dataframe[
        HOURLY_MEASUREMENT_COLUMNS
    ] = cleaned_dataframe[
        HOURLY_MEASUREMENT_COLUMNS
    ].replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )


    cleaned_dataframe[
        HOURLY_DATE_COLUMN
    ] = (
        pd.to_datetime(
            cleaned_dataframe[
                HOURLY_DATE_COLUMN
            ],
            errors="coerce"
        )
        .dt.normalize()
    )


    time_text = (
        cleaned_dataframe[
            HOURLY_TIME_COLUMN
        ]
        .astype(
            "string"
        )
        .str.strip()
    )


    cleaned_dataframe[
        HOURLY_TIME_COLUMN
    ] = time_text


    cleaned_dataframe[
        HOURLY_DATETIME_COLUMN
    ] = pd.to_datetime(
        (
            cleaned_dataframe[
                HOURLY_DATE_COLUMN
            ]
            .dt.strftime(
                "%Y-%m-%d"
            )
            +
            " "
            +
            time_text
        ),
        errors="coerce"
    )


    cleaned_dataframe[
        HOURLY_HOUR_COLUMN
    ] = (
        cleaned_dataframe[
            HOURLY_DATETIME_COLUMN
        ]
        .dt.hour
        .astype(
            "Int8"
        )
    )


    cleaned_dataframe[
        HOURLY_LOCATION_COLUMN
    ] = create_location_id(
        cleaned_dataframe[
            HOURLY_X_COLUMN
        ],
        cleaned_dataframe[
            HOURLY_Y_COLUMN
        ]
    )


    cleaned_dataframe = cleaned_dataframe[
        HOURLY_CLEAN_COLUMNS
    ]


    if len(
        cleaned_dataframe
    ) != len(
        raw_dataframe
    ):

        raise AssertionError(
            "Hourly cleaning changed the observation count."
        )


    return cleaned_dataframe


print("=" * 80)
print("CELL 42 - DEFINE THE HOURLY CLEANING PROCESS")
print("=" * 80)


hourly_cleaning_summary = pd.DataFrame(
    {
        "Cleaning Rule": [
            "Validate the hourly source schema",
            "Remove the common hourly table prefix",
            "Convert coordinates to numeric",
            "Convert network measurements to numeric",
            "Replace numeric infinities with missing values",
            "Convert en_dt to datetime",
            "Preserve time_interval",
            "Create hourly_datetime",
            "Create hour",
            "Create location_id using the daily convention",
            "Preserve the original observation count"
        ],
        "Applied": [
            True
        ] * 11
    }
)


display(
    hourly_cleaning_summary
)


print(
    f"\nSource columns : "
    f"{len(HOURLY_SOURCE_COLUMNS):,}"
)

print(
    f"Added columns  : "
    f"{len(HOURLY_ADDED_COLUMNS):,}"
)

print(
    f"Clean columns  : "
    f"{len(HOURLY_CLEAN_COLUMNS):,}"
)


assert len(
    HOURLY_CLEAN_COLUMNS
) == (
    len(
        HOURLY_SOURCE_COLUMNS
    )
    +
    3
)


print(
    "\nThe hourly cleaning process was "
    "defined successfully."
)

CELL 42 - DEFINE THE HOURLY CLEANING PROCESS


,Cleaning Rule,Applied
0,Validate the hourly source schema,True
1,Remove the common hourly table prefix,True
2,Convert coordinates to numeric,True
3,Convert network measurements to numeric,True
4,Replace numeric infinities with missing values,True
5,Convert en_dt to datetime,True
6,Preserve time_interval,True
7,Create hourly_datetime,True
8,Create hour,True
9,Create location_id using the daily convention,True



Source columns : 34
Added columns  : 3
Clean columns  : 37

The hourly cleaning process was defined successfully.


## What Cell 42 Does

This cell defines the cleaning rules applied to the hourly BT data.

The original hourly measurements are retained, while coordinates and network measurements are converted to numeric form and invalid infinite numeric values are represented as missing values.

Three fields are added:

- `location_id` identifies the BT grid location using the same coordinate-based convention as the daily workflow;
- `hourly_datetime` combines the observation date and time interval;
- `hour` provides the integer hour from 0 to 23.

The original observation count is preserved. Feature selection and model-specific imputation are deliberately not performed here because those operations belong to Model 2 rather than the general preprocessing notebook.

In [44]:
# ================================================================
# CELL 43 - TEST THE HOURLY CLEANING PROCESS
# ================================================================

HOURLY_TEST_SOURCE_FILE = (
    discovered_hourly_files[
        0
    ]
)


if (
    HOURLY_TEST_SOURCE_FILE
    .suffix
    .lower()
    ==
    ".csv"
):

    hourly_raw_test = pd.read_csv(
        HOURLY_TEST_SOURCE_FILE,
        nrows=10
    )


else:

    hourly_raw_test = (
        pd.read_parquet(
            HOURLY_TEST_SOURCE_FILE
        )
        .head(
            10
        )
        .copy()
    )


hourly_clean_test = (
    clean_bt_hourly_dataframe(
        hourly_raw_test
    )
)


print("=" * 80)
print("CELL 43 - TEST THE HOURLY CLEANING PROCESS")
print("=" * 80)


print(
    f"\nTest source file:\n"
    f"{HOURLY_TEST_SOURCE_FILE.name}"
)


print(
    "\nCleaned hourly sample:"
)


display(
    hourly_clean_test
)


print(
    "\nCleaned data types:"
)


display(
    pd.DataFrame(
        {
            "Column": (
                hourly_clean_test.columns
            ),
            "Data Type": [
                str(
                    data_type
                )
                for data_type
                in hourly_clean_test.dtypes
            ]
        }
    )
)


assert len(
    hourly_clean_test
) == len(
    hourly_raw_test
)

assert list(
    hourly_clean_test.columns
) == HOURLY_CLEAN_COLUMNS

assert (
    hourly_clean_test[
        HOURLY_LOCATION_COLUMN
    ]
    .notna()
    .all()
)

assert (
    hourly_clean_test[
        HOURLY_DATETIME_COLUMN
    ]
    .notna()
    .all()
)

assert (
    hourly_clean_test[
        HOURLY_HOUR_COLUMN
    ]
    .between(
        0,
        23
    )
    .all()
)


print(
    "\nThe hourly cleaning process passed "
    "the initial sample test."
)

CELL 43 - TEST THE HOURLY CLEANING PROCESS

Test source file:
GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2026-07-14_2026-07-15_Neon_v94_csv.csv

Cleaned hourly sample:


,time_interval,xbin,ybin,averagedownlinkthroughput,averageuplinkthroughput,csfallbackattempts,irathandoverattempts,linearaveragersrp,linearaveragersrq,minutesofuse,numberofconnections,pedestrianminutesofuse,s1handoverattempts,s1handoverfailures,s1handoversuccesses,stationaryminutesofuse,totalconnectionblocks,totalconnectiondrops,totalconnectionnormalreleases,totaldownlinkdataduration,totaldownlinkvolume,totalerabblocks,totalerabdrops,totalerabnormalreleases,totaluplinkdataduration,totaluplinkvolume,vehicularminutesofuse,voiceminutesofuse,x2handoverattempts,x2handoverfailures,x2handoversuccesses,indoorminutesofuse,outdoorminutesofuse,en_dt,location_id,hourly_datetime,hour
0,00:00,600150,5636800,0.000000,0.000000,0,NaN,NaN,NaN,0.170000,2,0,0,0,0,0.170000,0,0,2,0.010000,0,0,0,1,0.020000,0,0.000000,0,0,0,0,0.000000,0.170000,2026-07-14,600150_5636800,2026-07-14,0
1,00:00,600250,5636650,50.000000,0.000000,0,NaN,-69.500000,-7.250000,0.530000,5,0,0,0,0,0.530000,0,0,4,0.040000,2,0,0,6,0.040000,0,0.000000,0,0,0,0,0.000000,0.530000,2026-07-14,600250_5636650,2026-07-14,0
2,00:00,600300,5636700,"4,826.980000",236.690000,0,NaN,-85.500000,-6.750000,30.780000,32,0,0,0,0,30.780000,0,0,28,24.200000,116813,0,0,37,19.240000,4554,0.000000,0,0,0,0,27.150000,3.630000,2026-07-14,600300_5636700,2026-07-14,0
3,00:00,600400,5625050,100.000000,NaN,0,NaN,-107.500000,-9.750000,0.170000,1,0,0,0,0,0.170000,0,0,1,0.020000,2,0,0,2,0.000000,0,0.000000,0,0,0,0,0.170000,0.000000,2026-07-14,600400_5625050,2026-07-14,0
4,00:00,600400,5630950,0.000000,NaN,0,NaN,NaN,NaN,0.170000,1,0,0,0,0,0.170000,0,0,1,0.010000,0,0,0,2,0.000000,0,0.000000,0,0,0,0,0.000000,0.170000,2026-07-14,600400_5630950,2026-07-14,0
5,00:00,600600,5625150,100.000000,8.700000,0,NaN,-123.830000,-14.770000,0.220000,1,0,0,0,0,0.000000,0,0,0,0.030000,3,0,0,0,0.230000,2,0.220000,0,0,0,0,0.000000,0.220000,2026-07-14,600600_5625150,2026-07-14,0
6,00:00,600750,5636950,150.000000,NaN,0,NaN,-110.970000,-7.990000,1.230000,7,0,0,0,0,1.230000,0,0,7,0.080000,12,0,0,12,0.000000,0,0.000000,0,0,0,0,1.230000,0.000000,2026-07-14,600750_5636950,2026-07-14,0
7,00:00,600950,5620000,77.780000,5.440000,0,NaN,-110.360000,-10.640000,1.090000,6,0,0,0,0,1.090000,0,0,6,0.180000,14,0,0,8,5.700000,31,0.000000,0,0,0,0,0.490000,0.600000,2026-07-14,600950_5620000,2026-07-14,0
8,00:00,601250,5630700,0.000000,0.000000,0,NaN,NaN,NaN,0.170000,1,0,0,0,0,0.170000,0,0,1,0.010000,0,0,0,2,0.040000,0,0.000000,0,0,0,0,0.000000,0.170000,2026-07-14,601250_5630700,2026-07-14,0
9,00:00,601500,5623450,36.360000,10.930000,0,NaN,-109.500000,-10.260000,15.000000,1,0,0,0,0,15.000000,0,0,0,0.550000,20,0,0,0,3.020000,33,0.000000,0,0,0,0,15.000000,0.000000,2026-07-14,601500_5623450,2026-07-14,0



Cleaned data types:


,Column,Data Type
0,time_interval,string
1,xbin,int64
2,ybin,int64
3,averagedownlinkthroughput,float64
4,averageuplinkthroughput,float64
5,csfallbackattempts,int64
6,irathandoverattempts,float64
7,linearaveragersrp,float64
8,linearaveragersrq,float64
9,minutesofuse,float64



The hourly cleaning process passed the initial sample test.


## What Cell 43 Does

This cell applies the hourly cleaning function to a small sample from the first current hourly source file.

The test verifies that no observations are lost, the expected cleaned column order is produced, every tested observation receives a valid `location_id` and `hourly_datetime`, and the derived `hour` lies between 0 and 23.

This provides a lightweight validation of the cleaning logic before complete multi-million-row hourly files are written to disk.

In [45]:
# ================================================================
# CELL 44 - CONFIGURE CLEANED HOURLY PARTITION OUTPUT
# ================================================================

def hourly_source_signature(
    source_file
):
    """
    Create a lightweight fingerprint from source-file metadata.
    """

    source_stat = source_file.stat()

    signature_text = (
        f"{source_file.name}|"
        f"{source_stat.st_size}|"
        f"{source_stat.st_mtime_ns}"
    )

    return hashlib.sha256(
        signature_text.encode(
            "utf-8"
        )
    ).hexdigest()


def hourly_cleaned_output_path(
    source_file
):
    """
    Return the cleaned hourly Parquet path.
    """

    return (
        HOURLY_CLEANED_PARTITIONS_FOLDER
        /
        (
            source_file.stem
            +
            HOURLY_CLEANED_FILE_SUFFIX
        )
    )


def hourly_temporary_output_path(
    source_file
):
    """
    Return the temporary hourly Parquet path.
    """

    return (
        HOURLY_CLEANED_PARTITIONS_FOLDER
        /
        (
            source_file.stem
            +
            HOURLY_TEMPORARY_SUFFIX
        )
    )


def validate_hourly_cleaned_parquet(
    parquet_file,
    expected_rows
):
    """
    Validate one cleaned hourly partition.
    """

    if not parquet_file.exists():

        return False


    try:

        parquet_metadata = pq.ParquetFile(
            parquet_file
        )

        return (
            parquet_metadata.metadata.num_rows
            ==
            expected_rows
            and
            parquet_metadata.metadata.num_columns
            ==
            len(
                HOURLY_CLEAN_COLUMNS
            )
            and
            parquet_metadata.schema_arrow.names
            ==
            HOURLY_CLEAN_COLUMNS
        )


    except Exception:

        return False


if HOURLY_CLEANING_STATE_FILE.exists():

    try:

        with open(
            HOURLY_CLEANING_STATE_FILE,
            "r",
            encoding="utf-8"
        ) as state_file:

            previous_hourly_cleaning_state = (
                json.load(
                    state_file
                )
            )

    except Exception:

        previous_hourly_cleaning_state = {}


else:

    previous_hourly_cleaning_state = {}


previous_hourly_source_signatures = (
    previous_hourly_cleaning_state.get(
        "source_signatures",
        {}
    )
)


expected_hourly_cleaned_paths = [
    hourly_cleaned_output_path(
        source_file
    )
    for source_file
    in discovered_hourly_files
]


print("=" * 80)
print("CELL 44 - CONFIGURE CLEANED HOURLY PARTITION OUTPUT")
print("=" * 80)


hourly_output_configuration = pd.DataFrame(
    {
        "Setting": [
            "Hourly input partitions",
            "Expected clean columns",
            "Output folder",
            "Filename suffix",
            "Compression",
            "Overwrite valid outputs",
            "Previous cleaning state available"
        ],
        "Value": [
            HOURLY_SOURCE_FILE_COUNT,
            len(
                HOURLY_CLEAN_COLUMNS
            ),
            str(
                HOURLY_CLEANED_PARTITIONS_FOLDER
            ),
            HOURLY_CLEANED_FILE_SUFFIX,
            HOURLY_PARQUET_COMPRESSION,
            OVERWRITE_HOURLY_CLEANED_PARTITIONS,
            HOURLY_CLEANING_STATE_FILE.exists()
        ]
    }
)


display(
    hourly_output_configuration
)


assert len(
    expected_hourly_cleaned_paths
) == HOURLY_SOURCE_FILE_COUNT

assert len(
    set(
        expected_hourly_cleaned_paths
    )
) == HOURLY_SOURCE_FILE_COUNT


print(
    "\nThe cleaned hourly partition output "
    "was configured successfully."
)

CELL 44 - CONFIGURE CLEANED HOURLY PARTITION OUTPUT


,Setting,Value
0,Hourly input partitions,4
1,Expected clean columns,37
2,Output folder,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
3,Filename suffix,_cleaned_hourly.parquet
4,Compression,SNAPPY
5,Overwrite valid outputs,False
6,Previous cleaning state available,False



The cleaned hourly partition output was configured successfully.


## What Cell 44 Does

This cell configures the reusable cleaned hourly outputs.

Each hourly source file receives its own cleaned Parquet partition. A lightweight source fingerprint is created from the source filename, size and modification time.

The saved cleaning state allows an existing cleaned partition to be reused only when its source fingerprint is unchanged and its Parquet metadata still contains the expected number of observations and columns.

Temporary output paths are also defined so that an interrupted write does not leave an incomplete file appearing to be a valid cleaned partition.

In [46]:
# ================================================================
# CELL 45 - CREATE OR REUSE CLEANED HOURLY PARTITIONS
# ================================================================

def hourly_sql_identifier(
    column_name
):
    """
    Safely quote one SQL identifier.
    """

    return (
        '"'
        +
        str(
            column_name
        ).replace(
            '"',
            '""'
        )
        +
        '"'
    )


def hourly_clean_select_expression(
    raw_column_name
):
    """
    Create the DuckDB cleaning expression for one source column.
    """

    clean_column_name = (
        HOURLY_RENAME_MAP[
            raw_column_name
        ]
    )

    raw_identifier = (
        hourly_sql_identifier(
            raw_column_name
        )
    )

    clean_identifier = (
        hourly_sql_identifier(
            clean_column_name
        )
    )


    if clean_column_name in {
        HOURLY_X_COLUMN,
        HOURLY_Y_COLUMN
    }:

        return (
            f"TRY_CAST({raw_identifier} AS DOUBLE) "
            f"AS {clean_identifier}"
        )


    if clean_column_name == HOURLY_DATE_COLUMN:

        return (
            f"TRY_CAST({raw_identifier} AS TIMESTAMP) "
            f"AS {clean_identifier}"
        )


    if clean_column_name == HOURLY_TIME_COLUMN:

        return (
            f"TRIM(CAST({raw_identifier} AS VARCHAR)) "
            f"AS {clean_identifier}"
        )


    if clean_column_name in (
        HOURLY_MEASUREMENT_COLUMNS
    ):

        return (
            f"TRY_CAST({raw_identifier} AS DOUBLE) "
            f"AS {clean_identifier}"
        )


    return (
        f"{raw_identifier} "
        f"AS {clean_identifier}"
    )


hourly_cleaned_partition_records = []

current_hourly_source_signatures = {}


print("=" * 80)
print("CELL 45 - CREATE OR REUSE CLEANED HOURLY PARTITIONS")
print("=" * 80)


for file_number, source_file in enumerate(
    discovered_hourly_files,
    start=1
):

    partition_start_time = time.time()


    expected_rows = int(
        hourly_source_audit.loc[
            hourly_source_audit[
                "Filename"
            ]
            ==
            source_file.name,
            "Observations"
        ].iloc[
            0
        ]
    )


    output_file = (
        hourly_cleaned_output_path(
            source_file
        )
    )


    temporary_file = (
        hourly_temporary_output_path(
            source_file
        )
    )


    source_signature = (
        hourly_source_signature(
            source_file
        )
    )


    current_hourly_source_signatures[
        source_file.name
    ] = source_signature


    previous_signature = (
        previous_hourly_source_signatures.get(
            source_file.name
        )
    )


    valid_existing_output = (
        validate_hourly_cleaned_parquet(
            output_file,
            expected_rows
        )
    )


    can_reuse = (
        not OVERWRITE_HOURLY_CLEANED_PARTITIONS
        and
        valid_existing_output
        and
        previous_signature
        ==
        source_signature
    )


    if can_reuse:

        action = "Reused"


    else:

        if temporary_file.exists():

            temporary_file.unlink()


        source_relation = (
            hourly_source_relation_sql(
                source_file
            )
        )


        source_select_expressions = [
            hourly_clean_select_expression(
                raw_column_name
            )
            for raw_column_name
            in HOURLY_RAW_COLUMNS
        ]


        source_select_sql = ",\n                ".join(
            source_select_expressions
        )


        raw_x_identifier = (
            hourly_sql_identifier(
                HOURLY_RAW_X_COLUMN
            )
        )

        raw_y_identifier = (
            hourly_sql_identifier(
                HOURLY_RAW_Y_COLUMN
            )
        )

        raw_date_identifier = (
            hourly_sql_identifier(
                HOURLY_RAW_DATE_COLUMN
            )
        )

        raw_time_identifier = (
            hourly_sql_identifier(
                HOURLY_RAW_TIME_COLUMN
            )
        )


        safe_temporary_path = (
            hourly_sql_path(
                temporary_file
            )
        )


        duckdb.sql(
            f"""
            COPY (
                SELECT
                    {source_select_sql},

                    CONCAT(
                        CAST(
                            CAST(
                                ROUND(
                                    TRY_CAST(
                                        {raw_x_identifier}
                                        AS DOUBLE
                                    )
                                )
                                AS BIGINT
                            )
                            AS VARCHAR
                        ),
                        '_',
                        CAST(
                            CAST(
                                ROUND(
                                    TRY_CAST(
                                        {raw_y_identifier}
                                        AS DOUBLE
                                    )
                                )
                                AS BIGINT
                            )
                            AS VARCHAR
                        )
                    )
                    AS "{HOURLY_LOCATION_COLUMN}",

                    TRY_CAST(
                        CONCAT(
                            CAST(
                                TRY_CAST(
                                    {raw_date_identifier}
                                    AS DATE
                                )
                                AS VARCHAR
                            ),
                            ' ',
                            TRIM(
                                CAST(
                                    {raw_time_identifier}
                                    AS VARCHAR
                                )
                            )
                        )
                        AS TIMESTAMP
                    )
                    AS "{HOURLY_DATETIME_COLUMN}",

                    CAST(
                        EXTRACT(
                            HOUR
                            FROM TRY_CAST(
                                CONCAT(
                                    CAST(
                                        TRY_CAST(
                                            {raw_date_identifier}
                                            AS DATE
                                        )
                                        AS VARCHAR
                                    ),
                                    ' ',
                                    TRIM(
                                        CAST(
                                            {raw_time_identifier}
                                            AS VARCHAR
                                        )
                                    )
                                )
                                AS TIMESTAMP
                            )
                        )
                        AS TINYINT
                    )
                    AS "{HOURLY_HOUR_COLUMN}"

                FROM {source_relation}
            )

            TO '{safe_temporary_path}'

            (
                FORMAT PARQUET,
                COMPRESSION {HOURLY_PARQUET_COMPRESSION}
            )
            """
        )


        if not validate_hourly_cleaned_parquet(
            temporary_file,
            expected_rows
        ):

            raise RuntimeError(
                "The newly created hourly partition "
                f"failed validation: {source_file.name}"
            )


        if output_file.exists():

            output_file.unlink()


        temporary_file.replace(
            output_file
        )


        action = "Created"


    partition_seconds = (
        time.time()
        -
        partition_start_time
    )


    hourly_cleaned_partition_records.append(
        {
            "File Number": file_number,
            "Source File": (
                source_file.name
            ),
            "Observations": (
                expected_rows
            ),
            "Output File": (
                output_file.name
            ),
            "Action": action,
            "Minutes": round(
                partition_seconds
                / 60,
                4
            )
        }
    )


hourly_cleaned_partition_summary = pd.DataFrame(
    hourly_cleaned_partition_records
)


current_hourly_cleaning_state = {
    "version": 1,
    "source_file_count": (
        HOURLY_SOURCE_FILE_COUNT
    ),
    "total_rows": (
        HOURLY_TOTAL_ROWS
    ),
    "clean_columns": (
        HOURLY_CLEAN_COLUMNS
    ),
    "source_signatures": (
        current_hourly_source_signatures
    )
}


with open(
    HOURLY_CLEANING_STATE_FILE,
    "w",
    encoding="utf-8"
) as state_file:

    json.dump(
        current_hourly_cleaning_state,
        state_file,
        indent=4
    )


hourly_cleaned_files = [
    hourly_cleaned_output_path(
        source_file
    )
    for source_file
    in discovered_hourly_files
]


print(
    "\nCleaned hourly partition results:"
)


display(
    hourly_cleaned_partition_summary
)


print(
    f"\nCleaned hourly partitions available: "
    f"{len(hourly_cleaned_files):,}"
)


assert len(
    hourly_cleaned_files
) == HOURLY_SOURCE_FILE_COUNT

assert all(
    output_file.exists()
    for output_file
    in hourly_cleaned_files
)

assert all(
    validate_hourly_cleaned_parquet(
        output_file,
        int(
            hourly_source_audit.loc[
                hourly_source_audit[
                    "Filename"
                ]
                ==
                source_file.name,
                "Observations"
            ].iloc[
                0
            ]
        )
    )
    for source_file, output_file
    in zip(
        discovered_hourly_files,
        hourly_cleaned_files
    )
)

assert HOURLY_CLEANING_STATE_FILE.exists()


print(
    "\nAll required cleaned hourly partitions "
    "are available."
)

CELL 45 - CREATE OR REUSE CLEANED HOURLY PARTITIONS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Cleaned hourly partition results:


,File Number,Source File,Observations,Output File,Action,Minutes
0,1,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4896986,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,Created,0.076300
1,2,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4771010,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,Created,0.079900
2,3,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4329552,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,Created,0.073000
3,4,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,5131255,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,Created,0.083000



Cleaned hourly partitions available: 4

All required cleaned hourly partitions are available.


## What Cell 45 Does

This cell creates the complete cleaned hourly dataset.

Each usable hourly source file is transformed directly into a separate Parquet partition using DuckDB. The transformation applies the cleaning rules validated in the previous cells while avoiding the need to load millions of observations into pandas memory at once.

The original hourly BT fields are retained in cleaned form and `location_id`, `hourly_datetime` and `hour` are added.

Existing outputs are reused when both their source fingerprint and Parquet structure remain valid. If a new hourly file is added, only its new cleaned partition needs to be created. If an existing source file changes, its corresponding cleaned partition is rebuilt automatically.

Each new partition is first written to a temporary file and validated before becoming the official output. The current source fingerprints are then saved for future runs.

At this point the hourly BT source data have been cleaned, but POI context has not yet been attached. That enrichment is performed in the next part of the hourly workflow.

In [47]:
# ================================================================
# CELL 46 - VALIDATE ALL CLEANED HOURLY PARTITIONS
# ================================================================

hourly_cleaned_validation_records = []


for file_number, (
    source_file,
    cleaned_file
) in enumerate(
    zip(
        discovered_hourly_files,
        hourly_cleaned_files
    ),
    start=1
):

    expected_rows = int(
        hourly_source_audit.loc[
            hourly_source_audit[
                "Filename"
            ]
            ==
            source_file.name,
            "Observations"
        ].iloc[
            0
        ]
    )


    parquet_metadata = pq.read_metadata(
        cleaned_file
    )


    parquet_schema = pq.read_schema(
        cleaned_file
    )


    safe_cleaned_file = hourly_sql_path(
        cleaned_file
    )


    structural_validation = (
        duckdb.sql(
            f"""
            SELECT
                COUNT(*) AS observations,

                COUNT_IF(
                    "{HOURLY_LOCATION_COLUMN}"
                    IS NULL
                ) AS missing_location_ids,

                COUNT_IF(
                    "{HOURLY_DATE_COLUMN}"
                    IS NULL
                ) AS invalid_dates,

                COUNT_IF(
                    "{HOURLY_DATETIME_COLUMN}"
                    IS NULL
                ) AS invalid_datetimes,

                COUNT_IF(
                    "{HOURLY_HOUR_COLUMN}"
                    IS NULL
                    OR
                    "{HOURLY_HOUR_COLUMN}"
                    < 0
                    OR
                    "{HOURLY_HOUR_COLUMN}"
                    > 23
                ) AS invalid_hours

            FROM read_parquet(
                '{safe_cleaned_file}'
            )
            """
        )
        .df()
    )


    observed_rows = int(
        structural_validation.loc[
            0,
            "observations"
        ]
    )


    missing_location_ids = int(
        structural_validation.loc[
            0,
            "missing_location_ids"
        ]
    )


    invalid_dates = int(
        structural_validation.loc[
            0,
            "invalid_dates"
        ]
    )


    invalid_datetimes = int(
        structural_validation.loc[
            0,
            "invalid_datetimes"
        ]
    )


    invalid_hours = int(
        structural_validation.loc[
            0,
            "invalid_hours"
        ]
    )


    correct_schema = (
        parquet_schema.names
        ==
        HOURLY_CLEAN_COLUMNS
    )


    valid = (
        observed_rows
        ==
        expected_rows
        and
        int(
            parquet_metadata.num_columns
        )
        ==
        len(
            HOURLY_CLEAN_COLUMNS
        )
        and
        correct_schema
        and
        missing_location_ids
        ==
        0
        and
        invalid_dates
        ==
        0
        and
        invalid_datetimes
        ==
        0
        and
        invalid_hours
        ==
        0
    )


    hourly_cleaned_validation_records.append(
        {
            "File Number": file_number,
            "Source File": source_file.name,
            "Expected Rows": expected_rows,
            "Observed Rows": observed_rows,
            "Columns": int(
                parquet_metadata.num_columns
            ),
            "Correct Schema": correct_schema,
            "Missing Location IDs": (
                missing_location_ids
            ),
            "Invalid Dates": invalid_dates,
            "Invalid Datetimes": (
                invalid_datetimes
            ),
            "Invalid Hours": invalid_hours,
            "Valid": valid
        }
    )


hourly_cleaned_validation = pd.DataFrame(
    hourly_cleaned_validation_records
)


HOURLY_CLEANED_TOTAL_ROWS = int(
    hourly_cleaned_validation[
        "Observed Rows"
    ].sum()
)


HOURLY_VALID_CLEANED_PARTITIONS = int(
    hourly_cleaned_validation[
        "Valid"
    ].sum()
)


hourly_temporary_cleaning_files = sorted(
    HOURLY_CLEANED_PARTITIONS_FOLDER.glob(
        "*temporary.parquet"
    )
)


print("=" * 80)
print("CELL 46 - VALIDATE ALL CLEANED HOURLY PARTITIONS")
print("=" * 80)


display(
    hourly_cleaned_validation
)


print(
    f"\nValid cleaned partitions : "
    f"{HOURLY_VALID_CLEANED_PARTITIONS:,}"
    f"/"
    f"{HOURLY_SOURCE_FILE_COUNT:,}"
)


print(
    f"Cleaned observations     : "
    f"{HOURLY_CLEANED_TOTAL_ROWS:,}"
)


print(
    f"Expected observations    : "
    f"{HOURLY_TOTAL_ROWS:,}"
)


print(
    f"Temporary files remaining: "
    f"{len(hourly_temporary_cleaning_files):,}"
)


assert HOURLY_VALID_CLEANED_PARTITIONS == (
    HOURLY_SOURCE_FILE_COUNT
)


assert HOURLY_CLEANED_TOTAL_ROWS == (
    HOURLY_TOTAL_ROWS
)


assert len(
    hourly_temporary_cleaning_files
) == 0


print(
    "\nEvery cleaned hourly partition passed "
    "the complete validation checks."
)

CELL 46 - VALIDATE ALL CLEANED HOURLY PARTITIONS


,File Number,Source File,Expected Rows,Observed Rows,Columns,Correct Schema,Missing Location IDs,Invalid Dates,Invalid Datetimes,Invalid Hours,Valid
0,1,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4896986,4896986,37,True,0,0,0,0,True
1,2,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4771010,4771010,37,True,0,0,0,0,True
2,3,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4329552,4329552,37,True,0,0,0,0,True
3,4,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,5131255,5131255,37,True,0,0,0,0,True



Valid cleaned partitions : 4/4
Cleaned observations     : 19,128,803
Expected observations    : 19,128,803
Temporary files remaining: 0

Every cleaned hourly partition passed the complete validation checks.


## What Cell 46 Does

This cell independently validates every cleaned hourly Parquet partition.

For each source file it confirms that the cleaned output preserves the expected observation count and exact 37-column structure.

The validation also checks that every cleaned observation has a valid location identifier, date, combined date-time and hour-of-day value.

No incomplete temporary files are allowed to remain.

The total number of observations across all cleaned hourly partitions must exactly equal the complete validated hourly source population.

In [48]:
# ================================================================
# CELL 47 - BUILD THE UNIQUE HOURLY BT LOCATION POPULATION
# ================================================================

hourly_cleaned_file_sql = ",\n".join(
    [
        (
            "'"
            +
            hourly_sql_path(
                file_path
            )
            +
            "'"
        )
        for file_path
        in hourly_cleaned_files
    ]
)


hourly_unique_locations = (
    duckdb.sql(
        f"""
        SELECT DISTINCT
            "{HOURLY_LOCATION_COLUMN}"
                AS location_id,

            "{HOURLY_X_COLUMN}"
                AS xbin,

            "{HOURLY_Y_COLUMN}"
                AS ybin

        FROM read_parquet(
            [
                {hourly_cleaned_file_sql}
            ],
            union_by_name = true
        )

        ORDER BY
            location_id
        """
    )
    .df()
)


HOURLY_UNIQUE_LOCATION_COUNT = len(
    hourly_unique_locations
)


hourly_location_duplicate_ids = int(
    hourly_unique_locations[
        "location_id"
    ]
    .duplicated()
    .sum()
)


recreated_hourly_location_ids = (
    create_location_id(
        hourly_unique_locations[
            "xbin"
        ],
        hourly_unique_locations[
            "ybin"
        ]
    )
)


hourly_location_id_mismatches = int(
    (
        recreated_hourly_location_ids
        !=
        hourly_unique_locations[
            "location_id"
        ]
    )
    .fillna(
        True
    )
    .sum()
)


print("=" * 80)
print("CELL 47 - BUILD THE UNIQUE HOURLY BT LOCATION POPULATION")
print("=" * 80)


print(
    f"\nUnique hourly BT locations: "
    f"{HOURLY_UNIQUE_LOCATION_COUNT:,}"
)


print(
    f"Duplicate location IDs    : "
    f"{hourly_location_duplicate_ids:,}"
)


print(
    f"Location-ID mismatches    : "
    f"{hourly_location_id_mismatches:,}"
)


display(
    hourly_unique_locations.head(
        10
    )
)


assert HOURLY_UNIQUE_LOCATION_COUNT > 0


assert hourly_location_duplicate_ids == 0


assert hourly_location_id_mismatches == 0


assert (
    hourly_unique_locations[
        [
            "location_id",
            "xbin",
            "ybin"
        ]
    ]
    .notna()
    .all()
    .all()
)


print(
    "\nThe current unique hourly BT location "
    "population was created successfully."
)

CELL 47 - BUILD THE UNIQUE HOURLY BT LOCATION POPULATION

Unique hourly BT locations: 233,737
Duplicate location IDs    : 0
Location-ID mismatches    : 0


,location_id,xbin,ybin
0,600000_5617050,"600,000.000000","5,617,050.000000"
1,600000_5617200,"600,000.000000","5,617,200.000000"
2,600000_5617450,"600,000.000000","5,617,450.000000"
3,600000_5617500,"600,000.000000","5,617,500.000000"
4,600000_5617550,"600,000.000000","5,617,550.000000"
5,600000_5617600,"600,000.000000","5,617,600.000000"
6,600000_5617700,"600,000.000000","5,617,700.000000"
7,600000_5617750,"600,000.000000","5,617,750.000000"
8,600000_5617800,"600,000.000000","5,617,800.000000"
9,600000_5617850,"600,000.000000","5,617,850.000000"



The current unique hourly BT location population was created successfully.


## What Cell 47 Does

This cell reduces the complete cleaned hourly dataset to one row for every unique BT grid location.

The resulting table contains `location_id`, `xbin` and `ybin`.

POI distances depend only on spatial location rather than the number of observations at that location, so calculating them once per unique grid cell avoids repeating the same spatial calculation millions of times.

The stable location identifier is independently recreated from the coordinates to confirm that the hourly workflow uses exactly the same location convention as the daily workflow.

In [49]:
# ================================================================
# CELL 48 - CHECK HOURLY LOCATIONS AGAINST THE POI LOOKUP
# ================================================================

if not POI_LOCATION_FEATURES_FILE.exists():

    raise FileNotFoundError(
        "The shared POI location lookup does not exist."
    )


if not POI_SOURCE_STATE_FILE.exists():

    raise FileNotFoundError(
        "The POI source-state file does not exist."
    )


with POI_SOURCE_STATE_FILE.open(
    "r",
    encoding="utf-8"
) as state_handle:

    current_saved_poi_state = json.load(
        state_handle
    )


hourly_poi_source_state_matches = (
    current_saved_poi_state.get(
        "sha256"
    )
    ==
    POI_SOURCE_FINGERPRINT
)


if not hourly_poi_source_state_matches:

    raise ValueError(
        "The saved POI lookup does not correspond "
        "to the current processed POI source."
    )


saved_hourly_poi_schema = pq.read_schema(
    POI_LOCATION_FEATURES_FILE
)


if saved_hourly_poi_schema.names != (
    POI_LOOKUP_COLUMNS
):

    raise ValueError(
        "The shared POI lookup does not have "
        "the current expected POI structure."
    )


safe_poi_lookup = hourly_sql_path(
    POI_LOCATION_FEATURES_FILE
)


hourly_location_connection = (
    duckdb.connect()
)


hourly_location_connection.register(
    "hourly_unique_locations_table",
    hourly_unique_locations
)


hourly_poi_coverage = (
    hourly_location_connection.execute(
        f"""
        SELECT
            COUNT(*) AS hourly_locations,

            COUNT(*) FILTER (
                WHERE
                    p.location_id
                    IS NOT NULL
            ) AS matched_locations,

            COUNT(*) FILTER (
                WHERE
                    p.location_id
                    IS NULL
            ) AS new_locations

        FROM
            hourly_unique_locations_table
            AS h

        LEFT JOIN read_parquet(
            '{safe_poi_lookup}'
        )
        AS p

            ON
                h.location_id
                =
                p.location_id
        """
    )
    .df()
)


new_hourly_locations = (
    hourly_location_connection.execute(
        f"""
        SELECT
            h.location_id,
            h.xbin,
            h.ybin

        FROM
            hourly_unique_locations_table
            AS h

        LEFT JOIN read_parquet(
            '{safe_poi_lookup}'
        )
        AS p

            ON
                h.location_id
                =
                p.location_id

        WHERE
            p.location_id
            IS NULL

        ORDER BY
            h.location_id
        """
    )
    .df()
)


hourly_location_connection.close()


HOURLY_EXISTING_POI_LOCATION_COUNT = int(
    hourly_poi_coverage.loc[
        0,
        "matched_locations"
    ]
)


HOURLY_NEW_POI_LOCATION_COUNT = len(
    new_hourly_locations
)


print("=" * 80)
print("CELL 48 - CHECK HOURLY LOCATIONS AGAINST THE POI LOOKUP")
print("=" * 80)


display(
    hourly_poi_coverage
)


print(
    f"\nCurrent hourly locations : "
    f"{HOURLY_UNIQUE_LOCATION_COUNT:,}"
)


print(
    f"Already in POI lookup    : "
    f"{HOURLY_EXISTING_POI_LOCATION_COUNT:,}"
)


print(
    f"New locations requiring "
    f"POI calculation         : "
    f"{HOURLY_NEW_POI_LOCATION_COUNT:,}"
)


print(
    f"POI source state matches : "
    f"{hourly_poi_source_state_matches}"
)


if HOURLY_NEW_POI_LOCATION_COUNT > 0:

    print(
        "\nNew hourly locations:"
    )

    display(
        new_hourly_locations.head(
            10
        )
    )


else:

    print(
        "\nEvery current hourly location already "
        "has POI information in the shared lookup."
    )


assert (
    HOURLY_EXISTING_POI_LOCATION_COUNT
    +
    HOURLY_NEW_POI_LOCATION_COUNT
) == HOURLY_UNIQUE_LOCATION_COUNT


assert new_hourly_locations[
    "location_id"
].is_unique


print(
    "\nHourly POI coverage requirements were "
    "identified successfully."
)

CELL 48 - CHECK HOURLY LOCATIONS AGAINST THE POI LOOKUP


,hourly_locations,matched_locations,new_locations
0,233737,233737,0



Current hourly locations : 233,737
Already in POI lookup    : 233,737
New locations requiring POI calculation         : 0
POI source state matches : True

Every current hourly location already has POI information in the shared lookup.

Hourly POI coverage requirements were identified successfully.


## What Cell 48 Does

This cell compares the current unique hourly BT locations with the reusable POI lookup already created by the daily workflow.

The saved POI source fingerprint and feature structure are checked first so that stale contextual information cannot be used.

Hourly grid locations already present in the lookup require no new spatial calculation.

If future hourly files introduce genuinely new grid locations, only those new locations continue to the POI-distance calculation in the next cell.

This allows the same dynamically maintained POI lookup to support both daily and hourly BT data.

In [50]:
# ================================================================
# CELL 49 - CALCULATE POI FEATURES FOR NEW HOURLY LOCATIONS
# ================================================================

hourly_poi_calculation_start_time = (
    time.time()
)


if HOURLY_NEW_POI_LOCATION_COUNT == 0:

    new_hourly_poi_features = pd.DataFrame(
        columns=POI_LOOKUP_COLUMNS
    )


    hourly_poi_distance_summary = pd.DataFrame(
        columns=[
            "POI Category",
            "Locations",
            "Processing Time (Seconds)"
        ]
    )


    print("=" * 80)
    print(
        "CELL 49 - CALCULATE POI FEATURES "
        "FOR NEW HOURLY LOCATIONS"
    )
    print("=" * 80)


    print(
        "\nNo new hourly POI calculations "
        "are required."
    )


else:

    new_hourly_locations_gdf = (
        gpd.GeoDataFrame(
            new_hourly_locations.copy(),
            geometry=gpd.points_from_xy(
                new_hourly_locations[
                    "xbin"
                ],
                new_hourly_locations[
                    "ybin"
                ]
            ),
            crs=BT_CRS
        )
    )


    new_hourly_geometry_array = (
        new_hourly_locations_gdf
        .geometry
        .to_numpy()
    )


    new_hourly_poi_features = (
        new_hourly_locations[
            [
                "location_id",
                "xbin",
                "ybin"
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    hourly_poi_distance_records = []


    print("=" * 80)
    print(
        "CELL 49 - CALCULATE POI FEATURES "
        "FOR NEW HOURLY LOCATIONS"
    )
    print("=" * 80)


    print(
        f"\nNew hourly locations requiring "
        f"calculation: "
        f"{HOURLY_NEW_POI_LOCATION_COUNT:,}"
    )


    print(
        f"POI categories: "
        f"{len(POI_CATEGORIES):,}"
    )


    for (
        category_number,
        category_name
    ) in enumerate(
        POI_CATEGORIES,
        start=1
    ):

        category_start_time = (
            time.time()
        )


        distance_column = (
            POI_DISTANCE_COLUMN_BY_CATEGORY[
                category_name
            ]
        )


        category_distances = (
            query_nearest_in_batches(
                source_geometries=(
                    new_hourly_geometry_array
                ),
                target_geometries=(
                    poi_category_geometries[
                        category_name
                    ]
                ),
                batch_size=(
                    POI_CALCULATION_BATCH_SIZE
                ),
                return_target_indices=False
            )
        )


        new_hourly_poi_features[
            distance_column
        ] = category_distances


        category_seconds = (
            time.time()
            -
            category_start_time
        )


        hourly_poi_distance_records.append(
            {
                "POI Category": (
                    category_name
                ),
                "Locations": (
                    HOURLY_NEW_POI_LOCATION_COUNT
                ),
                "Minimum Distance": float(
                    np.min(
                        category_distances
                    )
                ),
                "Median Distance": float(
                    np.median(
                        category_distances
                    )
                ),
                "Maximum Distance": float(
                    np.max(
                        category_distances
                    )
                ),
                "Processing Time (Seconds)": (
                    round(
                        category_seconds,
                        2
                    )
                )
            }
        )


        print(
            f"[{category_number:02d}/"
            f"{len(POI_CATEGORIES):02d}] "
            f"{category_name} | "
            f"{category_seconds:,.2f} seconds"
        )


        del category_distances

        gc.collect()


    transport_distance_columns = [
        POI_DISTANCE_COLUMN_BY_CATEGORY[
            category_name
        ]
        for category_name
        in TRANSPORT_HUB_CATEGORIES
    ]


    new_hourly_poi_features[
        "nearest_transport_hub_distance"
    ] = (
        new_hourly_poi_features[
            transport_distance_columns
        ]
        .min(
            axis=1
        )
    )


    (
        hourly_overall_nearest_distances,
        hourly_overall_nearest_positions
    ) = query_nearest_in_batches(
        source_geometries=(
            new_hourly_geometry_array
        ),
        target_geometries=(
            overall_poi_geometries
        ),
        batch_size=(
            POI_CALCULATION_BATCH_SIZE
        ),
        return_target_indices=True
    )


    new_hourly_poi_features[
        "nearest_poi_distance"
    ] = (
        hourly_overall_nearest_distances
    )


    new_hourly_poi_features[
        "nearest_poi_type"
    ] = pd.Series(
        overall_poi_categories[
            hourly_overall_nearest_positions
        ],
        dtype="string"
    )


    new_hourly_poi_features[
        "nearest_poi_name"
    ] = pd.Series(
        overall_poi_names[
            hourly_overall_nearest_positions
        ],
        dtype="string"
    )


    new_hourly_poi_features = (
        new_hourly_poi_features[
            POI_LOOKUP_COLUMNS
        ]
    )


    hourly_poi_distance_summary = (
        pd.DataFrame(
            hourly_poi_distance_records
        )
    )


    print(
        "\nPOI distance calculation summary:"
    )


    display(
        hourly_poi_distance_summary
    )


HOURLY_NEW_POI_FEATURE_ROWS = len(
    new_hourly_poi_features
)


HOURLY_POI_CALCULATION_SECONDS = (
    time.time()
    -
    hourly_poi_calculation_start_time
)


print(
    f"\nPOI rows calculated: "
    f"{HOURLY_NEW_POI_FEATURE_ROWS:,}"
)


print(
    f"Processing time: "
    f"{HOURLY_POI_CALCULATION_SECONDS / 60:,.2f} minutes"
)


assert HOURLY_NEW_POI_FEATURE_ROWS == (
    HOURLY_NEW_POI_LOCATION_COUNT
)


assert list(
    new_hourly_poi_features.columns
) == POI_LOOKUP_COLUMNS


if HOURLY_NEW_POI_FEATURE_ROWS > 0:

    assert (
        new_hourly_poi_features[
            POI_DISTANCE_COLUMNS
        ]
        .notna()
        .all()
        .all()
    )


    assert (
        new_hourly_poi_features[
            POI_DISTANCE_COLUMNS
        ]
        >=
        0
    ).all().all()


print(
    "\nThe required hourly POI information "
    "is available."
)

CELL 49 - CALCULATE POI FEATURES FOR NEW HOURLY LOCATIONS

No new hourly POI calculations are required.

POI rows calculated: 0
Processing time: 0.00 minutes

The required hourly POI information is available.


## What Cell 49 Does

This cell calculates contextual POI information only for hourly grid locations that are not already present in the shared location lookup.

The exact same POI categories, coordinate reference system, nearest-distance method and dynamic feature structure used by the daily workflow are reused.

For every genuinely new hourly location, the cell calculates one nearest-distance field for each current POI category together with the nearest transport hub, overall nearest POI distance, nearest POI type and nearest POI name.

If all hourly locations are already represented in the lookup, no spatial calculation is repeated.

In [51]:
# ================================================================
# CELL 50 - UPDATE AND VALIDATE THE SHARED POI LOOKUP
# ================================================================

HOURLY_POI_EXTENSION_TEMP_FILE = (
    POI_LOCATION_FEATURES_FOLDER
    /
    "BT_Grid_POI_Hourly_Extension.temporary.parquet"
)


HOURLY_POI_LOOKUP_TEMP_FILE = (
    POI_LOCATION_FEATURES_FOLDER
    /
    "BT_Grid_POI_Features.hourly_update.temporary.parquet"
)


for temporary_file in [
    HOURLY_POI_EXTENSION_TEMP_FILE,
    HOURLY_POI_LOOKUP_TEMP_FILE
]:

    if temporary_file.exists():

        temporary_file.unlink()


if HOURLY_NEW_POI_FEATURE_ROWS > 0:

    new_hourly_poi_features.to_parquet(
        HOURLY_POI_EXTENSION_TEMP_FILE,
        index=False,
        engine="pyarrow",
        compression="snappy"
    )


    safe_existing_poi_lookup = (
        hourly_sql_path(
            POI_LOCATION_FEATURES_FILE
        )
    )


    safe_hourly_extension = (
        hourly_sql_path(
            HOURLY_POI_EXTENSION_TEMP_FILE
        )
    )


    safe_hourly_updated_lookup = (
        hourly_sql_path(
            HOURLY_POI_LOOKUP_TEMP_FILE
        )
    )


    duckdb.sql(
        f"""
        COPY (
            SELECT *
            FROM read_parquet(
                '{safe_existing_poi_lookup}'
            )

            UNION ALL

            SELECT *
            FROM read_parquet(
                '{safe_hourly_extension}'
            )
        )

        TO '{safe_hourly_updated_lookup}'

        (
            FORMAT PARQUET,
            COMPRESSION SNAPPY
        )
        """
    )


    updated_lookup_schema = pq.read_schema(
        HOURLY_POI_LOOKUP_TEMP_FILE
    )


    if updated_lookup_schema.names != (
        POI_LOOKUP_COLUMNS
    ):

        raise RuntimeError(
            "The updated shared POI lookup "
            "has an unexpected schema."
        )


    safe_updated_lookup = hourly_sql_path(
        HOURLY_POI_LOOKUP_TEMP_FILE
    )


    updated_lookup_validation = (
        duckdb.sql(
            f"""
            SELECT
                COUNT(*) AS rows,

                COUNT(
                    DISTINCT location_id
                ) AS unique_location_ids

            FROM read_parquet(
                '{safe_updated_lookup}'
            )
            """
        )
        .df()
    )


    if int(
        updated_lookup_validation.loc[
            0,
            "rows"
        ]
    ) != int(
        updated_lookup_validation.loc[
            0,
            "unique_location_ids"
        ]
    ):

        raise RuntimeError(
            "The updated POI lookup contains "
            "duplicate location identifiers."
        )


    POI_LOCATION_FEATURES_FILE.unlink()


    HOURLY_POI_LOOKUP_TEMP_FILE.replace(
        POI_LOCATION_FEATURES_FILE
    )


    HOURLY_POI_LOOKUP_ACTION = (
        "Extended with new hourly locations"
    )


else:

    HOURLY_POI_LOOKUP_ACTION = (
        "Reused without changes"
    )


if HOURLY_POI_EXTENSION_TEMP_FILE.exists():

    HOURLY_POI_EXTENSION_TEMP_FILE.unlink()


safe_final_poi_lookup = hourly_sql_path(
    POI_LOCATION_FEATURES_FILE
)


hourly_coverage_connection = duckdb.connect()


hourly_coverage_connection.register(
    "hourly_unique_locations_table",
    hourly_unique_locations
)


hourly_final_poi_coverage = (
    hourly_coverage_connection.execute(
        f"""
        SELECT
            COUNT(*) AS required_locations,

            COUNT(*) FILTER (
                WHERE
                    p.location_id
                    IS NOT NULL
            ) AS matched_locations,

            COUNT(*) FILTER (
                WHERE
                    p.location_id
                    IS NULL
            ) AS unmatched_locations

        FROM
            hourly_unique_locations_table
            AS h

        LEFT JOIN read_parquet(
            '{safe_final_poi_lookup}'
        )
        AS p

            ON
                h.location_id
                =
                p.location_id
        """
    )
    .df()
)


hourly_coverage_connection.close()


HOURLY_POI_MATCHED_LOCATIONS = int(
    hourly_final_poi_coverage.loc[
        0,
        "matched_locations"
    ]
)


HOURLY_POI_UNMATCHED_LOCATIONS = int(
    hourly_final_poi_coverage.loc[
        0,
        "unmatched_locations"
    ]
)


saved_hourly_poi_metadata = pq.read_metadata(
    POI_LOCATION_FEATURES_FILE
)


saved_hourly_poi_schema = pq.read_schema(
    POI_LOCATION_FEATURES_FILE
)


print("=" * 80)
print("CELL 50 - UPDATE AND VALIDATE THE SHARED POI LOOKUP")
print("=" * 80)


print(
    f"\nAction: "
    f"{HOURLY_POI_LOOKUP_ACTION}"
)


display(
    hourly_final_poi_coverage
)


print(
    f"\nPOI lookup rows       : "
    f"{saved_hourly_poi_metadata.num_rows:,}"
)


print(
    f"POI lookup columns    : "
    f"{saved_hourly_poi_metadata.num_columns:,}"
)


print(
    f"Hourly locations      : "
    f"{HOURLY_UNIQUE_LOCATION_COUNT:,}"
)


print(
    f"Hourly locations found: "
    f"{HOURLY_POI_MATCHED_LOCATIONS:,}"
)


assert saved_hourly_poi_schema.names == (
    POI_LOOKUP_COLUMNS
)


assert HOURLY_POI_MATCHED_LOCATIONS == (
    HOURLY_UNIQUE_LOCATION_COUNT
)


assert HOURLY_POI_UNMATCHED_LOCATIONS == 0


assert not HOURLY_POI_LOOKUP_TEMP_FILE.exists()


assert not HOURLY_POI_EXTENSION_TEMP_FILE.exists()


print(
    "\nThe shared POI lookup now covers every "
    "current hourly BT location."
)

CELL 50 - UPDATE AND VALIDATE THE SHARED POI LOOKUP

Action: Reused without changes


,required_locations,matched_locations,unmatched_locations
0,233737,233737,0



POI lookup rows       : 2,942,185
POI lookup columns    : 31
Hourly locations      : 233,737
Hourly locations found: 233,737

The shared POI lookup now covers every current hourly BT location.


## What Cell 50 Does

This cell updates the reusable shared POI lookup when new hourly BT locations have been discovered.

Only newly calculated hourly location rows are appended. Existing POI information is retained because the processed POI source has already been verified as unchanged.

If no new hourly locations are present, the lookup is reused without rewriting it.

The completed lookup is then independently checked to confirm that every current hourly BT location has exactly the POI structure expected by the notebook.

This allows the POI lookup to become a reusable spatial resource shared by both the daily and hourly preprocessing pipelines.

In [52]:
# ================================================================
# CELL 51 - CONFIGURE THE FINAL HOURLY MODEL-READY DATASET
# ================================================================

HOURLY_FINAL_MODEL_SUFFIX = (
    "_final_hourly_model.parquet"
)


HOURLY_FINAL_TEMPORARY_SUFFIX = (
    ".final_hourly.temporary.parquet"
)


HOURLY_FINAL_MODEL_STATE_FILE = (
    FINAL_HOURLY_MODEL_DATASET_FOLDER
    /
    "_final_hourly_model_state.json"
)


HOURLY_FINAL_MODEL_STATE_TEMP_FILE = (
    FINAL_HOURLY_MODEL_DATASET_FOLDER
    /
    "_final_hourly_model_state.temporary.json"
)


HOURLY_FINAL_MODEL_COLUMNS = [
    *HOURLY_CLEAN_COLUMNS,
    *POI_FEATURE_COLUMNS
]


HOURLY_FINAL_MODEL_COLUMN_COUNT = len(
    HOURLY_FINAL_MODEL_COLUMNS
)


HOURLY_FINAL_POI_JOIN_COLUMNS = [
    "location_id",
    *POI_FEATURE_COLUMNS
]


def hourly_final_output_path(
    source_file
):

    return (
        FINAL_HOURLY_MODEL_DATASET_FOLDER
        /
        (
            source_file.stem
            +
            HOURLY_FINAL_MODEL_SUFFIX
        )
    )


def hourly_final_temporary_path(
    source_file
):

    return (
        FINAL_HOURLY_MODEL_DATASET_FOLDER
        /
        (
            source_file.stem
            +
            HOURLY_FINAL_TEMPORARY_SUFFIX
        )
    )


def validate_hourly_final_parquet(
    parquet_file,
    expected_rows
):

    if not parquet_file.exists():

        return False


    try:

        metadata = pq.read_metadata(
            parquet_file
        )


        schema = pq.read_schema(
            parquet_file
        )


        return (
            int(
                metadata.num_rows
            )
            ==
            int(
                expected_rows
            )
            and
            int(
                metadata.num_columns
            )
            ==
            HOURLY_FINAL_MODEL_COLUMN_COUNT
            and
            schema.names
            ==
            HOURLY_FINAL_MODEL_COLUMNS
        )


    except Exception:

        return False


if HOURLY_FINAL_MODEL_STATE_FILE.exists():

    try:

        with HOURLY_FINAL_MODEL_STATE_FILE.open(
            "r",
            encoding="utf-8"
        ) as state_handle:

            previous_hourly_final_state = (
                json.load(
                    state_handle
                )
            )

    except Exception:

        previous_hourly_final_state = {}


else:

    previous_hourly_final_state = {}


previous_hourly_final_source_signatures = (
    previous_hourly_final_state.get(
        "source_signatures",
        {}
    )
)


previous_hourly_final_poi_fingerprint = (
    previous_hourly_final_state.get(
        "poi_source_sha256"
    )
)


previous_hourly_final_columns = (
    previous_hourly_final_state.get(
        "final_columns"
    )
)


# ------------------------------------------------
# Accept both the new stable POI-content fingerprint
# and the legacy physical GeoPackage fingerprint.
# ------------------------------------------------

HOURLY_FINAL_POI_STATE_MATCHES = (
    previous_hourly_final_poi_fingerprint
    in {
        POI_SOURCE_FINGERPRINT,
        POI_SOURCE_FILE_FINGERPRINT
    }
)


HOURLY_FINAL_COLUMN_STATE_MATCHES = (
    previous_hourly_final_columns
    ==
    HOURLY_FINAL_MODEL_COLUMNS
)


expected_hourly_final_files = [
    hourly_final_output_path(
        source_file
    )
    for source_file
    in discovered_hourly_files
]


print("=" * 80)
print("CELL 51 - CONFIGURE THE FINAL HOURLY MODEL-READY DATASET")
print("=" * 80)


hourly_final_configuration = pd.DataFrame(
    {
        "Setting": [
            "Required hourly partitions",
            "Cleaned hourly columns",
            "POI categories",
            "POI context fields",
            "Final hourly columns",
            "Final output folder",
            "Previous state available",
            "Previous POI state matches",
            "Previous column state matches"
        ],
        "Value": [
            HOURLY_SOURCE_FILE_COUNT,
            len(
                HOURLY_CLEAN_COLUMNS
            ),
            len(
                POI_CATEGORIES
            ),
            len(
                POI_FEATURE_COLUMNS
            ),
            HOURLY_FINAL_MODEL_COLUMN_COUNT,
            str(
                FINAL_HOURLY_MODEL_DATASET_FOLDER
            ),
            HOURLY_FINAL_MODEL_STATE_FILE.exists(),
            HOURLY_FINAL_POI_STATE_MATCHES,
            HOURLY_FINAL_COLUMN_STATE_MATCHES
        ]
    }
)


display(
    hourly_final_configuration
)


assert HOURLY_FINAL_MODEL_COLUMN_COUNT == (
    len(
        HOURLY_CLEAN_COLUMNS
    )
    +
    len(
        POI_FEATURE_COLUMNS
    )
)


assert len(
    expected_hourly_final_files
) == HOURLY_SOURCE_FILE_COUNT


assert len(
    set(
        expected_hourly_final_files
    )
) == HOURLY_SOURCE_FILE_COUNT


print(
    "\nThe final hourly model-ready dataset "
    "was configured successfully."
)

CELL 51 - CONFIGURE THE FINAL HOURLY MODEL-READY DATASET


,Setting,Value
0,Required hourly partitions,4
1,Cleaned hourly columns,37
2,POI categories,24
3,POI context fields,28
4,Final hourly columns,65
5,Final output folder,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
6,Previous state available,False
7,Previous POI state matches,False
8,Previous column state matches,False



The final hourly model-ready dataset was configured successfully.


## What Cell 51 Does

This cell defines the final hourly dataset that will be supplied to Notebook 4.

Each final hourly partition contains every cleaned hourly BT field followed by the complete current set of POI context fields.

The final schema is dynamic. If the processed POI dataset later gains or loses categories, the expected hourly model-ready structure changes automatically.

A separate hourly final-dataset state file records the source-file fingerprints, POI-source fingerprint and final column structure so that unchanged model-ready partitions can be reused safely on later runs.

In [53]:
# ================================================================
# CELL 52 - TEST THE COMPLETE HOURLY POI JOIN
# ================================================================

HOURLY_JOIN_TEST_FILE = (
    hourly_cleaned_files[
        0
    ]
)


HOURLY_JOIN_TEST_EXPECTED_ROWS = int(
    hourly_cleaned_validation.loc[
        0,
        "Observed Rows"
    ]
)


safe_hourly_join_test_file = (
    hourly_sql_path(
        HOURLY_JOIN_TEST_FILE
    )
)


safe_hourly_poi_lookup = (
    hourly_sql_path(
        POI_LOCATION_FEATURES_FILE
    )
)


hourly_poi_join_validation = (
    duckdb.sql(
        f"""
        SELECT
            COUNT(*) AS observations,

            COUNT(*) FILTER (
                WHERE
                    p.location_id
                    IS NULL
            ) AS unmatched_locations,

            COUNT(*) FILTER (
                WHERE
                    p.nearest_poi_distance
                    IS NULL
            ) AS missing_nearest_poi_distance,

            COUNT(*) FILTER (
                WHERE
                    p.nearest_poi_type
                    IS NULL
            ) AS missing_nearest_poi_type

        FROM read_parquet(
            '{safe_hourly_join_test_file}'
        )
        AS h

        LEFT JOIN read_parquet(
            '{safe_hourly_poi_lookup}'
        )
        AS p

            ON
                h.location_id
                =
                p.location_id
        """
    )
    .df()
)


hourly_test_poi_select_sql = ",\n".join(
    [
        (
            "p."
            +
            hourly_sql_identifier(
                column_name
            )
            +
            " AS "
            +
            hourly_sql_identifier(
                column_name
            )
        )
        for column_name
        in POI_FEATURE_COLUMNS
    ]
)


hourly_poi_join_sample = (
    duckdb.sql(
        f"""
        SELECT
            h.*,

            {hourly_test_poi_select_sql}

        FROM read_parquet(
            '{safe_hourly_join_test_file}'
        )
        AS h

        LEFT JOIN read_parquet(
            '{safe_hourly_poi_lookup}'
        )
        AS p

            ON
                h.location_id
                =
                p.location_id

        LIMIT 10
        """
    )
    .df()
)


HOURLY_JOIN_TEST_ROWS = int(
    hourly_poi_join_validation.loc[
        0,
        "observations"
    ]
)


HOURLY_JOIN_TEST_UNMATCHED = int(
    hourly_poi_join_validation.loc[
        0,
        "unmatched_locations"
    ]
)


print("=" * 80)
print("CELL 52 - TEST THE COMPLETE HOURLY POI JOIN")
print("=" * 80)


display(
    hourly_poi_join_validation
)


print(
    "\nJoined sample:"
)


display(
    hourly_poi_join_sample
)


print(
    f"\nExpected rows : "
    f"{HOURLY_JOIN_TEST_EXPECTED_ROWS:,}"
)


print(
    f"Observed rows : "
    f"{HOURLY_JOIN_TEST_ROWS:,}"
)


print(
    f"Final columns : "
    f"{len(hourly_poi_join_sample.columns):,}"
)


assert HOURLY_JOIN_TEST_ROWS == (
    HOURLY_JOIN_TEST_EXPECTED_ROWS
)


assert HOURLY_JOIN_TEST_UNMATCHED == 0


assert int(
    hourly_poi_join_validation.loc[
        0,
        "missing_nearest_poi_distance"
    ]
) == 0


assert int(
    hourly_poi_join_validation.loc[
        0,
        "missing_nearest_poi_type"
    ]
) == 0


assert list(
    hourly_poi_join_sample.columns
) == HOURLY_FINAL_MODEL_COLUMNS


print(
    "\nThe complete hourly POI join passed "
    "all validation checks."
)

CELL 52 - TEST THE COMPLETE HOURLY POI JOIN


,observations,unmatched_locations,missing_nearest_poi_distance,missing_nearest_poi_type
0,4896986,0,0,0



Joined sample:


,time_interval,xbin,ybin,averagedownlinkthroughput,averageuplinkthroughput,csfallbackattempts,irathandoverattempts,linearaveragersrp,linearaveragersrq,minutesofuse,numberofconnections,pedestrianminutesofuse,s1handoverattempts,s1handoverfailures,s1handoversuccesses,stationaryminutesofuse,totalconnectionblocks,totalconnectiondrops,totalconnectionnormalreleases,totaldownlinkdataduration,totaldownlinkvolume,totalerabblocks,totalerabdrops,totalerabnormalreleases,totaluplinkdataduration,totaluplinkvolume,vehicularminutesofuse,voiceminutesofuse,x2handoverattempts,x2handoverfailures,x2handoversuccesses,indoorminutesofuse,outdoorminutesofuse,en_dt,location_id,hourly_datetime,hour,nearest_airports_distance,nearest_attractions_distance,nearest_beaches_distance,nearest_bus_stations_distance,nearest_bus_stops_distance,nearest_cafes_distance,nearest_fast_foods_distance,nearest_gyms_distance,nearest_holiday_parks_distance,nearest_hospitals_distance,nearest_hotels_distance,nearest_offices_distance,nearest_parks_distance,nearest_pubs_distance,nearest_railway_stations_distance,nearest_restaurants_distance,nearest_schools_distance,nearest_shopping_malls_distance,nearest_sport_centres_distance,nearest_stadiums_distance,nearest_supermarkets_distance,nearest_theme_parks_distance,nearest_universities_distance,nearest_zoos_distance,nearest_transport_hub_distance,nearest_poi_distance,nearest_poi_type,nearest_poi_name
0,00:00,"600,150.000000","5,636,800.000000",0.000000,0.000000,0.000000,NaN,NaN,NaN,0.170000,2.000000,0.000000,0.000000,0.000000,0.000000,0.170000,0.000000,0.000000,2.000000,0.010000,0.000000,0.000000,0.000000,1.000000,0.020000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.170000,2026-07-14,600150_5636800,2026-07-14,0,"17,690.581081","1,404.820965","1,002.251978","12,316.204672",19.121374,168.838589,205.074917,"7,834.947319","4,058.683256","4,168.216800",42.571566,126.733970,"5,342.942309",163.887318,"3,931.500876",149.697087,166.672954,"12,389.877420",525.864769,"13,528.375393","6,598.646133","8,369.976297","11,778.046709","3,107.811972",19.121374,19.121374,bus_stops,Lyndhurst Romsey Road
1,00:00,"600,250.000000","5,636,650.000000",50.000000,0.000000,0.000000,NaN,-69.500000,-7.250000,0.530000,5.000000,0.000000,0.000000,0.000000,0.000000,0.530000,0.000000,0.000000,4.000000,0.040000,2.000000,0.000000,0.000000,6.000000,0.040000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.530000,2026-07-14,600250_5636650,2026-07-14,0,"17,680.506170","1,256.623436",859.186178,"12,268.428635",113.770941,22.576764,25.256807,"7,860.354197","4,173.170689","4,153.089385",178.419284,41.017892,"5,344.654129",60.665407,"3,915.873987",33.777881,235.216158,"12,337.812519",448.610008,"13,475.494004","6,668.540138","8,503.218845","11,781.762093","3,141.009304",113.770941,22.576764,cafes,Peggy May's Cafe
2,00:00,"600,300.000000","5,636,700.000000","4,826.980000",236.690000,0.000000,NaN,-85.500000,-6.750000,30.780000,32.000000,0.000000,0.000000,0.000000,0.000000,30.780000,0.000000,0.000000,28.000000,24.200000,"116,813.000000",0.000000,0.000000,37.000000,19.240000,"4,554.000000",0.000000,0.000000,0.000000,0.000000,0.000000,27.150000,3.630000,2026-07-14,600300_5636700,2026-07-14,0,"17,611.971265","1,310.259247",826.452159,"12,205.062401",153.518466,73.947052,83.863748,"7,789.714569","4,111.515667","4,084.791987",164.632207,110.639797,"5,274.988890",62.758951,"3,847.606494",104.184393,280.610750,"12,275.247416",519.169824,"13,413.102137","6,599.447919","8,446.671565","11,712.066453","3,205.959020",153.518466,62.758951,pubs,Stag Hotel
3,00:00,"600,400.000000","5,625,050.000000",100.000000,NaN,0.000000,NaN,-107.500000,-9.750000,0.170000,1.000000,0.000000,0.000000,0.000000,0.000000,0.170000,0.000000,0.000000,1.000000,0.020000,2.000000,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.170000,0.000000,2026-07-14,600400_5625050,2026-07-14,0,"18,667.291372","4,879.646327","5,461.698612","8,793.539499",1


Expected rows : 4,896,986
Observed rows : 4,896,986
Final columns : 65

The complete hourly POI join passed all validation checks.


## What Cell 52 Does

This cell tests the complete POI enrichment process on one full cleaned hourly partition before all final hourly files are created.

The join uses the stable `location_id` field.

The checks confirm that the join preserves the complete observation count, introduces no unmatched BT locations and supplies valid nearest-POI distance and type information.

A sample of the joined data is also checked against the complete dynamically defined final hourly schema.

In [54]:
# ================================================================
# CELL 53 - CREATE OR REUSE FINAL HOURLY MODEL-READY PARTITIONS
# ================================================================

hourly_final_partition_records = []


poi_feature_select_sql = ",\n                    ".join(
    [
        (
            "p."
            +
            hourly_sql_identifier(
                column_name
            )
            +
            " AS "
            +
            hourly_sql_identifier(
                column_name
            )
        )
        for column_name
        in POI_FEATURE_COLUMNS
    ]
)


expected_hourly_final_set = set(
    expected_hourly_final_files
)


observed_hourly_final_files = set(
    FINAL_HOURLY_MODEL_DATASET_FOLDER.glob(
        f"*{HOURLY_FINAL_MODEL_SUFFIX}"
    )
)


stale_hourly_final_files = sorted(
    observed_hourly_final_files
    -
    expected_hourly_final_set
)


for stale_file in stale_hourly_final_files:

    stale_file.unlink()


print("=" * 80)
print(
    "CELL 53 - CREATE OR REUSE FINAL "
    "HOURLY MODEL-READY PARTITIONS"
)
print("=" * 80)


for file_number, (
    source_file,
    cleaned_file,
    final_file
) in enumerate(
    zip(
        discovered_hourly_files,
        hourly_cleaned_files,
        expected_hourly_final_files
    ),
    start=1
):

    partition_start_time = time.time()


    expected_rows = int(
        hourly_source_audit.loc[
            hourly_source_audit[
                "Filename"
            ]
            ==
            source_file.name,
            "Observations"
        ].iloc[
            0
        ]
    )


    current_source_signature = (
        current_hourly_source_signatures[
            source_file.name
        ]
    )


    previous_source_signature = (
        previous_hourly_final_source_signatures.get(
            source_file.name
        )
    )


    existing_output_valid = (
        validate_hourly_final_parquet(
            final_file,
            expected_rows
        )
    )


    can_reuse_final = (
        existing_output_valid
        and
        previous_source_signature
        ==
        current_source_signature
        and
        HOURLY_FINAL_POI_STATE_MATCHES
        and
        HOURLY_FINAL_COLUMN_STATE_MATCHES
    )


    if can_reuse_final:

        action = "Reused"


    else:

        temporary_file = (
            hourly_final_temporary_path(
                source_file
            )
        )


        if temporary_file.exists():

            temporary_file.unlink()


        safe_cleaned_file = (
            hourly_sql_path(
                cleaned_file
            )
        )


        safe_poi_lookup = (
            hourly_sql_path(
                POI_LOCATION_FEATURES_FILE
            )
        )


        safe_temporary_file = (
            hourly_sql_path(
                temporary_file
            )
        )


        duckdb.sql(
            f"""
            COPY (
                SELECT
                    h.*,

                    {poi_feature_select_sql}

                FROM read_parquet(
                    '{safe_cleaned_file}'
                )
                AS h

                LEFT JOIN read_parquet(
                    '{safe_poi_lookup}'
                )
                AS p

                    ON
                        h.location_id
                        =
                        p.location_id
            )

            TO '{safe_temporary_file}'

            (
                FORMAT PARQUET,
                COMPRESSION SNAPPY
            )
            """
        )


        if not validate_hourly_final_parquet(
            temporary_file,
            expected_rows
        ):

            raise RuntimeError(
                "The new final hourly partition "
                "failed structural validation: "
                f"{source_file.name}"
            )


        if final_file.exists():

            final_file.unlink()


        temporary_file.replace(
            final_file
        )


        action = "Created"


    partition_seconds = (
        time.time()
        -
        partition_start_time
    )


    hourly_final_partition_records.append(
        {
            "File Number": file_number,
            "Source File": (
                source_file.name
            ),
            "Observations": (
                expected_rows
            ),
            "Final File": (
                final_file.name
            ),
            "Action": action,
            "Minutes": round(
                partition_seconds
                / 60,
                4
            )
        }
    )


hourly_final_partition_summary = pd.DataFrame(
    hourly_final_partition_records
)


hourly_final_files = [
    file_path
    for file_path
    in expected_hourly_final_files
    if file_path.exists()
]


hourly_final_state_to_save = {
    "version": 1,
    "source_file_count": (
        HOURLY_SOURCE_FILE_COUNT
    ),
    "total_rows": (
        HOURLY_TOTAL_ROWS
    ),
    "poi_source_sha256": (
        POI_SOURCE_FINGERPRINT
    ),
    "poi_categories": (
        POI_CATEGORIES
    ),
    "poi_feature_columns": (
        POI_FEATURE_COLUMNS
    ),
    "final_columns": (
        HOURLY_FINAL_MODEL_COLUMNS
    ),
    "source_signatures": (
        current_hourly_source_signatures
    )
}


if HOURLY_FINAL_MODEL_STATE_TEMP_FILE.exists():

    HOURLY_FINAL_MODEL_STATE_TEMP_FILE.unlink()


with HOURLY_FINAL_MODEL_STATE_TEMP_FILE.open(
    "w",
    encoding="utf-8"
) as state_handle:

    json.dump(
        hourly_final_state_to_save,
        state_handle,
        indent=4
    )


if HOURLY_FINAL_MODEL_STATE_FILE.exists():

    HOURLY_FINAL_MODEL_STATE_FILE.unlink()


HOURLY_FINAL_MODEL_STATE_TEMP_FILE.replace(
    HOURLY_FINAL_MODEL_STATE_FILE
)


print(
    "\nFinal hourly partition results:"
)


display(
    hourly_final_partition_summary
)


print(
    f"\nFinal hourly partitions available: "
    f"{len(hourly_final_files):,}"
)


assert len(
    hourly_final_files
) == HOURLY_SOURCE_FILE_COUNT


assert all(
    file_path.exists()
    for file_path
    in hourly_final_files
)


assert HOURLY_FINAL_MODEL_STATE_FILE.exists()


assert not HOURLY_FINAL_MODEL_STATE_TEMP_FILE.exists()


print(
    "\nAll required final hourly model-ready "
    "partitions are available."
)

CELL 53 - CREATE OR REUSE FINAL HOURLY MODEL-READY PARTITIONS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Final hourly partition results:


,File Number,Source File,Observations,Final File,Action,Minutes
0,1,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4896986,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,Created,0.109900
1,2,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4771010,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,Created,0.103400
2,3,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4329552,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,Created,0.093700
3,4,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,5131255,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,Created,0.110800



Final hourly partitions available: 4

All required final hourly model-ready partitions are available.


## What Cell 53 Does

This cell creates the final POI-enriched hourly Parquet partitions supplied to Model 2.

Each cleaned hourly partition is joined directly to the shared POI lookup using DuckDB and written as a separate model-ready output.

Existing outputs are reused only when their corresponding source file is unchanged, the POI source fingerprint is unchanged, the final column structure is unchanged and the existing Parquet file still passes structural validation.

This means that adding a new hourly source file does not force all previously valid hourly partitions to be rebuilt.

If the POI source itself changes, the final hourly partitions are rebuilt so that all observations use the updated contextual information.

Outputs that no longer correspond to a current hourly source file are removed from the final hourly dataset.

In [55]:
# ================================================================
# CELL 54 - VERIFY THE FINAL HOURLY MODEL-READY DATASET
# ================================================================

hourly_final_validation_records = []


for file_number, (
    source_file,
    final_file
) in enumerate(
    zip(
        discovered_hourly_files,
        expected_hourly_final_files
    ),
    start=1
):

    expected_rows = int(
        hourly_source_audit.loc[
            hourly_source_audit[
                "Filename"
            ]
            ==
            source_file.name,
            "Observations"
        ].iloc[
            0
        ]
    )


    metadata = pq.read_metadata(
        final_file
    )


    schema = pq.read_schema(
        final_file
    )


    safe_final_file = (
        hourly_sql_path(
            final_file
        )
    )


    content_validation = (
        duckdb.sql(
            f"""
            SELECT
                COUNT(*) AS observations,

                COUNT_IF(
                    location_id
                    IS NULL
                ) AS missing_location_ids,

                COUNT_IF(
                    hourly_datetime
                    IS NULL
                ) AS missing_datetimes,

                COUNT_IF(
                    hour
                    IS NULL
                    OR
                    hour < 0
                    OR
                    hour > 23
                ) AS invalid_hours,

                COUNT_IF(
                    nearest_poi_distance
                    IS NULL
                ) AS missing_poi_distances,

                COUNT_IF(
                    nearest_poi_type
                    IS NULL
                ) AS missing_poi_types

            FROM read_parquet(
                '{safe_final_file}'
            )
            """
        )
        .df()
    )


    observed_rows = int(
        content_validation.loc[
            0,
            "observations"
        ]
    )


    missing_location_ids = int(
        content_validation.loc[
            0,
            "missing_location_ids"
        ]
    )


    missing_datetimes = int(
        content_validation.loc[
            0,
            "missing_datetimes"
        ]
    )


    invalid_hours = int(
        content_validation.loc[
            0,
            "invalid_hours"
        ]
    )


    missing_poi_distances = int(
        content_validation.loc[
            0,
            "missing_poi_distances"
        ]
    )


    missing_poi_types = int(
        content_validation.loc[
            0,
            "missing_poi_types"
        ]
    )


    correct_schema = (
        schema.names
        ==
        HOURLY_FINAL_MODEL_COLUMNS
    )


    valid = (
        observed_rows
        ==
        expected_rows
        and
        int(
            metadata.num_columns
        )
        ==
        HOURLY_FINAL_MODEL_COLUMN_COUNT
        and
        correct_schema
        and
        missing_location_ids
        ==
        0
        and
        missing_datetimes
        ==
        0
        and
        invalid_hours
        ==
        0
        and
        missing_poi_distances
        ==
        0
        and
        missing_poi_types
        ==
        0
    )


    hourly_final_validation_records.append(
        {
            "File Number": file_number,
            "Source File": (
                source_file.name
            ),
            "Expected Rows": (
                expected_rows
            ),
            "Observed Rows": (
                observed_rows
            ),
            "Columns": int(
                metadata.num_columns
            ),
            "Correct Schema": (
                correct_schema
            ),
            "Missing Location IDs": (
                missing_location_ids
            ),
            "Missing Datetimes": (
                missing_datetimes
            ),
            "Invalid Hours": (
                invalid_hours
            ),
            "Missing POI Distances": (
                missing_poi_distances
            ),
            "Missing POI Types": (
                missing_poi_types
            ),
            "Valid": valid
        }
    )


hourly_final_validation = pd.DataFrame(
    hourly_final_validation_records
)


HOURLY_VERIFIED_FINAL_ROWS = int(
    hourly_final_validation[
        "Observed Rows"
    ].sum()
)


HOURLY_VALID_FINAL_PARTITIONS = int(
    hourly_final_validation[
        "Valid"
    ].sum()
)


HOURLY_INVALID_FINAL_PARTITIONS = (
    HOURLY_SOURCE_FILE_COUNT
    -
    HOURLY_VALID_FINAL_PARTITIONS
)


hourly_final_temporary_files = sorted(
    FINAL_HOURLY_MODEL_DATASET_FOLDER.glob(
        "*.temporary.parquet"
    )
)


observed_hourly_final_set = set(
    FINAL_HOURLY_MODEL_DATASET_FOLDER.glob(
        f"*{HOURLY_FINAL_MODEL_SUFFIX}"
    )
)


unexpected_hourly_final_files = sorted(
    observed_hourly_final_set
    -
    set(
        expected_hourly_final_files
    )
)


HOURLY_FINAL_DATASET_SIZE_GB = (
    sum(
        file_path.stat().st_size
        for file_path
        in hourly_final_files
    )
    /
    (1024 ** 3)
)


print("=" * 80)
print("CELL 54 - VERIFY THE FINAL HOURLY MODEL-READY DATASET")
print("=" * 80)


display(
    hourly_final_validation
)


hourly_final_dataset_summary = pd.DataFrame(
    {
        "Check": [
            "Hourly source files",
            "Expected final outputs",
            "Valid final partitions",
            "Invalid final partitions",
            "Expected observations",
            "Verified observations",
            "Row difference",
            "POI categories",
            "POI context fields",
            "Columns per partition",
            "Unexpected old outputs",
            "Temporary files",
            "Combined size (GB)"
        ],
        "Result": [
            HOURLY_SOURCE_FILE_COUNT,
            len(
                expected_hourly_final_files
            ),
            HOURLY_VALID_FINAL_PARTITIONS,
            HOURLY_INVALID_FINAL_PARTITIONS,
            HOURLY_TOTAL_ROWS,
            HOURLY_VERIFIED_FINAL_ROWS,
            (
                HOURLY_VERIFIED_FINAL_ROWS
                -
                HOURLY_TOTAL_ROWS
            ),
            len(
                POI_CATEGORIES
            ),
            len(
                POI_FEATURE_COLUMNS
            ),
            HOURLY_FINAL_MODEL_COLUMN_COUNT,
            len(
                unexpected_hourly_final_files
            ),
            len(
                hourly_final_temporary_files
            ),
            round(
                HOURLY_FINAL_DATASET_SIZE_GB,
                3
            )
        ]
    }
)


print(
    "\nFinal hourly dataset summary:"
)


display(
    hourly_final_dataset_summary
)


assert HOURLY_VALID_FINAL_PARTITIONS == (
    HOURLY_SOURCE_FILE_COUNT
)


assert HOURLY_INVALID_FINAL_PARTITIONS == 0


assert HOURLY_VERIFIED_FINAL_ROWS == (
    HOURLY_TOTAL_ROWS
)


assert len(
    unexpected_hourly_final_files
) == 0


assert len(
    hourly_final_temporary_files
) == 0


assert HOURLY_FINAL_MODEL_STATE_FILE.exists()


print(
    "\nEvery current hourly source file has one "
    "verified final model-ready partition."
)

CELL 54 - VERIFY THE FINAL HOURLY MODEL-READY DATASET


,File Number,Source File,Expected Rows,Observed Rows,Columns,Correct Schema,Missing Location IDs,Missing Datetimes,Invalid Hours,Missing POI Distances,Missing POI Types,Valid
0,1,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4896986,4896986,65,True,0,0,0,0,0,True
1,2,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4771010,4771010,65,True,0,0,0,0,0,True
2,3,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,4329552,4329552,65,True,0,0,0,0,0,True
3,4,GEO_HOURLY_Export_Neon_600k_5617k_622k_5647k_2...,5131255,5131255,65,True,0,0,0,0,0,True



Final hourly dataset summary:


,Check,Result
0,Hourly source files,4.000000
1,Expected final outputs,4.000000
2,Valid final partitions,4.000000
3,Invalid final partitions,0.000000
4,Expected observations,"19,128,803.000000"
5,Verified observations,"19,128,803.000000"
6,Row difference,0.000000
7,POI categories,24.000000
8,POI context fields,28.000000
9,Columns per partition,65.000000



Every current hourly source file has one verified final model-ready partition.


## What Cell 54 Does

This cell independently verifies every final hourly model-ready partition.

Each final file must preserve the complete observation count of its corresponding hourly source file, contain the dynamically defined final column structure and retain valid hourly spatial and temporal identifiers.

The validation also confirms that every observation has the required nearest-POI contextual information.

Across the complete hourly dataset, the verified final observation count must exactly equal the validated source population.

Unexpected old outputs and incomplete temporary files are not permitted.

In [56]:
# ================================================================
# CELL 55 - COMPLETE NOTEBOOK 2
# ================================================================

NOTEBOOK_2_DAILY_READY = (
    len(
        final_model_partition_files
    )
    ==
    len(
        usable_daily_files
    )
    and
    invalid_final_partitions
    ==
    0
    and
    verified_final_rows
    ==
    expected_final_rows
)


NOTEBOOK_2_HOURLY_READY = (
    len(
        hourly_final_files
    )
    ==
    HOURLY_SOURCE_FILE_COUNT
    and
    HOURLY_INVALID_FINAL_PARTITIONS
    ==
    0
    and
    HOURLY_VERIFIED_FINAL_ROWS
    ==
    HOURLY_TOTAL_ROWS
)


notebook_2_completion_summary = pd.DataFrame(
    {
        "Dataset": [
            "Daily",
            "Hourly"
        ],
        "Source Files": [
            len(
                usable_daily_files
            ),
            HOURLY_SOURCE_FILE_COUNT
        ],
        "Final Partitions": [
            len(
                final_model_partition_files
            ),
            len(
                hourly_final_files
            )
        ],
        "Observations": [
            verified_final_rows,
            HOURLY_VERIFIED_FINAL_ROWS
        ],
        "Base/Clean Columns": [
            len(
                CLEANED_BT_COLUMNS
            ),
            len(
                HOURLY_CLEAN_COLUMNS
            )
        ],
        "POI Context Fields": [
            len(
                POI_FEATURE_COLUMNS
            ),
            len(
                POI_FEATURE_COLUMNS
            )
        ],
        "Final Columns": [
            len(
                FINAL_MODEL_COLUMNS
            ),
            HOURLY_FINAL_MODEL_COLUMN_COUNT
        ],
        "Ready": [
            NOTEBOOK_2_DAILY_READY,
            NOTEBOOK_2_HOURLY_READY
        ]
    }
)


print("=" * 80)
print("CELL 55 - COMPLETE NOTEBOOK 2")
print("=" * 80)


display(
    notebook_2_completion_summary
)


print(
    f"\nDaily Model 1 dataset:\n"
    f"{FINAL_MODEL_DATASET_FOLDER}"
)


print(
    f"\nHourly Model 2 dataset:\n"
    f"{FINAL_HOURLY_MODEL_DATASET_FOLDER}"
)


print(
    f"\nShared POI lookup:\n"
    f"{POI_LOCATION_FEATURES_FILE}"
)


print(
    f"\nCurrent POI categories : "
    f"{len(POI_CATEGORIES):,}"
)


print(
    f"Current POI fields     : "
    f"{len(POI_FEATURE_COLUMNS):,}"
)


print(
    f"\nDaily dataset ready    : "
    f"{NOTEBOOK_2_DAILY_READY}"
)


print(
    f"Hourly dataset ready   : "
    f"{NOTEBOOK_2_HOURLY_READY}"
)


assert NOTEBOOK_2_DAILY_READY


assert NOTEBOOK_2_HOURLY_READY


assert FINAL_MODEL_DATASET_FOLDER.exists()


assert FINAL_HOURLY_MODEL_DATASET_FOLDER.exists()


assert POI_LOCATION_FEATURES_FILE.exists()


print(
    "\n"
    +
    "=" * 80
)


print(
    "NOTEBOOK 2 COMPLETE"
)


print(
    "=" * 80
)


print(
    "\nThe daily and hourly BT datasets have "
    "both been cleaned, validated and enriched "
    "with the current POI context."
)


print(
    "\nNotebook 3 can use the daily final dataset "
    "and Notebook 4 can use the hourly final dataset."
)

CELL 55 - COMPLETE NOTEBOOK 2


,Dataset,Source Files,Final Partitions,Observations,Base/Clean Columns,POI Context Fields,Final Columns,Ready
0,Daily,90,90,119267039,34,28,62,True
1,Hourly,4,4,19128803,37,28,65,True



Daily Model 1 dataset:
C:\Users\adaml\OneDrive\Desktop\BT_Dissertation_AL\data\cleaned_model_data\bt_final_model_dataset

Hourly Model 2 dataset:
C:\Users\adaml\OneDrive\Desktop\BT_Dissertation_AL\data\cleaned_model_data\bt_final_hourly_model_dataset

Shared POI lookup:
C:\Users\adaml\OneDrive\Desktop\BT_Dissertation_AL\data\cleaned_model_data\poi_location_features\BT_Grid_POI_Features.parquet

Current POI categories : 24
Current POI fields     : 28

Daily dataset ready    : True
Hourly dataset ready   : True

NOTEBOOK 2 COMPLETE

The daily and hourly BT datasets have both been cleaned, validated and enriched with the current POI context.

Notebook 3 can use the daily final dataset and Notebook 4 can use the hourly final dataset.


## What Cell 55 Does

This cell performs the overall completion check for Notebook 2.

The daily and hourly preprocessing pipelines are verified independently.

The daily model-ready dataset is confirmed as the input to Notebook 3 / Model 1.

The hourly model-ready dataset is confirmed as the input to Notebook 4 / Model 2.

Both datasets use the same dynamically maintained POI feature structure while remaining separate because they represent different temporal resolutions.

# Notebook 2 Complete

Notebook 2 performs cleaning and feature engineering for both BT datasets:

**Daily BT data → `bt_final_model_dataset` → Notebook 3 / Model 1**

**Hourly BT data → `bt_final_hourly_model_dataset` → Notebook 4 / Model 2**

Both datasets are therefore fully prepared and independently verified before the Isolation Forest modelling notebooks begin.
